# Étude d'hyperparamètres U-Net (ISIC 2016)

Ce notebook compare plusieurs configurations d'un U-Net pour la segmentation de lésions cutanées.
Le principe est de faire des comparaisons contrôlées : **on modifie un seul facteur à la fois** et on suit les mêmes métriques.

**Métriques principales :** Dice, IoU, accuracy binaire.


In [ ]:
# ── Setup : imports, pipeline, modèle paramétrable, métriques, helpers ──

import os, random, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt

AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

# ── Pipeline ──

# Auto-détection Kaggle vs local
import glob as _glob
_kaggle = _glob.glob('/kaggle/input/*/dataset_ISIC')
if _kaggle:
    _base = _kaggle[0]  # Kaggle
else:
    _base = 'dataset_ISIC'  # local
IMAGES_DIR = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_Data')
MASKS_DIR  = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_GroundTruth')

def load_image_mask(img_path, mask_path):
    img  = tf.image.decode_jpeg(tf.io.read_file(img_path), channels=3)
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    img  = tf.image.convert_image_dtype(img, tf.float32)
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def preprocess(img, mask, img_size):
    img  = tf.image.resize(img, img_size, method='bilinear')
    mask = tf.image.resize(mask, img_size, method='nearest')
    return img, mask

def augment_flip(img, mask):
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    return img, mask

def augment_heavy(img, mask):
    img, mask = augment_flip(img, mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img  = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, mask

def make_ds(img_files, mask_files, img_size=(256,256), batch_size=8, augment_fn=None):
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    ds = ds.map(lambda i, m: load_image_mask(i, m), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda i, m: preprocess(i, m, img_size), num_parallel_calls=AUTOTUNE)
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.shuffle(len(img_files)).batch(batch_size).prefetch(AUTOTUNE)
    return ds

def split_paths(scale=0.5, val_ratio=0.2, test_ratio=0.1, seed=SEED):
    imgs  = sorted([os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR)])
    masks = sorted([os.path.join(MASKS_DIR, f) for f in os.listdir(MASKS_DIR)])
    rng = random.Random(seed)
    combined = list(zip(imgs, masks))
    rng.shuffle(combined)
    imgs, masks = zip(*combined)
    n = max(1, int(len(imgs) * scale))
    imgs, masks = list(imgs[:n]), list(masks[:n])
    n_train = int(n * (1 - val_ratio - test_ratio))
    n_val   = int(n * val_ratio)
    return ((imgs[:n_train], masks[:n_train]),
            (imgs[n_train:n_train+n_val], masks[n_train:n_train+n_val]),
            (imgs[n_train+n_val:], masks[n_train+n_val:]))

# ── Métriques & losses ──

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def iou_loss(y_true, y_pred):
    return 1.0 - iou_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

METRICS = [dice_coefficient, iou_coefficient, 'binary_accuracy']

# ── Modèle paramétrable ──

def conv_block(x, filters, activation='relu'):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
    return x

def build_unet(input_shape=(256,256,3), base_filters=64, depth=4,
               use_skip=True, dropout_rate=0.0, activation='relu'):
    inputs = layers.Input(shape=input_shape)
    skips = []
    x = inputs
    # Encodeur
    for i in range(depth):
        x = conv_block(x, base_filters * (2**i), activation)
        skips.append(x)
        x = layers.MaxPool2D((2,2))(x)
    # Bottleneck
    x = conv_block(x, base_filters * (2**depth), activation)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    # Décodeur
    for i in reversed(range(depth)):
        x = layers.UpSampling2D((2,2))(x)
        if use_skip:
            x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, base_filters * (2**i), activation)
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return models.Model(inputs, outputs, name=f'unet_f{base_filters}_d{depth}')

# ── Experiment runner ──

EPOCHS = 30
PATIENCE = 8
results = []  # stocke tous les résultats

def run_experiment(name, model, train_ds, val_ds, lr=1e-4, loss_fn=bce_dice_loss):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=loss_fn, metrics=METRICS)
    cb = [tf.keras.callbacks.EarlyStopping(patience=PATIENCE, monitor='val_loss',
                                           restore_best_weights=True)]
    print(f'\n{"="*60}')
    print(f'  {name}  |  params: {model.count_params():,}')
    print(f'{"="*60}')
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=2)
    elapsed = time.time() - t0
    history_dict = {k: [float(x) for x in v] for k, v in hist.history.items()}
    best_dice = max(history_dict['val_dice_coefficient'])
    best_iou  = max(history_dict['val_iou_coefficient'])
    res = dict(name=name, best_val_dice=best_dice, best_val_iou=best_iou,
               params=model.count_params(), time_s=round(elapsed,1), history=history_dict,
               stopped_epoch=len(history_dict['loss']))
    results.append(res)
    print(f'  → Dice={best_dice:.4f}  IoU={best_iou:.4f}  ({elapsed:.0f}s, {len(history_dict["loss"])} epochs)')
    del hist, model
    K.clear_session()
    gc.collect()
    return res

def plot_compare(exp_results, title=''):
    n = len(exp_results)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for r in exp_results:
        h = r['history']
        axes[0].plot(h['val_dice_coefficient'], label=r['name'])
        axes[1].plot(h['val_iou_coefficient'], label=r['name'])
    axes[0].set(title=f'{title} — Val Dice', xlabel='Epoch', ylabel='Dice')
    axes[1].set(title=f'{title} — Val IoU', xlabel='Epoch', ylabel='IoU')
    for ax in axes: ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

print('Setup OK')

## 0) Préparation des données et baseline

Cette étape crée le split `train/val/test` sur 50% des données, puis entraîne une configuration de référence.

**Configuration baseline complète (référence de toutes les comparaisons)**

- Données: auto-détection du dossier (`/kaggle/input/*/dataset_ISIC` ou `dataset_ISIC` en local).
- Split: `scale=0.5`, `val_ratio=0.2`, `test_ratio=0.1`, mélange avec `seed=42`.
- Prétraitement image: décodage JPEG RGB, conversion en `float32` dans `[0,1]`, resize en `256x256` (bilinear).
- Prétraitement masque: décodage PNG 1 canal, binarisation `mask > 127`, resize en `256x256` (nearest).
- Dataset train: `batch_size=8`, `augment_flip` (flip horizontal/vertical aléatoire), `shuffle`, `prefetch(AUTOTUNE)`.
- Dataset validation: `batch_size=8`, sans augmentation.
- Entrée modèle: `input_shape=(256,256,3)`.
- U-Net: `base_filters=64`, `depth=4`, `use_skip=True`, `dropout_rate=0.0`, `activation='relu'`.
- Progression des filtres encodeur (x2 à chaque étage): `64 -> 128 -> 256 -> 512`, puis bottleneck à `1024`.
- Bloc convolutionnel: `Conv2D(3x3, same, he_normal) -> BatchNorm -> ReLU`, répété 2 fois par bloc.
- Downsampling / upsampling: `MaxPool2D(2x2)` à l'encodage, `UpSampling2D(2x2)` au décodage.
- Décodage: concaténation des skip connections puis `conv_block` avec filtres symétriques (`512 -> 256 -> 128 -> 64`).
- Sortie: `Conv2D(1x1)` avec activation `sigmoid` (segmentation binaire).
- Optimisation: `Adam(lr=1e-4)`.
- Loss baseline: `BCE + Dice` (`binary_crossentropy + dice_loss`).
- Métriques suivies: `dice_coefficient`, `iou_coefficient`, `binary_accuracy`.
- Entraînement: `EPOCHS=30`, `EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)`.

Cette baseline sert de point de comparaison pour toutes les expériences suivantes.


In [ ]:
# ── Données (scale=0.5) + Baseline ──

(train_img, train_mask), (val_img, val_mask), (test_img, test_mask) = split_paths(scale=0.5)
print(f'Train: {len(train_img)} | Val: {len(val_img)} | Test: {len(test_img)}')

train_ds = make_ds(train_img, train_mask, augment_fn=augment_flip)
val_ds   = make_ds(val_img, val_mask)

# Baseline : config par défaut
baseline = run_experiment(
    'baseline (f64/d4/skip/bce+dice)',
    build_unet(base_filters=64, depth=4, use_skip=True),
    train_ds, val_ds
)

### Logs `baseline`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
Setup OK

--- Contenu de /kaggle/input/ ---
  /kaggle/input//
  /kaggle/input/datasets/
  /kaggle/input/datasets/sfarmehdi/
  /kaggle/input/datasets/sfarmehdi/dataset-isic/
  /kaggle/input/datasets/sfarmehdi/dataset-isic/dataset_ISIC/
  /kaggle/input/datasets/sfarmehdi/dataset-isic/dataset_ISIC/ISBI2016_ISIC_Part1_Training_Data/
  /kaggle/input/datasets/sfarmehdi/dataset-isic/dataset_ISIC/ISBI2016_ISIC_Part1_Training_GroundTruth/
Train: 315 | Val: 90 | Test: 45

============================================================
</pre>


### Récap `baseline`

Aucun résultat chiffré `→ Dice=...` détecté dans ce bloc (setup uniquement).


## 1) Expérience : capacité du réseau (`base_filters`)

**Ce qu'on teste :** `base_filters = 32, 64, 128`.

Objectif : voir l'impact de la largeur du réseau sur la qualité de segmentation et le coût de calcul.
Tous les autres paramètres restent identiques à la baseline.


In [ ]:
# ── Exp 1 : base_filters (32 / 64 / 128) ──

exp_filters = []
for f in [32, 64, 128]:
    r = run_experiment(
        f'filters={f}',
        build_unet(base_filters=f, depth=4, use_skip=True),
        train_ds, val_ds
    )
    exp_filters.append(r)

plot_compare(exp_filters, 'base_filters')

### Logs `base_filters`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  filters=32  |  params: 7,858,433
============================================================
Epoch 1/30
40/40 - 62s - 2s/step - binary_accuracy: 0.8179 - dice_coefficient: 0.5532 - iou_coefficient: 0.3890 - loss: 0.8840 - val_binary_accuracy: 0.7365 - val_dice_coefficient: 0.3481 - val_iou_coefficient: 0.2113 - val_loss: 1.1913
Epoch 2/30
40/40 - 13s - 325ms/step - binary_accuracy: 0.8755 - dice_coefficient: 0.6599 - iou_coefficient: 0.4960 - loss: 0.6693 - val_binary_accuracy: 0.7534 - val_dice_coefficient: 0.3398 - val_iou_coefficient: 0.2079 - val_loss: 1.1591
Epoch 3/30
40/40 - 13s - 331ms/step - binary_accuracy: 0.8951 - dice_coefficient: 0.7072 - iou_coefficient: 0.5510 - loss: 0.5774 - val_binary_accuracy: 0.8263 - val_dice_coefficient: 0.5861 - val_iou_coefficient: 0.4216 - val_loss: 0.8847
Epoch 4/30
40/40 - 13s - 330ms/step - binary_accuracy: 0.9110 - dice_coefficient: 0.7385 - iou_coefficient: 0.5879 - loss: 0.5061 - val_binary_accuracy: 0.8394 - val_dice_coefficient: 0.5728 - val_iou_coefficient: 0.4057 - val_loss: 0.8440
Epoch 5/30
40/40 - 13s - 321ms/step - binary_accuracy: 0.9147 - dice_coefficient: 0.7524 - iou_coefficient: 0.6068 - loss: 0.4841 - val_binary_accuracy: 0.8674 - val_dice_coefficient: 0.6604 - val_iou_coefficient: 0.4982 - val_loss: 0.7036
Epoch 6/30
40/40 - 13s - 327ms/step - binary_accuracy: 0.9281 - dice_coefficient: 0.7793 - iou_coefficient: 0.6407 - loss: 0.4269 - val_binary_accuracy: 0.8875 - val_dice_coefficient: 0.6973 - val_iou_coefficient: 0.5427 - val_loss: 0.6213
Epoch 7/30
40/40 - 13s - 323ms/step - binary_accuracy: 0.9257 - dice_coefficient: 0.7801 - iou_coefficient: 0.6428 - loss: 0.4303 - val_binary_accuracy: 0.8802 - val_dice_coefficient: 0.7078 - val_iou_coefficient: 0.5500 - val_loss: 0.6164
Epoch 8/30
40/40 - 13s - 321ms/step - binary_accuracy: 0.9197 - dice_coefficient: 0.7699 - iou_coefficient: 0.6308 - loss: 0.4471 - val_binary_accuracy: 0.8970 - val_dice_coefficient: 0.7504 - val_iou_coefficient: 0.6057 - val_loss: 0.5332
Epoch 9/30
40/40 - 13s - 322ms/step - binary_accuracy: 0.9337 - dice_coefficient: 0.7927 - iou_coefficient: 0.6596 - loss: 0.3973 - val_binary_accuracy: 0.8942 - val_dice_coefficient: 0.7548 - val_iou_coefficient: 0.6074 - val_loss: 0.5419
Epoch 10/30
40/40 - 13s - 319ms/step - binary_accuracy: 0.9278 - dice_coefficient: 0.7954 - iou_coefficient: 0.6628 - loss: 0.4052 - val_binary_accuracy: 0.8925 - val_dice_coefficient: 0.7740 - val_iou_coefficient: 0.6362 - val_loss: 0.5372
Epoch 11/30
40/40 - 13s - 318ms/step - binary_accuracy: 0.9364 - dice_coefficient: 0.8046 - iou_coefficient: 0.6774 - loss: 0.3746 - val_binary_accuracy: 0.9286 - val_dice_coefficient: 0.7977 - val_iou_coefficient: 0.6683 - val_loss: 0.4064
Epoch 12/30
40/40 - 13s - 322ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8104 - iou_coefficient: 0.6847 - loss: 0.3646 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.7981 - val_iou_coefficient: 0.6710 - val_loss: 0.3678
Epoch 13/30
40/40 - 13s - 320ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8116 - iou_coefficient: 0.6889 - loss: 0.3655 - val_binary_accuracy: 0.9325 - val_dice_coefficient: 0.8224 - val_iou_coefficient: 0.7002 - val_loss: 0.3642
Epoch 14/30
40/40 - 13s - 330ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8204 - iou_coefficient: 0.6974 - loss: 0.3522 - val_binary_accuracy: 0.9268 - val_dice_coefficient: 0.8253 - val_iou_coefficient: 0.7046 - val_loss: 0.3871
Epoch 15/30
40/40 - 13s - 326ms/step - binary_accuracy: 0.9411 - dice_coefficient: 0.8244 - iou_coefficient: 0.7039 - loss: 0.3413 - val_binary_accuracy: 0.9395 - val_dice_coefficient: 0.8407 - val_iou_coefficient: 0.7269 - val_loss: 0.3278
Epoch 16/30
40/40 - 13s - 322ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8311 - iou_coefficient: 0.7142 - loss: 0.3287 - val_binary_accuracy: 0.9394 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7287 - val_loss: 0.3266
Epoch 17/30
40/40 - 13s - 320ms/step - binary_accuracy: 0.9443 - dice_coefficient: 0.8367 - iou_coefficient: 0.7212 - loss: 0.3199 - val_binary_accuracy: 0.9318 - val_dice_coefficient: 0.8342 - val_iou_coefficient: 0.7167 - val_loss: 0.3525
Epoch 18/30
40/40 - 13s - 317ms/step - binary_accuracy: 0.9469 - dice_coefficient: 0.8434 - iou_coefficient: 0.7313 - loss: 0.3067 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8409 - val_iou_coefficient: 0.7276 - val_loss: 0.3291
Epoch 19/30
40/40 - 13s - 318ms/step - binary_accuracy: 0.9437 - dice_coefficient: 0.8335 - iou_coefficient: 0.7187 - loss: 0.3228 - val_binary_accuracy: 0.9419 - val_dice_coefficient: 0.8256 - val_iou_coefficient: 0.7089 - val_loss: 0.3227
Epoch 20/30
40/40 - 13s - 325ms/step - binary_accuracy: 0.9487 - dice_coefficient: 0.8443 - iou_coefficient: 0.7338 - loss: 0.2951 - val_binary_accuracy: 0.9303 - val_dice_coefficient: 0.8226 - val_iou_coefficient: 0.7027 - val_loss: 0.3678
Epoch 21/30
40/40 - 13s - 314ms/step - binary_accuracy: 0.9420 - dice_coefficient: 0.8343 - iou_coefficient: 0.7201 - loss: 0.3271 - val_binary_accuracy: 0.9414 - val_dice_coefficient: 0.8464 - val_iou_coefficient: 0.7347 - val_loss: 0.3081
Epoch 22/30
40/40 - 13s - 317ms/step - binary_accuracy: 0.9472 - dice_coefficient: 0.8468 - iou_coefficient: 0.7362 - loss: 0.3027 - val_binary_accuracy: 0.9434 - val_dice_coefficient: 0.8525 - val_iou_coefficient: 0.7444 - val_loss: 0.2987
Epoch 23/30
40/40 - 13s - 324ms/step - binary_accuracy: 0.9507 - dice_coefficient: 0.8536 - iou_coefficient: 0.7470 - loss: 0.2846 - val_binary_accuracy: 0.9381 - val_dice_coefficient: 0.8379 - val_iou_coefficient: 0.7225 - val_loss: 0.3315
Epoch 24/30
40/40 - 13s - 320ms/step - binary_accuracy: 0.9492 - dice_coefficient: 0.8490 - iou_coefficient: 0.7417 - loss: 0.2931 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8538 - val_iou_coefficient: 0.7460 - val_loss: 0.3238
Epoch 25/30
40/40 - 13s - 325ms/step - binary_accuracy: 0.9488 - dice_coefficient: 0.8544 - iou_coefficient: 0.7484 - loss: 0.2875 - val_binary_accuracy: 0.9445 - val_dice_coefficient: 0.8484 - val_iou_coefficient: 0.7401 - val_loss: 0.3054
Epoch 26/30
40/40 - 13s - 323ms/step - binary_accuracy: 0.9503 - dice_coefficient: 0.8589 - iou_coefficient: 0.7556 - loss: 0.2808 - val_binary_accuracy: 0.9414 - val_dice_coefficient: 0.8587 - val_iou_coefficient: 0.7532 - val_loss: 0.3022
Epoch 27/30
40/40 - 13s - 321ms/step - binary_accuracy: 0.9546 - dice_coefficient: 0.8657 - iou_coefficient: 0.7649 - loss: 0.2625 - val_binary_accuracy: 0.9429 - val_dice_coefficient: 0.8513 - val_iou_coefficient: 0.7432 - val_loss: 0.3008
Epoch 28/30
40/40 - 13s - 323ms/step - binary_accuracy: 0.9533 - dice_coefficient: 0.8641 - iou_coefficient: 0.7637 - loss: 0.2654 - val_binary_accuracy: 0.9414 - val_dice_coefficient: 0.8532 - val_iou_coefficient: 0.7454 - val_loss: 0.3079
Epoch 29/30
40/40 - 13s - 325ms/step - binary_accuracy: 0.9506 - dice_coefficient: 0.8623 - iou_coefficient: 0.7599 - loss: 0.2768 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8531 - val_iou_coefficient: 0.7455 - val_loss: 0.3166
Epoch 30/30
40/40 - 13s - 326ms/step - binary_accuracy: 0.9542 - dice_coefficient: 0.8717 - iou_coefficient: 0.7741 - loss: 0.2561 - val_binary_accuracy: 0.9426 - val_dice_coefficient: 0.8525 - val_iou_coefficient: 0.7457 - val_loss: 0.3072
  → Dice=0.8587  IoU=0.7532  (436s, 30 epochs)

============================================================
  filters=64  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 81s - 2s/step - binary_accuracy: 0.8144 - dice_coefficient: 0.6017 - iou_coefficient: 0.4359 - loss: 0.8471 - val_binary_accuracy: 0.5770 - val_dice_coefficient: 0.3520 - val_iou_coefficient: 0.2168 - val_loss: 1.5150
Epoch 2/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.8899 - dice_coefficient: 0.6904 - iou_coefficient: 0.5332 - loss: 0.6110 - val_binary_accuracy: 0.7007 - val_dice_coefficient: 0.4912 - val_iou_coefficient: 0.3311 - val_loss: 1.1518
Epoch 3/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.8962 - dice_coefficient: 0.7255 - iou_coefficient: 0.5720 - loss: 0.5492 - val_binary_accuracy: 0.8270 - val_dice_coefficient: 0.6142 - val_iou_coefficient: 0.4494 - val_loss: 0.8588
Epoch 4/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9111 - dice_coefficient: 0.7534 - iou_coefficient: 0.6102 - loss: 0.4960 - val_binary_accuracy: 0.8621 - val_dice_coefficient: 0.6303 - val_iou_coefficient: 0.4660 - val_loss: 0.7637
Epoch 5/30
40/40 - 19s - 481ms/step - binary_accuracy: 0.9136 - dice_coefficient: 0.7716 - iou_coefficient: 0.6307 - loss: 0.4604 - val_binary_accuracy: 0.8635 - val_dice_coefficient: 0.6795 - val_iou_coefficient: 0.5188 - val_loss: 0.7079
Epoch 6/30
40/40 - 20s - 491ms/step - binary_accuracy: 0.9182 - dice_coefficient: 0.7826 - iou_coefficient: 0.6461 - loss: 0.4341 - val_binary_accuracy: 0.9002 - val_dice_coefficient: 0.7046 - val_iou_coefficient: 0.5486 - val_loss: 0.5678
Epoch 7/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9219 - dice_coefficient: 0.7909 - iou_coefficient: 0.6568 - loss: 0.4180 - val_binary_accuracy: 0.9086 - val_dice_coefficient: 0.7795 - val_iou_coefficient: 0.6416 - val_loss: 0.4646
Epoch 8/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9222 - dice_coefficient: 0.8002 - iou_coefficient: 0.6701 - loss: 0.4059 - val_binary_accuracy: 0.9127 - val_dice_coefficient: 0.7794 - val_iou_coefficient: 0.6426 - val_loss: 0.4677
Epoch 9/30
40/40 - 19s - 478ms/step - binary_accuracy: 0.9310 - dice_coefficient: 0.8158 - iou_coefficient: 0.6914 - loss: 0.3745 - val_binary_accuracy: 0.9197 - val_dice_coefficient: 0.7943 - val_iou_coefficient: 0.6613 - val_loss: 0.4219
Epoch 10/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9294 - dice_coefficient: 0.8145 - iou_coefficient: 0.6894 - loss: 0.3735 - val_binary_accuracy: 0.9129 - val_dice_coefficient: 0.8085 - val_iou_coefficient: 0.6816 - val_loss: 0.4233
Epoch 11/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9307 - dice_coefficient: 0.8185 - iou_coefficient: 0.6956 - loss: 0.3715 - val_binary_accuracy: 0.9327 - val_dice_coefficient: 0.8444 - val_iou_coefficient: 0.7331 - val_loss: 0.3397
Epoch 12/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9352 - dice_coefficient: 0.8276 - iou_coefficient: 0.7085 - loss: 0.3486 - val_binary_accuracy: 0.9259 - val_dice_coefficient: 0.8188 - val_iou_coefficient: 0.6957 - val_loss: 0.3858
Epoch 13/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9344 - dice_coefficient: 0.8262 - iou_coefficient: 0.7067 - loss: 0.3538 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8347 - val_iou_coefficient: 0.7184 - val_loss: 0.3493
Epoch 14/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9418 - dice_coefficient: 0.8473 - iou_coefficient: 0.7366 - loss: 0.3138 - val_binary_accuracy: 0.9347 - val_dice_coefficient: 0.8311 - val_iou_coefficient: 0.7205 - val_loss: 0.3192
Epoch 15/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8385 - iou_coefficient: 0.7256 - loss: 0.3283 - val_binary_accuracy: 0.9332 - val_dice_coefficient: 0.8210 - val_iou_coefficient: 0.7039 - val_loss: 0.3406
Epoch 16/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9417 - dice_coefficient: 0.8393 - iou_coefficient: 0.7267 - loss: 0.3128 - val_binary_accuracy: 0.9342 - val_dice_coefficient: 0.8272 - val_iou_coefficient: 0.7081 - val_loss: 0.3585
Epoch 17/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9390 - dice_coefficient: 0.8379 - iou_coefficient: 0.7239 - loss: 0.3279 - val_binary_accuracy: 0.9326 - val_dice_coefficient: 0.8513 - val_iou_coefficient: 0.7438 - val_loss: 0.3210
Epoch 18/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9397 - dice_coefficient: 0.8411 - iou_coefficient: 0.7286 - loss: 0.3218 - val_binary_accuracy: 0.9358 - val_dice_coefficient: 0.8426 - val_iou_coefficient: 0.7294 - val_loss: 0.3341
Epoch 19/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9418 - dice_coefficient: 0.8508 - iou_coefficient: 0.7429 - loss: 0.3041 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8512 - val_iou_coefficient: 0.7438 - val_loss: 0.3234
Epoch 20/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9415 - dice_coefficient: 0.8492 - iou_coefficient: 0.7399 - loss: 0.3080 - val_binary_accuracy: 0.9454 - val_dice_coefficient: 0.8737 - val_iou_coefficient: 0.7769 - val_loss: 0.2732
Epoch 21/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9442 - dice_coefficient: 0.8567 - iou_coefficient: 0.7512 - loss: 0.2953 - val_binary_accuracy: 0.9410 - val_dice_coefficient: 0.8617 - val_iou_coefficient: 0.7595 - val_loss: 0.2975
Epoch 22/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9475 - dice_coefficient: 0.8616 - iou_coefficient: 0.7588 - loss: 0.2801 - val_binary_accuracy: 0.9458 - val_dice_coefficient: 0.8703 - val_iou_coefficient: 0.7717 - val_loss: 0.2766
Epoch 23/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9507 - dice_coefficient: 0.8726 - iou_coefficient: 0.7758 - loss: 0.2619 - val_binary_accuracy: 0.9438 - val_dice_coefficient: 0.8732 - val_iou_coefficient: 0.7763 - val_loss: 0.2814
Epoch 24/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9472 - dice_coefficient: 0.8600 - iou_coefficient: 0.7578 - loss: 0.2774 - val_binary_accuracy: 0.9418 - val_dice_coefficient: 0.8598 - val_iou_coefficient: 0.7567 - val_loss: 0.3007
Epoch 25/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9430 - dice_coefficient: 0.8531 - iou_coefficient: 0.7463 - loss: 0.2981 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8579 - val_iou_coefficient: 0.7528 - val_loss: 0.3012
Epoch 26/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9486 - dice_coefficient: 0.8634 - iou_coefficient: 0.7629 - loss: 0.2772 - val_binary_accuracy: 0.9431 - val_dice_coefficient: 0.8585 - val_iou_coefficient: 0.7537 - val_loss: 0.2921
Epoch 27/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9497 - dice_coefficient: 0.8711 - iou_coefficient: 0.7731 - loss: 0.2672 - val_binary_accuracy: 0.9437 - val_dice_coefficient: 0.8585 - val_iou_coefficient: 0.7535 - val_loss: 0.2936
Epoch 28/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9544 - dice_coefficient: 0.8817 - iou_coefficient: 0.7901 - loss: 0.2398 - val_binary_accuracy: 0.9407 - val_dice_coefficient: 0.8704 - val_iou_coefficient: 0.7718 - val_loss: 0.2990
  → Dice=0.8737  IoU=0.7769  (592s, 28 epochs)

============================================================
  filters=128  |  params: 125,547,521
============================================================
Epoch 1/30
40/40 - 161s - 4s/step - binary_accuracy: 0.8297 - dice_coefficient: 0.6174 - iou_coefficient: 0.4529 - loss: 0.8004 - val_binary_accuracy: 0.5116 - val_dice_coefficient: 0.4888 - val_iou_coefficient: 0.3314 - val_loss: 3.9148
Epoch 2/30
40/40 - 40s - 996ms/step - binary_accuracy: 0.8770 - dice_coefficient: 0.6902 - iou_coefficient: 0.5307 - loss: 0.6335 - val_binary_accuracy: 0.3857 - val_dice_coefficient: 0.4410 - val_iou_coefficient: 0.2883 - val_loss: 4.1139
Epoch 3/30
40/40 - 40s - 1s/step - binary_accuracy: 0.8904 - dice_coefficient: 0.7277 - iou_coefficient: 0.5751 - loss: 0.5730 - val_binary_accuracy: 0.6140 - val_dice_coefficient: 0.4384 - val_iou_coefficient: 0.2847 - val_loss: 2.2181
Epoch 4/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9088 - dice_coefficient: 0.7775 - iou_coefficient: 0.6387 - loss: 0.4738 - val_binary_accuracy: 0.8393 - val_dice_coefficient: 0.7096 - val_iou_coefficient: 0.5577 - val_loss: 1.3132
Epoch 5/30
40/40 - 41s - 1s/step - binary_accuracy: 0.9163 - dice_coefficient: 0.7971 - iou_coefficient: 0.6644 - loss: 0.4330 - val_binary_accuracy: 0.8767 - val_dice_coefficient: 0.7322 - val_iou_coefficient: 0.5861 - val_loss: 0.8308
Epoch 6/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9140 - dice_coefficient: 0.7896 - iou_coefficient: 0.6554 - loss: 0.4423 - val_binary_accuracy: 0.8912 - val_dice_coefficient: 0.7514 - val_iou_coefficient: 0.6056 - val_loss: 0.5721
Epoch 7/30
40/40 - 40s - 992ms/step - binary_accuracy: 0.9178 - dice_coefficient: 0.8005 - iou_coefficient: 0.6704 - loss: 0.4190 - val_binary_accuracy: 0.8205 - val_dice_coefficient: 0.7219 - val_iou_coefficient: 0.5679 - val_loss: 0.9654
Epoch 8/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9130 - dice_coefficient: 0.7869 - iou_coefficient: 0.6530 - loss: 0.4420 - val_binary_accuracy: 0.9208 - val_dice_coefficient: 0.8133 - val_iou_coefficient: 0.6880 - val_loss: 0.4592
Epoch 9/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9230 - dice_coefficient: 0.8081 - iou_coefficient: 0.6807 - loss: 0.3961 - val_binary_accuracy: 0.9358 - val_dice_coefficient: 0.8283 - val_iou_coefficient: 0.7082 - val_loss: 0.3475
Epoch 10/30
40/40 - 40s - 992ms/step - binary_accuracy: 0.9298 - dice_coefficient: 0.8250 - iou_coefficient: 0.7041 - loss: 0.3611 - val_binary_accuracy: 0.8795 - val_dice_coefficient: 0.7490 - val_iou_coefficient: 0.6003 - val_loss: 0.6116
Epoch 11/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9304 - dice_coefficient: 0.8281 - iou_coefficient: 0.7105 - loss: 0.3667 - val_binary_accuracy: 0.9319 - val_dice_coefficient: 0.8449 - val_iou_coefficient: 0.7358 - val_loss: 0.3349
Epoch 12/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9298 - dice_coefficient: 0.8275 - iou_coefficient: 0.7096 - loss: 0.3599 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8493 - val_iou_coefficient: 0.7394 - val_loss: 0.3333
Epoch 13/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9292 - dice_coefficient: 0.8235 - iou_coefficient: 0.7035 - loss: 0.3645 - val_binary_accuracy: 0.9362 - val_dice_coefficient: 0.8510 - val_iou_coefficient: 0.7426 - val_loss: 0.3282
Epoch 14/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9382 - dice_coefficient: 0.8473 - iou_coefficient: 0.7368 - loss: 0.3208 - val_binary_accuracy: 0.9416 - val_dice_coefficient: 0.8572 - val_iou_coefficient: 0.7515 - val_loss: 0.3068
Epoch 15/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9353 - dice_coefficient: 0.8401 - iou_coefficient: 0.7276 - loss: 0.3335 - val_binary_accuracy: 0.9428 - val_dice_coefficient: 0.8608 - val_iou_coefficient: 0.7597 - val_loss: 0.2953
Epoch 16/30
40/40 - 39s - 985ms/step - binary_accuracy: 0.9423 - dice_coefficient: 0.8533 - iou_coefficient: 0.7473 - loss: 0.3012 - val_binary_accuracy: 0.9401 - val_dice_coefficient: 0.8508 - val_iou_coefficient: 0.7431 - val_loss: 0.3102
Epoch 17/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9432 - dice_coefficient: 0.8612 - iou_coefficient: 0.7583 - loss: 0.2915 - val_binary_accuracy: 0.9411 - val_dice_coefficient: 0.8792 - val_iou_coefficient: 0.7859 - val_loss: 0.2942
Epoch 18/30
40/40 - 40s - 995ms/step - binary_accuracy: 0.9410 - dice_coefficient: 0.8551 - iou_coefficient: 0.7494 - loss: 0.3038 - val_binary_accuracy: 0.9327 - val_dice_coefficient: 0.8630 - val_iou_coefficient: 0.7622 - val_loss: 0.3377
Epoch 19/30
40/40 - 40s - 992ms/step - binary_accuracy: 0.9399 - dice_coefficient: 0.8554 - iou_coefficient: 0.7487 - loss: 0.3095 - val_binary_accuracy: 0.9397 - val_dice_coefficient: 0.8538 - val_iou_coefficient: 0.7474 - val_loss: 0.3011
Epoch 20/30
40/40 - 40s - 990ms/step - binary_accuracy: 0.9389 - dice_coefficient: 0.8549 - iou_coefficient: 0.7495 - loss: 0.2985 - val_binary_accuracy: 0.9421 - val_dice_coefficient: 0.8677 - val_iou_coefficient: 0.7700 - val_loss: 0.2944
Epoch 21/30
40/40 - 40s - 996ms/step - binary_accuracy: 0.9372 - dice_coefficient: 0.8508 - iou_coefficient: 0.7421 - loss: 0.3180 - val_binary_accuracy: 0.9404 - val_dice_coefficient: 0.8742 - val_iou_coefficient: 0.7782 - val_loss: 0.3150
Epoch 22/30
40/40 - 40s - 998ms/step - binary_accuracy: 0.9422 - dice_coefficient: 0.8587 - iou_coefficient: 0.7544 - loss: 0.2966 - val_binary_accuracy: 0.9448 - val_dice_coefficient: 0.8624 - val_iou_coefficient: 0.7660 - val_loss: 0.2728
Epoch 23/30
40/40 - 40s - 993ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8631 - iou_coefficient: 0.7636 - loss: 0.2715 - val_binary_accuracy: 0.9421 - val_dice_coefficient: 0.8607 - val_iou_coefficient: 0.7580 - val_loss: 0.2947
Epoch 24/30
40/40 - 40s - 990ms/step - binary_accuracy: 0.9370 - dice_coefficient: 0.8440 - iou_coefficient: 0.7336 - loss: 0.3276 - val_binary_accuracy: 0.9368 - val_dice_coefficient: 0.8464 - val_iou_coefficient: 0.7389 - val_loss: 0.3469
Epoch 25/30
40/40 - 40s - 994ms/step - binary_accuracy: 0.9354 - dice_coefficient: 0.8451 - iou_coefficient: 0.7344 - loss: 0.3346 - val_binary_accuracy: 0.9444 - val_dice_coefficient: 0.8847 - val_iou_coefficient: 0.7941 - val_loss: 0.3405
Epoch 26/30
40/40 - 40s - 988ms/step - binary_accuracy: 0.9504 - dice_coefficient: 0.8774 - iou_coefficient: 0.7831 - loss: 0.2613 - val_binary_accuracy: 0.9425 - val_dice_coefficient: 0.8698 - val_iou_coefficient: 0.7714 - val_loss: 0.3187
Epoch 27/30
40/40 - 40s - 993ms/step - binary_accuracy: 0.9519 - dice_coefficient: 0.8856 - iou_coefficient: 0.7957 - loss: 0.2438 - val_binary_accuracy: 0.9250 - val_dice_coefficient: 0.8312 - val_iou_coefficient: 0.7169 - val_loss: 0.3724
Epoch 28/30
40/40 - 40s - 990ms/step - binary_accuracy: 0.9460 - dice_coefficient: 0.8674 - iou_coefficient: 0.7688 - loss: 0.2777 - val_binary_accuracy: 0.9417 - val_dice_coefficient: 0.8647 - val_iou_coefficient: 0.7633 - val_loss: 0.2941
Epoch 29/30
40/40 - 40s - 992ms/step - binary_accuracy: 0.9457 - dice_coefficient: 0.8728 - iou_coefficient: 0.7758 - loss: 0.2799 - val_binary_accuracy: 0.9398 - val_dice_coefficient: 0.8684 - val_iou_coefficient: 0.7686 - val_loss: 0.3034
Epoch 30/30
40/40 - 40s - 1s/step - binary_accuracy: 0.9506 - dice_coefficient: 0.8810 - iou_coefficient: 0.7886 - loss: 0.2551 - val_binary_accuracy: 0.9441 - val_dice_coefficient: 0.8783 - val_iou_coefficient: 0.7854 - val_loss: 0.2694
  → Dice=0.8847  IoU=0.7941  (1320s, 30 epochs)

============================================================
</pre>


### Récap `base_filters`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `filters=32` | 7,858,433 | 0.8587 | 0.7532 | 436s, 30 epochs |
| `filters=64` | 31,402,497 | 0.8737 | 0.7769 | 592s, 28 epochs |
| `filters=128` | 125,547,521 | 0.8847 | 0.7941 | 1320s, 30 epochs |

**Meilleur:** `filters=128` (Dice=0.8847, IoU=0.7941).


### Figure — Courbes de validation pour `base_filters`

![Figure — Courbes de validation pour `base_filters`](Screenshot%202026-02-18%20at%2023.54.01.png)


## 2) Expérience : profondeur du U-Net (`depth`)

**Ce qu'on teste :** `depth = 3, 4, 5` blocs d'encodage.

Objectif : mesurer l'effet d'un réseau plus ou moins profond sur la performance de validation,
à configuration identique par ailleurs.


In [ ]:
# ── Exp 2 : depth (3 / 4 / 5 blocs encodeur) ──

exp_depth = []
for d in [3, 4, 5]:
    r = run_experiment(
        f'depth={d}',
        build_unet(base_filters=64, depth=d, use_skip=True),
        train_ds, val_ds
    )
    exp_depth.append(r)

plot_compare(exp_depth, 'depth')

### Logs `depth`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  depth=3  |  params: 7,794,177
============================================================
Epoch 1/30
40/40 - 41s - 1s/step - binary_accuracy: 0.8013 - dice_coefficient: 0.5864 - iou_coefficient: 0.4210 - loss: 0.8821 - val_binary_accuracy: 0.7147 - val_dice_coefficient: 0.3174 - val_iou_coefficient: 0.1899 - val_loss: 1.2632
Epoch 2/30
40/40 - 16s - 412ms/step - binary_accuracy: 0.8597 - dice_coefficient: 0.6381 - iou_coefficient: 0.4737 - loss: 0.7264 - val_binary_accuracy: 0.7581 - val_dice_coefficient: 0.3503 - val_iou_coefficient: 0.2129 - val_loss: 1.1713
Epoch 3/30
40/40 - 17s - 417ms/step - binary_accuracy: 0.8767 - dice_coefficient: 0.6743 - iou_coefficient: 0.5133 - loss: 0.6436 - val_binary_accuracy: 0.7983 - val_dice_coefficient: 0.4397 - val_iou_coefficient: 0.2873 - val_loss: 1.0777
Epoch 4/30
40/40 - 17s - 415ms/step - binary_accuracy: 0.8925 - dice_coefficient: 0.7140 - iou_coefficient: 0.5575 - loss: 0.5700 - val_binary_accuracy: 0.8471 - val_dice_coefficient: 0.5481 - val_iou_coefficient: 0.3842 - val_loss: 0.8362
Epoch 5/30
40/40 - 17s - 417ms/step - binary_accuracy: 0.9051 - dice_coefficient: 0.7420 - iou_coefficient: 0.5933 - loss: 0.5126 - val_binary_accuracy: 0.8777 - val_dice_coefficient: 0.6637 - val_iou_coefficient: 0.5019 - val_loss: 0.6753
Epoch 6/30
40/40 - 17s - 413ms/step - binary_accuracy: 0.9050 - dice_coefficient: 0.7461 - iou_coefficient: 0.5981 - loss: 0.5132 - val_binary_accuracy: 0.8845 - val_dice_coefficient: 0.6733 - val_iou_coefficient: 0.5131 - val_loss: 0.6288
Epoch 7/30
40/40 - 16s - 412ms/step - binary_accuracy: 0.9060 - dice_coefficient: 0.7523 - iou_coefficient: 0.6064 - loss: 0.5018 - val_binary_accuracy: 0.9011 - val_dice_coefficient: 0.7087 - val_iou_coefficient: 0.5546 - val_loss: 0.5479
Epoch 8/30
40/40 - 17s - 415ms/step - binary_accuracy: 0.9086 - dice_coefficient: 0.7574 - iou_coefficient: 0.6136 - loss: 0.4837 - val_binary_accuracy: 0.9051 - val_dice_coefficient: 0.7664 - val_iou_coefficient: 0.6247 - val_loss: 0.4943
Epoch 9/30
40/40 - 17s - 421ms/step - binary_accuracy: 0.9161 - dice_coefficient: 0.7733 - iou_coefficient: 0.6347 - loss: 0.4569 - val_binary_accuracy: 0.8970 - val_dice_coefficient: 0.7456 - val_iou_coefficient: 0.6013 - val_loss: 0.5269
Epoch 10/30
40/40 - 17s - 413ms/step - binary_accuracy: 0.9208 - dice_coefficient: 0.7844 - iou_coefficient: 0.6480 - loss: 0.4318 - val_binary_accuracy: 0.9218 - val_dice_coefficient: 0.7645 - val_iou_coefficient: 0.6370 - val_loss: 0.4184
Epoch 11/30
40/40 - 17s - 416ms/step - binary_accuracy: 0.9196 - dice_coefficient: 0.7879 - iou_coefficient: 0.6536 - loss: 0.4325 - val_binary_accuracy: 0.9108 - val_dice_coefficient: 0.7514 - val_iou_coefficient: 0.6157 - val_loss: 0.4848
Epoch 12/30
40/40 - 17s - 414ms/step - binary_accuracy: 0.9171 - dice_coefficient: 0.7825 - iou_coefficient: 0.6464 - loss: 0.4439 - val_binary_accuracy: 0.9245 - val_dice_coefficient: 0.7984 - val_iou_coefficient: 0.6685 - val_loss: 0.4051
Epoch 13/30
40/40 - 17s - 419ms/step - binary_accuracy: 0.9230 - dice_coefficient: 0.7953 - iou_coefficient: 0.6633 - loss: 0.4122 - val_binary_accuracy: 0.9171 - val_dice_coefficient: 0.7901 - val_iou_coefficient: 0.6552 - val_loss: 0.4248
Epoch 14/30
40/40 - 16s - 412ms/step - binary_accuracy: 0.9221 - dice_coefficient: 0.7930 - iou_coefficient: 0.6611 - loss: 0.4214 - val_binary_accuracy: 0.9248 - val_dice_coefficient: 0.8179 - val_iou_coefficient: 0.6949 - val_loss: 0.3826
Epoch 15/30
40/40 - 16s - 411ms/step - binary_accuracy: 0.9281 - dice_coefficient: 0.8142 - iou_coefficient: 0.6888 - loss: 0.3773 - val_binary_accuracy: 0.9157 - val_dice_coefficient: 0.7978 - val_iou_coefficient: 0.6659 - val_loss: 0.4273
Epoch 16/30
40/40 - 16s - 411ms/step - binary_accuracy: 0.9255 - dice_coefficient: 0.8018 - iou_coefficient: 0.6721 - loss: 0.3936 - val_binary_accuracy: 0.9336 - val_dice_coefficient: 0.8088 - val_iou_coefficient: 0.6879 - val_loss: 0.3524
Epoch 17/30
40/40 - 17s - 416ms/step - binary_accuracy: 0.9198 - dice_coefficient: 0.7929 - iou_coefficient: 0.6607 - loss: 0.4189 - val_binary_accuracy: 0.8988 - val_dice_coefficient: 0.7623 - val_iou_coefficient: 0.6183 - val_loss: 0.4743
Epoch 18/30
40/40 - 17s - 417ms/step - binary_accuracy: 0.9311 - dice_coefficient: 0.8145 - iou_coefficient: 0.6904 - loss: 0.3751 - val_binary_accuracy: 0.9164 - val_dice_coefficient: 0.8058 - val_iou_coefficient: 0.6784 - val_loss: 0.4033
Epoch 19/30
40/40 - 17s - 419ms/step - binary_accuracy: 0.9311 - dice_coefficient: 0.8211 - iou_coefficient: 0.6985 - loss: 0.3647 - val_binary_accuracy: 0.9328 - val_dice_coefficient: 0.8231 - val_iou_coefficient: 0.7018 - val_loss: 0.3542
Epoch 20/30
40/40 - 17s - 416ms/step - binary_accuracy: 0.9304 - dice_coefficient: 0.8233 - iou_coefficient: 0.7020 - loss: 0.3615 - val_binary_accuracy: 0.9361 - val_dice_coefficient: 0.8465 - val_iou_coefficient: 0.7357 - val_loss: 0.3279
Epoch 21/30
40/40 - 17s - 415ms/step - binary_accuracy: 0.9319 - dice_coefficient: 0.8224 - iou_coefficient: 0.7017 - loss: 0.3591 - val_binary_accuracy: 0.9297 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7296 - val_loss: 0.3475
Epoch 22/30
40/40 - 16s - 411ms/step - binary_accuracy: 0.9348 - dice_coefficient: 0.8304 - iou_coefficient: 0.7124 - loss: 0.3465 - val_binary_accuracy: 0.9348 - val_dice_coefficient: 0.8268 - val_iou_coefficient: 0.7081 - val_loss: 0.3388
Epoch 23/30
40/40 - 17s - 416ms/step - binary_accuracy: 0.9398 - dice_coefficient: 0.8420 - iou_coefficient: 0.7284 - loss: 0.3180 - val_binary_accuracy: 0.9295 - val_dice_coefficient: 0.8239 - val_iou_coefficient: 0.7029 - val_loss: 0.3534
Epoch 24/30
40/40 - 17s - 416ms/step - binary_accuracy: 0.9353 - dice_coefficient: 0.8312 - iou_coefficient: 0.7152 - loss: 0.3442 - val_binary_accuracy: 0.9342 - val_dice_coefficient: 0.8459 - val_iou_coefficient: 0.7343 - val_loss: 0.3291
Epoch 25/30
40/40 - 16s - 411ms/step - binary_accuracy: 0.9361 - dice_coefficient: 0.8360 - iou_coefficient: 0.7203 - loss: 0.3409 - val_binary_accuracy: 0.9376 - val_dice_coefficient: 0.8428 - val_iou_coefficient: 0.7292 - val_loss: 0.3240
Epoch 26/30
40/40 - 17s - 414ms/step - binary_accuracy: 0.9368 - dice_coefficient: 0.8351 - iou_coefficient: 0.7197 - loss: 0.3346 - val_binary_accuracy: 0.9338 - val_dice_coefficient: 0.8265 - val_iou_coefficient: 0.7080 - val_loss: 0.3550
Epoch 27/30
40/40 - 16s - 411ms/step - binary_accuracy: 0.9334 - dice_coefficient: 0.8284 - iou_coefficient: 0.7105 - loss: 0.3483 - val_binary_accuracy: 0.9320 - val_dice_coefficient: 0.8375 - val_iou_coefficient: 0.7228 - val_loss: 0.3410
Epoch 28/30
40/40 - 17s - 418ms/step - binary_accuracy: 0.9381 - dice_coefficient: 0.8395 - iou_coefficient: 0.7257 - loss: 0.3258 - val_binary_accuracy: 0.9345 - val_dice_coefficient: 0.8293 - val_iou_coefficient: 0.7131 - val_loss: 0.3333
Epoch 29/30
40/40 - 17s - 414ms/step - binary_accuracy: 0.9330 - dice_coefficient: 0.8325 - iou_coefficient: 0.7165 - loss: 0.3484 - val_binary_accuracy: 0.9323 - val_dice_coefficient: 0.8456 - val_iou_coefficient: 0.7341 - val_loss: 0.3375
Epoch 30/30
40/40 - 17s - 414ms/step - binary_accuracy: 0.9401 - dice_coefficient: 0.8439 - iou_coefficient: 0.7329 - loss: 0.3186 - val_binary_accuracy: 0.9239 - val_dice_coefficient: 0.8097 - val_iou_coefficient: 0.6858 - val_loss: 0.3744
  → Dice=0.8465  IoU=0.7357  (523s, 30 epochs)

============================================================
  depth=4  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8313 - dice_coefficient: 0.6031 - iou_coefficient: 0.4392 - loss: 0.7980 - val_binary_accuracy: 0.7818 - val_dice_coefficient: 0.3711 - val_iou_coefficient: 0.2304 - val_loss: 1.1329
Epoch 2/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.8849 - dice_coefficient: 0.7125 - iou_coefficient: 0.5588 - loss: 0.5977 - val_binary_accuracy: 0.8215 - val_dice_coefficient: 0.5069 - val_iou_coefficient: 0.3430 - val_loss: 0.9306
Epoch 3/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.8982 - dice_coefficient: 0.7307 - iou_coefficient: 0.5800 - loss: 0.5287 - val_binary_accuracy: 0.8526 - val_dice_coefficient: 0.5943 - val_iou_coefficient: 0.4267 - val_loss: 0.8001
Epoch 4/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9119 - dice_coefficient: 0.7684 - iou_coefficient: 0.6277 - loss: 0.4622 - val_binary_accuracy: 0.8568 - val_dice_coefficient: 0.6480 - val_iou_coefficient: 0.4869 - val_loss: 0.7539
Epoch 5/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9192 - dice_coefficient: 0.7919 - iou_coefficient: 0.6586 - loss: 0.4300 - val_binary_accuracy: 0.8555 - val_dice_coefficient: 0.6675 - val_iou_coefficient: 0.5082 - val_loss: 0.7525
Epoch 6/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9164 - dice_coefficient: 0.7822 - iou_coefficient: 0.6456 - loss: 0.4381 - val_binary_accuracy: 0.9005 - val_dice_coefficient: 0.7554 - val_iou_coefficient: 0.6120 - val_loss: 0.5021
Epoch 7/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9309 - dice_coefficient: 0.8198 - iou_coefficient: 0.6968 - loss: 0.3711 - val_binary_accuracy: 0.9125 - val_dice_coefficient: 0.7662 - val_iou_coefficient: 0.6228 - val_loss: 0.4669
Epoch 8/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9269 - dice_coefficient: 0.8102 - iou_coefficient: 0.6845 - loss: 0.3926 - val_binary_accuracy: 0.9066 - val_dice_coefficient: 0.7890 - val_iou_coefficient: 0.6535 - val_loss: 0.4781
Epoch 9/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9309 - dice_coefficient: 0.8235 - iou_coefficient: 0.7027 - loss: 0.3662 - val_binary_accuracy: 0.9310 - val_dice_coefficient: 0.8212 - val_iou_coefficient: 0.6989 - val_loss: 0.3626
Epoch 10/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9334 - dice_coefficient: 0.8255 - iou_coefficient: 0.7052 - loss: 0.3540 - val_binary_accuracy: 0.9266 - val_dice_coefficient: 0.8315 - val_iou_coefficient: 0.7139 - val_loss: 0.3687
Epoch 11/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9342 - dice_coefficient: 0.8295 - iou_coefficient: 0.7119 - loss: 0.3498 - val_binary_accuracy: 0.9388 - val_dice_coefficient: 0.8588 - val_iou_coefficient: 0.7544 - val_loss: 0.3137
Epoch 12/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9388 - dice_coefficient: 0.8402 - iou_coefficient: 0.7261 - loss: 0.3240 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8480 - val_iou_coefficient: 0.7375 - val_loss: 0.3222
Epoch 13/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9404 - dice_coefficient: 0.8469 - iou_coefficient: 0.7363 - loss: 0.3156 - val_binary_accuracy: 0.9384 - val_dice_coefficient: 0.8484 - val_iou_coefficient: 0.7385 - val_loss: 0.3143
Epoch 14/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9410 - dice_coefficient: 0.8429 - iou_coefficient: 0.7316 - loss: 0.3113 - val_binary_accuracy: 0.9398 - val_dice_coefficient: 0.8592 - val_iou_coefficient: 0.7545 - val_loss: 0.3100
Epoch 15/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9391 - dice_coefficient: 0.8421 - iou_coefficient: 0.7300 - loss: 0.3199 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8399 - val_iou_coefficient: 0.7261 - val_loss: 0.3296
Epoch 16/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8468 - iou_coefficient: 0.7358 - loss: 0.3172 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8498 - val_iou_coefficient: 0.7433 - val_loss: 0.3118
Epoch 17/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9425 - dice_coefficient: 0.8462 - iou_coefficient: 0.7383 - loss: 0.3053 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8592 - val_iou_coefficient: 0.7555 - val_loss: 0.3121
Epoch 18/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9438 - dice_coefficient: 0.8548 - iou_coefficient: 0.7485 - loss: 0.3062 - val_binary_accuracy: 0.9403 - val_dice_coefficient: 0.8453 - val_iou_coefficient: 0.7358 - val_loss: 0.3014
Epoch 19/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9492 - dice_coefficient: 0.8666 - iou_coefficient: 0.7661 - loss: 0.2688 - val_binary_accuracy: 0.9366 - val_dice_coefficient: 0.8521 - val_iou_coefficient: 0.7451 - val_loss: 0.3212
Epoch 20/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9442 - dice_coefficient: 0.8611 - iou_coefficient: 0.7579 - loss: 0.2870 - val_binary_accuracy: 0.9407 - val_dice_coefficient: 0.8574 - val_iou_coefficient: 0.7517 - val_loss: 0.2991
Epoch 21/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9494 - dice_coefficient: 0.8665 - iou_coefficient: 0.7663 - loss: 0.2682 - val_binary_accuracy: 0.9445 - val_dice_coefficient: 0.8663 - val_iou_coefficient: 0.7660 - val_loss: 0.2873
Epoch 22/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9490 - dice_coefficient: 0.8684 - iou_coefficient: 0.7688 - loss: 0.2699 - val_binary_accuracy: 0.9432 - val_dice_coefficient: 0.8701 - val_iou_coefficient: 0.7718 - val_loss: 0.2845
Epoch 23/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9497 - dice_coefficient: 0.8703 - iou_coefficient: 0.7725 - loss: 0.2691 - val_binary_accuracy: 0.9431 - val_dice_coefficient: 0.8609 - val_iou_coefficient: 0.7587 - val_loss: 0.2886
Epoch 24/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9515 - dice_coefficient: 0.8761 - iou_coefficient: 0.7809 - loss: 0.2538 - val_binary_accuracy: 0.9337 - val_dice_coefficient: 0.8545 - val_iou_coefficient: 0.7468 - val_loss: 0.3155
Epoch 25/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9517 - dice_coefficient: 0.8755 - iou_coefficient: 0.7802 - loss: 0.2528 - val_binary_accuracy: 0.9396 - val_dice_coefficient: 0.8507 - val_iou_coefficient: 0.7439 - val_loss: 0.3075
Epoch 26/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9495 - dice_coefficient: 0.8742 - iou_coefficient: 0.7783 - loss: 0.2572 - val_binary_accuracy: 0.9297 - val_dice_coefficient: 0.8390 - val_iou_coefficient: 0.7252 - val_loss: 0.3504
Epoch 27/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9461 - dice_coefficient: 0.8597 - iou_coefficient: 0.7559 - loss: 0.2853 - val_binary_accuracy: 0.9439 - val_dice_coefficient: 0.8674 - val_iou_coefficient: 0.7697 - val_loss: 0.2834
Epoch 28/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9494 - dice_coefficient: 0.8746 - iou_coefficient: 0.7783 - loss: 0.2626 - val_binary_accuracy: 0.9403 - val_dice_coefficient: 0.8706 - val_iou_coefficient: 0.7728 - val_loss: 0.2929
Epoch 29/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9461 - dice_coefficient: 0.8654 - iou_coefficient: 0.7650 - loss: 0.2834 - val_binary_accuracy: 0.9286 - val_dice_coefficient: 0.8513 - val_iou_coefficient: 0.7427 - val_loss: 0.3712
Epoch 30/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9510 - dice_coefficient: 0.8732 - iou_coefficient: 0.7770 - loss: 0.2574 - val_binary_accuracy: 0.9402 - val_dice_coefficient: 0.8710 - val_iou_coefficient: 0.7722 - val_loss: 0.2939
  → Dice=0.8710  IoU=0.7728  (597s, 30 epochs)

============================================================
  depth=5  |  params: 125,805,057
============================================================
Epoch 1/30
40/40 - 68s - 2s/step - binary_accuracy: 0.8255 - dice_coefficient: 0.6035 - iou_coefficient: 0.4385 - loss: 0.8080 - val_binary_accuracy: 0.7772 - val_dice_coefficient: 0.4937 - val_iou_coefficient: 0.3297 - val_loss: 1.1152
Epoch 2/30
40/40 - 22s - 539ms/step - binary_accuracy: 0.8914 - dice_coefficient: 0.7031 - iou_coefficient: 0.5479 - loss: 0.5865 - val_binary_accuracy: 0.7671 - val_dice_coefficient: 0.3741 - val_iou_coefficient: 0.2324 - val_loss: 1.1164
Epoch 3/30
40/40 - 22s - 550ms/step - binary_accuracy: 0.9047 - dice_coefficient: 0.7497 - iou_coefficient: 0.6033 - loss: 0.5062 - val_binary_accuracy: 0.8619 - val_dice_coefficient: 0.5946 - val_iou_coefficient: 0.4338 - val_loss: 0.7847
Epoch 4/30
40/40 - 22s - 545ms/step - binary_accuracy: 0.9162 - dice_coefficient: 0.7684 - iou_coefficient: 0.6268 - loss: 0.4630 - val_binary_accuracy: 0.8801 - val_dice_coefficient: 0.7207 - val_iou_coefficient: 0.5675 - val_loss: 0.6539
Epoch 5/30
40/40 - 22s - 551ms/step - binary_accuracy: 0.9216 - dice_coefficient: 0.7859 - iou_coefficient: 0.6510 - loss: 0.4338 - val_binary_accuracy: 0.8969 - val_dice_coefficient: 0.7450 - val_iou_coefficient: 0.5981 - val_loss: 0.5545
Epoch 6/30
40/40 - 22s - 552ms/step - binary_accuracy: 0.9292 - dice_coefficient: 0.8055 - iou_coefficient: 0.6779 - loss: 0.3871 - val_binary_accuracy: 0.9093 - val_dice_coefficient: 0.7466 - val_iou_coefficient: 0.6079 - val_loss: 0.5005
Epoch 7/30
40/40 - 22s - 539ms/step - binary_accuracy: 0.9361 - dice_coefficient: 0.8182 - iou_coefficient: 0.6960 - loss: 0.3571 - val_binary_accuracy: 0.8990 - val_dice_coefficient: 0.7377 - val_iou_coefficient: 0.5870 - val_loss: 0.5351
Epoch 8/30
40/40 - 22s - 548ms/step - binary_accuracy: 0.9318 - dice_coefficient: 0.8089 - iou_coefficient: 0.6852 - loss: 0.3765 - val_binary_accuracy: 0.9206 - val_dice_coefficient: 0.8152 - val_iou_coefficient: 0.6904 - val_loss: 0.4904
Epoch 9/30
40/40 - 22s - 553ms/step - binary_accuracy: 0.9330 - dice_coefficient: 0.8201 - iou_coefficient: 0.6975 - loss: 0.3638 - val_binary_accuracy: 0.9263 - val_dice_coefficient: 0.8244 - val_iou_coefficient: 0.7070 - val_loss: 0.3733
Epoch 10/30
40/40 - 22s - 542ms/step - binary_accuracy: 0.9325 - dice_coefficient: 0.8243 - iou_coefficient: 0.7042 - loss: 0.3585 - val_binary_accuracy: 0.9025 - val_dice_coefficient: 0.7546 - val_iou_coefficient: 0.6111 - val_loss: 0.5137
Epoch 11/30
40/40 - 22s - 546ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8415 - iou_coefficient: 0.7292 - loss: 0.3155 - val_binary_accuracy: 0.9469 - val_dice_coefficient: 0.8635 - val_iou_coefficient: 0.7622 - val_loss: 0.2828
Epoch 12/30
40/40 - 21s - 537ms/step - binary_accuracy: 0.9400 - dice_coefficient: 0.8425 - iou_coefficient: 0.7298 - loss: 0.3172 - val_binary_accuracy: 0.9333 - val_dice_coefficient: 0.8433 - val_iou_coefficient: 0.7321 - val_loss: 0.3574
Epoch 13/30
40/40 - 21s - 533ms/step - binary_accuracy: 0.9454 - dice_coefficient: 0.8508 - iou_coefficient: 0.7426 - loss: 0.3013 - val_binary_accuracy: 0.9419 - val_dice_coefficient: 0.8525 - val_iou_coefficient: 0.7449 - val_loss: 0.3038
Epoch 14/30
40/40 - 21s - 536ms/step - binary_accuracy: 0.9470 - dice_coefficient: 0.8562 - iou_coefficient: 0.7508 - loss: 0.2867 - val_binary_accuracy: 0.9384 - val_dice_coefficient: 0.8553 - val_iou_coefficient: 0.7492 - val_loss: 0.3126
Epoch 15/30
40/40 - 22s - 546ms/step - binary_accuracy: 0.9470 - dice_coefficient: 0.8591 - iou_coefficient: 0.7547 - loss: 0.2844 - val_binary_accuracy: 0.9461 - val_dice_coefficient: 0.8668 - val_iou_coefficient: 0.7654 - val_loss: 0.2764
Epoch 16/30
40/40 - 21s - 536ms/step - binary_accuracy: 0.9478 - dice_coefficient: 0.8533 - iou_coefficient: 0.7504 - loss: 0.2804 - val_binary_accuracy: 0.9397 - val_dice_coefficient: 0.8573 - val_iou_coefficient: 0.7526 - val_loss: 0.3286
Epoch 17/30
40/40 - 21s - 532ms/step - binary_accuracy: 0.9459 - dice_coefficient: 0.8593 - iou_coefficient: 0.7546 - loss: 0.2865 - val_binary_accuracy: 0.8992 - val_dice_coefficient: 0.8075 - val_iou_coefficient: 0.6815 - val_loss: 0.5631
Epoch 18/30
40/40 - 21s - 536ms/step - binary_accuracy: 0.9466 - dice_coefficient: 0.8561 - iou_coefficient: 0.7528 - loss: 0.2863 - val_binary_accuracy: 0.9370 - val_dice_coefficient: 0.8529 - val_iou_coefficient: 0.7466 - val_loss: 0.3086
Epoch 19/30
40/40 - 21s - 535ms/step - binary_accuracy: 0.9457 - dice_coefficient: 0.8558 - iou_coefficient: 0.7505 - loss: 0.2905 - val_binary_accuracy: 0.9392 - val_dice_coefficient: 0.8578 - val_iou_coefficient: 0.7536 - val_loss: 0.3118
Epoch 20/30
40/40 - 21s - 536ms/step - binary_accuracy: 0.9526 - dice_coefficient: 0.8761 - iou_coefficient: 0.7809 - loss: 0.2508 - val_binary_accuracy: 0.9423 - val_dice_coefficient: 0.8763 - val_iou_coefficient: 0.7812 - val_loss: 0.2784
Epoch 21/30
40/40 - 21s - 535ms/step - binary_accuracy: 0.9522 - dice_coefficient: 0.8727 - iou_coefficient: 0.7753 - loss: 0.2548 - val_binary_accuracy: 0.9394 - val_dice_coefficient: 0.8470 - val_iou_coefficient: 0.7360 - val_loss: 0.3186
Epoch 22/30
40/40 - 22s - 539ms/step - binary_accuracy: 0.9547 - dice_coefficient: 0.8823 - iou_coefficient: 0.7909 - loss: 0.2401 - val_binary_accuracy: 0.9432 - val_dice_coefficient: 0.8585 - val_iou_coefficient: 0.7536 - val_loss: 0.2912
Epoch 23/30
40/40 - 21s - 533ms/step - binary_accuracy: 0.9562 - dice_coefficient: 0.8854 - iou_coefficient: 0.7954 - loss: 0.2332 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8534 - val_iou_coefficient: 0.7458 - val_loss: 0.3254
  → Dice=0.8763  IoU=0.7812  (544s, 23 epochs)

============================================================
</pre>


### Récap `depth`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `depth=3` | 7,794,177 | 0.8465 | 0.7357 | 523s, 30 epochs |
| `depth=4` | 31,402,497 | 0.8710 | 0.7728 | 597s, 30 epochs |
| `depth=5` | 125,805,057 | 0.8763 | 0.7812 | 544s, 23 epochs |

**Meilleur:** `depth=5` (Dice=0.8763, IoU=0.7812).


## 3) Expérience : rôle des skip connections

**Ce qu'on teste :** modèle **avec** puis **sans** connexions skip.

Objectif : quantifier l'apport des connexions encodeur-décodeur pour récupérer les détails spatiaux fins.


In [ ]:
# ── Exp 3 : skip connections (avec / sans) ──

exp_skip = []
for skip in [True, False]:
    tag = 'skip' if skip else 'no_skip'
    r = run_experiment(
        tag,
        build_unet(base_filters=64, depth=4, use_skip=skip),
        train_ds, val_ds
    )
    exp_skip.append(r)

plot_compare(exp_skip, 'skip connections')

### Logs `skip`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  skip  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8240 - dice_coefficient: 0.5991 - iou_coefficient: 0.4335 - loss: 0.8417 - val_binary_accuracy: 0.7153 - val_dice_coefficient: 0.2361 - val_iou_coefficient: 0.1351 - val_loss: 1.4865
Epoch 2/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.8815 - dice_coefficient: 0.6881 - iou_coefficient: 0.5294 - loss: 0.6330 - val_binary_accuracy: 0.8009 - val_dice_coefficient: 0.4348 - val_iou_coefficient: 0.2813 - val_loss: 1.0646
Epoch 3/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9080 - dice_coefficient: 0.7428 - iou_coefficient: 0.5942 - loss: 0.5099 - val_binary_accuracy: 0.8678 - val_dice_coefficient: 0.6599 - val_iou_coefficient: 0.4962 - val_loss: 0.7271
Epoch 4/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9086 - dice_coefficient: 0.7570 - iou_coefficient: 0.6122 - loss: 0.4928 - val_binary_accuracy: 0.8887 - val_dice_coefficient: 0.6759 - val_iou_coefficient: 0.5154 - val_loss: 0.6560
Epoch 5/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9122 - dice_coefficient: 0.7687 - iou_coefficient: 0.6303 - loss: 0.4610 - val_binary_accuracy: 0.8846 - val_dice_coefficient: 0.7112 - val_iou_coefficient: 0.5580 - val_loss: 0.6227
Epoch 6/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9235 - dice_coefficient: 0.7808 - iou_coefficient: 0.6439 - loss: 0.4301 - val_binary_accuracy: 0.9122 - val_dice_coefficient: 0.7521 - val_iou_coefficient: 0.6190 - val_loss: 0.4744
Epoch 7/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9200 - dice_coefficient: 0.7874 - iou_coefficient: 0.6527 - loss: 0.4261 - val_binary_accuracy: 0.9092 - val_dice_coefficient: 0.7841 - val_iou_coefficient: 0.6488 - val_loss: 0.4855
Epoch 8/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9276 - dice_coefficient: 0.8106 - iou_coefficient: 0.6835 - loss: 0.3823 - val_binary_accuracy: 0.8963 - val_dice_coefficient: 0.7673 - val_iou_coefficient: 0.6259 - val_loss: 0.5338
Epoch 9/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9314 - dice_coefficient: 0.8143 - iou_coefficient: 0.6889 - loss: 0.3698 - val_binary_accuracy: 0.9120 - val_dice_coefficient: 0.7828 - val_iou_coefficient: 0.6499 - val_loss: 0.4486
Epoch 10/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9317 - dice_coefficient: 0.8196 - iou_coefficient: 0.6963 - loss: 0.3689 - val_binary_accuracy: 0.9337 - val_dice_coefficient: 0.8252 - val_iou_coefficient: 0.7057 - val_loss: 0.3495
Epoch 11/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9322 - dice_coefficient: 0.8223 - iou_coefficient: 0.7014 - loss: 0.3647 - val_binary_accuracy: 0.9204 - val_dice_coefficient: 0.8228 - val_iou_coefficient: 0.7044 - val_loss: 0.3963
Epoch 12/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9350 - dice_coefficient: 0.8275 - iou_coefficient: 0.7075 - loss: 0.3490 - val_binary_accuracy: 0.9216 - val_dice_coefficient: 0.8179 - val_iou_coefficient: 0.6983 - val_loss: 0.4018
Epoch 13/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9362 - dice_coefficient: 0.8323 - iou_coefficient: 0.7148 - loss: 0.3429 - val_binary_accuracy: 0.9364 - val_dice_coefficient: 0.8396 - val_iou_coefficient: 0.7252 - val_loss: 0.3357
Epoch 14/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9384 - dice_coefficient: 0.8351 - iou_coefficient: 0.7193 - loss: 0.3336 - val_binary_accuracy: 0.9427 - val_dice_coefficient: 0.8614 - val_iou_coefficient: 0.7582 - val_loss: 0.2979
Epoch 15/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8318 - iou_coefficient: 0.7163 - loss: 0.3385 - val_binary_accuracy: 0.9370 - val_dice_coefficient: 0.8274 - val_iou_coefficient: 0.7124 - val_loss: 0.3322
Epoch 16/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9409 - dice_coefficient: 0.8410 - iou_coefficient: 0.7281 - loss: 0.3198 - val_binary_accuracy: 0.9285 - val_dice_coefficient: 0.8231 - val_iou_coefficient: 0.7021 - val_loss: 0.3722
Epoch 17/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9386 - dice_coefficient: 0.8400 - iou_coefficient: 0.7257 - loss: 0.3265 - val_binary_accuracy: 0.9414 - val_dice_coefficient: 0.8432 - val_iou_coefficient: 0.7309 - val_loss: 0.3077
Epoch 18/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9449 - dice_coefficient: 0.8518 - iou_coefficient: 0.7455 - loss: 0.2999 - val_binary_accuracy: 0.9418 - val_dice_coefficient: 0.8581 - val_iou_coefficient: 0.7537 - val_loss: 0.3029
Epoch 19/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9344 - dice_coefficient: 0.8352 - iou_coefficient: 0.7201 - loss: 0.3409 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8338 - val_iou_coefficient: 0.7173 - val_loss: 0.3354
Epoch 20/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9409 - dice_coefficient: 0.8457 - iou_coefficient: 0.7355 - loss: 0.3162 - val_binary_accuracy: 0.9419 - val_dice_coefficient: 0.8635 - val_iou_coefficient: 0.7619 - val_loss: 0.2979
Epoch 21/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9468 - dice_coefficient: 0.8610 - iou_coefficient: 0.7580 - loss: 0.2831 - val_binary_accuracy: 0.9440 - val_dice_coefficient: 0.8671 - val_iou_coefficient: 0.7669 - val_loss: 0.2873
Epoch 22/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9451 - dice_coefficient: 0.8564 - iou_coefficient: 0.7505 - loss: 0.2949 - val_binary_accuracy: 0.9410 - val_dice_coefficient: 0.8619 - val_iou_coefficient: 0.7594 - val_loss: 0.3101
Epoch 23/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9487 - dice_coefficient: 0.8619 - iou_coefficient: 0.7601 - loss: 0.2740 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8672 - val_iou_coefficient: 0.7667 - val_loss: 0.3056
Epoch 24/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9487 - dice_coefficient: 0.8688 - iou_coefficient: 0.7694 - loss: 0.2705 - val_binary_accuracy: 0.9204 - val_dice_coefficient: 0.8338 - val_iou_coefficient: 0.7207 - val_loss: 0.3730
Epoch 25/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9472 - dice_coefficient: 0.8644 - iou_coefficient: 0.7628 - loss: 0.2793 - val_binary_accuracy: 0.9420 - val_dice_coefficient: 0.8553 - val_iou_coefficient: 0.7487 - val_loss: 0.3003
Epoch 26/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9509 - dice_coefficient: 0.8706 - iou_coefficient: 0.7731 - loss: 0.2557 - val_binary_accuracy: 0.9386 - val_dice_coefficient: 0.8620 - val_iou_coefficient: 0.7592 - val_loss: 0.3054
Epoch 27/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9462 - dice_coefficient: 0.8616 - iou_coefficient: 0.7590 - loss: 0.2851 - val_binary_accuracy: 0.9388 - val_dice_coefficient: 0.8610 - val_iou_coefficient: 0.7574 - val_loss: 0.3071
Epoch 28/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9488 - dice_coefficient: 0.8694 - iou_coefficient: 0.7706 - loss: 0.2652 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8584 - val_iou_coefficient: 0.7537 - val_loss: 0.3031
Epoch 29/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9479 - dice_coefficient: 0.8666 - iou_coefficient: 0.7682 - loss: 0.2782 - val_binary_accuracy: 0.9370 - val_dice_coefficient: 0.8640 - val_iou_coefficient: 0.7615 - val_loss: 0.3068
  → Dice=0.8672  IoU=0.7669  (579s, 29 epochs)

============================================================
  no_skip  |  params: 28,269,057
============================================================
Epoch 1/30
40/40 - 59s - 1s/step - binary_accuracy: 0.8396 - dice_coefficient: 0.6121 - iou_coefficient: 0.4474 - loss: 0.7791 - val_binary_accuracy: 0.7546 - val_dice_coefficient: 0.3445 - val_iou_coefficient: 0.2100 - val_loss: 1.2255
Epoch 2/30
40/40 - 17s - 437ms/step - binary_accuracy: 0.8855 - dice_coefficient: 0.7177 - iou_coefficient: 0.5633 - loss: 0.5766 - val_binary_accuracy: 0.7848 - val_dice_coefficient: 0.4184 - val_iou_coefficient: 0.2663 - val_loss: 1.1180
Epoch 3/30
40/40 - 18s - 443ms/step - binary_accuracy: 0.9075 - dice_coefficient: 0.7633 - iou_coefficient: 0.6215 - loss: 0.4786 - val_binary_accuracy: 0.8141 - val_dice_coefficient: 0.5191 - val_iou_coefficient: 0.3572 - val_loss: 1.0477
Epoch 4/30
40/40 - 18s - 444ms/step - binary_accuracy: 0.9108 - dice_coefficient: 0.7798 - iou_coefficient: 0.6422 - loss: 0.4511 - val_binary_accuracy: 0.8740 - val_dice_coefficient: 0.6882 - val_iou_coefficient: 0.5278 - val_loss: 0.6805
Epoch 5/30
40/40 - 18s - 444ms/step - binary_accuracy: 0.9219 - dice_coefficient: 0.7950 - iou_coefficient: 0.6637 - loss: 0.4119 - val_binary_accuracy: 0.8725 - val_dice_coefficient: 0.7082 - val_iou_coefficient: 0.5540 - val_loss: 0.6534
Epoch 6/30
40/40 - 17s - 434ms/step - binary_accuracy: 0.9266 - dice_coefficient: 0.8128 - iou_coefficient: 0.6865 - loss: 0.3842 - val_binary_accuracy: 0.8956 - val_dice_coefficient: 0.7737 - val_iou_coefficient: 0.6362 - val_loss: 0.5506
Epoch 7/30
40/40 - 18s - 441ms/step - binary_accuracy: 0.9240 - dice_coefficient: 0.8070 - iou_coefficient: 0.6791 - loss: 0.3922 - val_binary_accuracy: 0.9213 - val_dice_coefficient: 0.7991 - val_iou_coefficient: 0.6675 - val_loss: 0.4068
Epoch 8/30
40/40 - 18s - 440ms/step - binary_accuracy: 0.9238 - dice_coefficient: 0.8114 - iou_coefficient: 0.6854 - loss: 0.3927 - val_binary_accuracy: 0.9212 - val_dice_coefficient: 0.8079 - val_iou_coefficient: 0.6798 - val_loss: 0.3950
Epoch 9/30
40/40 - 17s - 430ms/step - binary_accuracy: 0.9296 - dice_coefficient: 0.8218 - iou_coefficient: 0.7008 - loss: 0.3686 - val_binary_accuracy: 0.9124 - val_dice_coefficient: 0.7711 - val_iou_coefficient: 0.6411 - val_loss: 0.4537
Epoch 10/30
40/40 - 17s - 432ms/step - binary_accuracy: 0.9368 - dice_coefficient: 0.8348 - iou_coefficient: 0.7185 - loss: 0.3332 - val_binary_accuracy: 0.9350 - val_dice_coefficient: 0.8360 - val_iou_coefficient: 0.7226 - val_loss: 0.3433
Epoch 11/30
40/40 - 17s - 433ms/step - binary_accuracy: 0.9395 - dice_coefficient: 0.8465 - iou_coefficient: 0.7367 - loss: 0.3158 - val_binary_accuracy: 0.9269 - val_dice_coefficient: 0.8331 - val_iou_coefficient: 0.7171 - val_loss: 0.3731
Epoch 12/30
40/40 - 18s - 445ms/step - binary_accuracy: 0.9437 - dice_coefficient: 0.8555 - iou_coefficient: 0.7496 - loss: 0.2943 - val_binary_accuracy: 0.9356 - val_dice_coefficient: 0.8488 - val_iou_coefficient: 0.7387 - val_loss: 0.3205
Epoch 13/30
40/40 - 17s - 436ms/step - binary_accuracy: 0.9364 - dice_coefficient: 0.8309 - iou_coefficient: 0.7146 - loss: 0.3371 - val_binary_accuracy: 0.9373 - val_dice_coefficient: 0.8460 - val_iou_coefficient: 0.7350 - val_loss: 0.3098
Epoch 14/30
40/40 - 17s - 430ms/step - binary_accuracy: 0.9387 - dice_coefficient: 0.8461 - iou_coefficient: 0.7354 - loss: 0.3190 - val_binary_accuracy: 0.9364 - val_dice_coefficient: 0.8389 - val_iou_coefficient: 0.7240 - val_loss: 0.3236
Epoch 15/30
40/40 - 17s - 435ms/step - binary_accuracy: 0.9391 - dice_coefficient: 0.8458 - iou_coefficient: 0.7345 - loss: 0.3233 - val_binary_accuracy: 0.9314 - val_dice_coefficient: 0.8493 - val_iou_coefficient: 0.7399 - val_loss: 0.3333
Epoch 16/30
40/40 - 18s - 444ms/step - binary_accuracy: 0.9422 - dice_coefficient: 0.8527 - iou_coefficient: 0.7453 - loss: 0.3020 - val_binary_accuracy: 0.9399 - val_dice_coefficient: 0.8661 - val_iou_coefficient: 0.7655 - val_loss: 0.3007
Epoch 17/30
40/40 - 18s - 441ms/step - binary_accuracy: 0.9455 - dice_coefficient: 0.8597 - iou_coefficient: 0.7558 - loss: 0.2871 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8618 - val_iou_coefficient: 0.7578 - val_loss: 0.2913
Epoch 18/30
40/40 - 17s - 432ms/step - binary_accuracy: 0.9480 - dice_coefficient: 0.8725 - iou_coefficient: 0.7752 - loss: 0.2712 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8522 - val_iou_coefficient: 0.7463 - val_loss: 0.3065
Epoch 19/30
40/40 - 17s - 432ms/step - binary_accuracy: 0.9431 - dice_coefficient: 0.8547 - iou_coefficient: 0.7496 - loss: 0.2959 - val_binary_accuracy: 0.9384 - val_dice_coefficient: 0.8620 - val_iou_coefficient: 0.7596 - val_loss: 0.3021
Epoch 20/30
40/40 - 18s - 438ms/step - binary_accuracy: 0.9448 - dice_coefficient: 0.8609 - iou_coefficient: 0.7574 - loss: 0.2850 - val_binary_accuracy: 0.9291 - val_dice_coefficient: 0.8376 - val_iou_coefficient: 0.7237 - val_loss: 0.3577
Epoch 21/30
40/40 - 17s - 434ms/step - binary_accuracy: 0.9457 - dice_coefficient: 0.8669 - iou_coefficient: 0.7664 - loss: 0.2765 - val_binary_accuracy: 0.9299 - val_dice_coefficient: 0.8396 - val_iou_coefficient: 0.7269 - val_loss: 0.3580
Epoch 22/30
40/40 - 17s - 435ms/step - binary_accuracy: 0.9447 - dice_coefficient: 0.8568 - iou_coefficient: 0.7519 - loss: 0.2899 - val_binary_accuracy: 0.9456 - val_dice_coefficient: 0.8618 - val_iou_coefficient: 0.7592 - val_loss: 0.2935
Epoch 23/30
40/40 - 18s - 438ms/step - binary_accuracy: 0.9460 - dice_coefficient: 0.8565 - iou_coefficient: 0.7530 - loss: 0.2866 - val_binary_accuracy: 0.9452 - val_dice_coefficient: 0.8770 - val_iou_coefficient: 0.7842 - val_loss: 0.2732
Epoch 24/30
40/40 - 18s - 444ms/step - binary_accuracy: 0.9511 - dice_coefficient: 0.8751 - iou_coefficient: 0.7795 - loss: 0.2596 - val_binary_accuracy: 0.9421 - val_dice_coefficient: 0.8601 - val_iou_coefficient: 0.7558 - val_loss: 0.2936
Epoch 25/30
40/40 - 18s - 442ms/step - binary_accuracy: 0.9508 - dice_coefficient: 0.8755 - iou_coefficient: 0.7801 - loss: 0.2536 - val_binary_accuracy: 0.9428 - val_dice_coefficient: 0.8677 - val_iou_coefficient: 0.7674 - val_loss: 0.2882
Epoch 26/30
40/40 - 17s - 437ms/step - binary_accuracy: 0.9537 - dice_coefficient: 0.8850 - iou_coefficient: 0.7954 - loss: 0.2421 - val_binary_accuracy: 0.9398 - val_dice_coefficient: 0.8770 - val_iou_coefficient: 0.7828 - val_loss: 0.2886
Epoch 27/30
40/40 - 18s - 438ms/step - binary_accuracy: 0.9504 - dice_coefficient: 0.8772 - iou_coefficient: 0.7836 - loss: 0.2573 - val_binary_accuracy: 0.9438 - val_dice_coefficient: 0.8735 - val_iou_coefficient: 0.7774 - val_loss: 0.2818
Epoch 28/30
40/40 - 17s - 437ms/step - binary_accuracy: 0.9532 - dice_coefficient: 0.8829 - iou_coefficient: 0.7918 - loss: 0.2430 - val_binary_accuracy: 0.9435 - val_dice_coefficient: 0.8704 - val_iou_coefficient: 0.7724 - val_loss: 0.2842
Epoch 29/30
40/40 - 17s - 437ms/step - binary_accuracy: 0.9541 - dice_coefficient: 0.8851 - iou_coefficient: 0.7955 - loss: 0.2375 - val_binary_accuracy: 0.9402 - val_dice_coefficient: 0.8671 - val_iou_coefficient: 0.7679 - val_loss: 0.3034
Epoch 30/30
40/40 - 18s - 438ms/step - binary_accuracy: 0.9548 - dice_coefficient: 0.8850 - iou_coefficient: 0.7956 - loss: 0.2340 - val_binary_accuracy: 0.9426 - val_dice_coefficient: 0.8692 - val_iou_coefficient: 0.7711 - val_loss: 0.2884
  → Dice=0.8770  IoU=0.7842  (567s, 30 epochs)

============================================================
</pre>


### Récap `skip`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `skip` | 31,402,497 | 0.8672 | 0.7669 | 579s, 29 epochs |
| `no_skip` | 28,269,057 | 0.8770 | 0.7842 | 567s, 30 epochs |

**Meilleur:** `no_skip` (Dice=0.8770, IoU=0.7842).


## 4) Expérience : fonction de coût

**Ce qu'on teste :** `BCE`, `Dice`, `BCE+Dice`, `IoU`.

Objectif : comparer quelles losses optimisent le mieux la segmentation sur ce dataset,
en gardant la même architecture de base.


In [ ]:
# ── Exp 4 : loss (BCE / Dice / BCE+Dice / IoU) ──

loss_fns = {
    'bce':       tf.keras.losses.binary_crossentropy,
    'dice':      dice_loss,
    'bce+dice':  bce_dice_loss,
    'iou':       iou_loss,
}

exp_loss = []
for name, lfn in loss_fns.items():
    r = run_experiment(
        f'loss={name}',
        build_unet(base_filters=64, depth=4, use_skip=True),
        train_ds, val_ds,
        loss_fn=lfn
    )
    exp_loss.append(r)

plot_compare(exp_loss, 'loss function')

### Logs `loss`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  loss=bce  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8417 - dice_coefficient: 0.5683 - iou_coefficient: 0.4011 - loss: 0.4118 - val_binary_accuracy: 0.7276 - val_dice_coefficient: 0.3417 - val_iou_coefficient: 0.2078 - val_loss: 0.5554
Epoch 2/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.8845 - dice_coefficient: 0.6626 - iou_coefficient: 0.4980 - loss: 0.2962 - val_binary_accuracy: 0.7406 - val_dice_coefficient: 0.3306 - val_iou_coefficient: 0.1999 - val_loss: 0.5347
Epoch 3/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9031 - dice_coefficient: 0.6963 - iou_coefficient: 0.5375 - loss: 0.2642 - val_binary_accuracy: 0.7955 - val_dice_coefficient: 0.4327 - val_iou_coefficient: 0.2792 - val_loss: 0.4736
Epoch 4/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9087 - dice_coefficient: 0.7256 - iou_coefficient: 0.5723 - loss: 0.2501 - val_binary_accuracy: 0.8162 - val_dice_coefficient: 0.5064 - val_iou_coefficient: 0.3488 - val_loss: 0.4860
Epoch 5/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9103 - dice_coefficient: 0.7291 - iou_coefficient: 0.5766 - loss: 0.2402 - val_binary_accuracy: 0.8591 - val_dice_coefficient: 0.6385 - val_iou_coefficient: 0.4740 - val_loss: 0.3875
Epoch 6/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9084 - dice_coefficient: 0.7338 - iou_coefficient: 0.5825 - loss: 0.2359 - val_binary_accuracy: 0.9111 - val_dice_coefficient: 0.7501 - val_iou_coefficient: 0.6059 - val_loss: 0.2366
Epoch 7/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9265 - dice_coefficient: 0.7710 - iou_coefficient: 0.6299 - loss: 0.1969 - val_binary_accuracy: 0.9148 - val_dice_coefficient: 0.7526 - val_iou_coefficient: 0.6088 - val_loss: 0.2340
Epoch 8/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9313 - dice_coefficient: 0.7818 - iou_coefficient: 0.6445 - loss: 0.1851 - val_binary_accuracy: 0.9054 - val_dice_coefficient: 0.7484 - val_iou_coefficient: 0.6051 - val_loss: 0.2388
Epoch 9/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9290 - dice_coefficient: 0.7822 - iou_coefficient: 0.6451 - loss: 0.1888 - val_binary_accuracy: 0.9241 - val_dice_coefficient: 0.7913 - val_iou_coefficient: 0.6585 - val_loss: 0.2122
Epoch 10/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9273 - dice_coefficient: 0.7827 - iou_coefficient: 0.6456 - loss: 0.1876 - val_binary_accuracy: 0.9230 - val_dice_coefficient: 0.7660 - val_iou_coefficient: 0.6230 - val_loss: 0.2039
Epoch 11/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9383 - dice_coefficient: 0.8097 - iou_coefficient: 0.6818 - loss: 0.1661 - val_binary_accuracy: 0.9336 - val_dice_coefficient: 0.7756 - val_iou_coefficient: 0.6380 - val_loss: 0.1809
Epoch 12/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9316 - dice_coefficient: 0.7934 - iou_coefficient: 0.6621 - loss: 0.1783 - val_binary_accuracy: 0.9234 - val_dice_coefficient: 0.7991 - val_iou_coefficient: 0.6701 - val_loss: 0.1910
Epoch 13/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9334 - dice_coefficient: 0.7988 - iou_coefficient: 0.6676 - loss: 0.1745 - val_binary_accuracy: 0.9361 - val_dice_coefficient: 0.8342 - val_iou_coefficient: 0.7179 - val_loss: 0.1693
Epoch 14/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9375 - dice_coefficient: 0.8060 - iou_coefficient: 0.6781 - loss: 0.1645 - val_binary_accuracy: 0.9390 - val_dice_coefficient: 0.8392 - val_iou_coefficient: 0.7246 - val_loss: 0.1519
Epoch 15/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9404 - dice_coefficient: 0.8187 - iou_coefficient: 0.6955 - loss: 0.1581 - val_binary_accuracy: 0.9385 - val_dice_coefficient: 0.8342 - val_iou_coefficient: 0.7171 - val_loss: 0.1542
Epoch 16/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9421 - dice_coefficient: 0.8219 - iou_coefficient: 0.6999 - loss: 0.1502 - val_binary_accuracy: 0.9376 - val_dice_coefficient: 0.8060 - val_iou_coefficient: 0.6780 - val_loss: 0.1614
Epoch 17/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9438 - dice_coefficient: 0.8275 - iou_coefficient: 0.7075 - loss: 0.1464 - val_binary_accuracy: 0.9366 - val_dice_coefficient: 0.8316 - val_iou_coefficient: 0.7133 - val_loss: 0.1642
Epoch 18/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9417 - dice_coefficient: 0.8223 - iou_coefficient: 0.7015 - loss: 0.1547 - val_binary_accuracy: 0.9316 - val_dice_coefficient: 0.8255 - val_iou_coefficient: 0.7044 - val_loss: 0.1680
Epoch 19/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8303 - iou_coefficient: 0.7116 - loss: 0.1465 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8478 - val_iou_coefficient: 0.7374 - val_loss: 0.1477
Epoch 20/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9449 - dice_coefficient: 0.8320 - iou_coefficient: 0.7151 - loss: 0.1416 - val_binary_accuracy: 0.9448 - val_dice_coefficient: 0.8362 - val_iou_coefficient: 0.7223 - val_loss: 0.1425
Epoch 21/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9465 - dice_coefficient: 0.8322 - iou_coefficient: 0.7154 - loss: 0.1398 - val_binary_accuracy: 0.9408 - val_dice_coefficient: 0.8566 - val_iou_coefficient: 0.7512 - val_loss: 0.1469
Epoch 22/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9464 - dice_coefficient: 0.8344 - iou_coefficient: 0.7176 - loss: 0.1399 - val_binary_accuracy: 0.9424 - val_dice_coefficient: 0.8503 - val_iou_coefficient: 0.7419 - val_loss: 0.1459
Epoch 23/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9465 - dice_coefficient: 0.8382 - iou_coefficient: 0.7239 - loss: 0.1381 - val_binary_accuracy: 0.9396 - val_dice_coefficient: 0.8391 - val_iou_coefficient: 0.7240 - val_loss: 0.1541
Epoch 24/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9444 - dice_coefficient: 0.8376 - iou_coefficient: 0.7226 - loss: 0.1389 - val_binary_accuracy: 0.9473 - val_dice_coefficient: 0.8600 - val_iou_coefficient: 0.7556 - val_loss: 0.1368
Epoch 25/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9442 - dice_coefficient: 0.8319 - iou_coefficient: 0.7158 - loss: 0.1461 - val_binary_accuracy: 0.9354 - val_dice_coefficient: 0.8374 - val_iou_coefficient: 0.7221 - val_loss: 0.1716
Epoch 26/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9486 - dice_coefficient: 0.8370 - iou_coefficient: 0.7221 - loss: 0.1359 - val_binary_accuracy: 0.9409 - val_dice_coefficient: 0.8287 - val_iou_coefficient: 0.7092 - val_loss: 0.1582
Epoch 27/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9546 - dice_coefficient: 0.8645 - iou_coefficient: 0.7630 - loss: 0.1185 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8391 - val_iou_coefficient: 0.7243 - val_loss: 0.1443
Epoch 28/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9525 - dice_coefficient: 0.8525 - iou_coefficient: 0.7453 - loss: 0.1238 - val_binary_accuracy: 0.9468 - val_dice_coefficient: 0.8472 - val_iou_coefficient: 0.7367 - val_loss: 0.1320
Epoch 29/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9541 - dice_coefficient: 0.8619 - iou_coefficient: 0.7585 - loss: 0.1163 - val_binary_accuracy: 0.9389 - val_dice_coefficient: 0.8264 - val_iou_coefficient: 0.7055 - val_loss: 0.1518
Epoch 30/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9527 - dice_coefficient: 0.8531 - iou_coefficient: 0.7466 - loss: 0.1235 - val_binary_accuracy: 0.9385 - val_dice_coefficient: 0.8420 - val_iou_coefficient: 0.7281 - val_loss: 0.1534
  → Dice=0.8600  IoU=0.7556  (597s, 30 epochs)

============================================================
  loss=dice  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8404 - dice_coefficient: 0.6592 - iou_coefficient: 0.5026 - loss: 0.3423 - val_binary_accuracy: 0.7252 - val_dice_coefficient: 0.2914 - val_iou_coefficient: 0.1715 - val_loss: 0.7075
Epoch 2/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.8707 - dice_coefficient: 0.7419 - iou_coefficient: 0.5967 - loss: 0.2595 - val_binary_accuracy: 0.7295 - val_dice_coefficient: 0.5300 - val_iou_coefficient: 0.3654 - val_loss: 0.4607
Epoch 3/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.8866 - dice_coefficient: 0.7671 - iou_coefficient: 0.6282 - loss: 0.2343 - val_binary_accuracy: 0.8126 - val_dice_coefficient: 0.5325 - val_iou_coefficient: 0.3700 - val_loss: 0.4612
Epoch 4/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9046 - dice_coefficient: 0.8001 - iou_coefficient: 0.6705 - loss: 0.2008 - val_binary_accuracy: 0.8300 - val_dice_coefficient: 0.5780 - val_iou_coefficient: 0.4118 - val_loss: 0.4193
Epoch 5/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9137 - dice_coefficient: 0.8221 - iou_coefficient: 0.7024 - loss: 0.1788 - val_binary_accuracy: 0.8558 - val_dice_coefficient: 0.6890 - val_iou_coefficient: 0.5393 - val_loss: 0.3221
Epoch 6/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9111 - dice_coefficient: 0.8078 - iou_coefficient: 0.6827 - loss: 0.1891 - val_binary_accuracy: 0.8979 - val_dice_coefficient: 0.7681 - val_iou_coefficient: 0.6354 - val_loss: 0.2155
Epoch 7/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9200 - dice_coefficient: 0.8325 - iou_coefficient: 0.7179 - loss: 0.1681 - val_binary_accuracy: 0.8799 - val_dice_coefficient: 0.7757 - val_iou_coefficient: 0.6395 - val_loss: 0.2253
Epoch 8/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9177 - dice_coefficient: 0.8289 - iou_coefficient: 0.7101 - loss: 0.1717 - val_binary_accuracy: 0.9104 - val_dice_coefficient: 0.8167 - val_iou_coefficient: 0.6950 - val_loss: 0.1842
Epoch 9/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9283 - dice_coefficient: 0.8473 - iou_coefficient: 0.7387 - loss: 0.1521 - val_binary_accuracy: 0.9299 - val_dice_coefficient: 0.8651 - val_iou_coefficient: 0.7637 - val_loss: 0.1354
Epoch 10/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9313 - dice_coefficient: 0.8587 - iou_coefficient: 0.7537 - loss: 0.1413 - val_binary_accuracy: 0.9324 - val_dice_coefficient: 0.8545 - val_iou_coefficient: 0.7506 - val_loss: 0.1406
Epoch 11/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9359 - dice_coefficient: 0.8687 - iou_coefficient: 0.7707 - loss: 0.1321 - val_binary_accuracy: 0.9326 - val_dice_coefficient: 0.8692 - val_iou_coefficient: 0.7742 - val_loss: 0.1346
Epoch 12/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9303 - dice_coefficient: 0.8580 - iou_coefficient: 0.7536 - loss: 0.1421 - val_binary_accuracy: 0.9200 - val_dice_coefficient: 0.8589 - val_iou_coefficient: 0.7585 - val_loss: 0.1462
Epoch 13/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9329 - dice_coefficient: 0.8589 - iou_coefficient: 0.7561 - loss: 0.1407 - val_binary_accuracy: 0.9316 - val_dice_coefficient: 0.8459 - val_iou_coefficient: 0.7392 - val_loss: 0.1415
Epoch 14/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9377 - dice_coefficient: 0.8737 - iou_coefficient: 0.7779 - loss: 0.1259 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8902 - val_iou_coefficient: 0.8050 - val_loss: 0.1115
Epoch 15/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9340 - dice_coefficient: 0.8649 - iou_coefficient: 0.7649 - loss: 0.1355 - val_binary_accuracy: 0.9363 - val_dice_coefficient: 0.8756 - val_iou_coefficient: 0.7815 - val_loss: 0.1264
Epoch 16/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9359 - dice_coefficient: 0.8679 - iou_coefficient: 0.7701 - loss: 0.1279 - val_binary_accuracy: 0.9400 - val_dice_coefficient: 0.8857 - val_iou_coefficient: 0.7976 - val_loss: 0.1179
Epoch 17/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9365 - dice_coefficient: 0.8707 - iou_coefficient: 0.7754 - loss: 0.1299 - val_binary_accuracy: 0.9314 - val_dice_coefficient: 0.8626 - val_iou_coefficient: 0.7610 - val_loss: 0.1328
Epoch 18/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9402 - dice_coefficient: 0.8791 - iou_coefficient: 0.7861 - loss: 0.1209 - val_binary_accuracy: 0.9293 - val_dice_coefficient: 0.8582 - val_iou_coefficient: 0.7548 - val_loss: 0.1391
Epoch 19/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9407 - dice_coefficient: 0.8828 - iou_coefficient: 0.7923 - loss: 0.1167 - val_binary_accuracy: 0.9038 - val_dice_coefficient: 0.7997 - val_iou_coefficient: 0.6792 - val_loss: 0.2086
Epoch 20/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9361 - dice_coefficient: 0.8710 - iou_coefficient: 0.7755 - loss: 0.1282 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8832 - val_iou_coefficient: 0.7938 - val_loss: 0.1153
Epoch 21/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9384 - dice_coefficient: 0.8707 - iou_coefficient: 0.7768 - loss: 0.1286 - val_binary_accuracy: 0.9362 - val_dice_coefficient: 0.8849 - val_iou_coefficient: 0.7961 - val_loss: 0.1168
Epoch 22/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9389 - dice_coefficient: 0.8741 - iou_coefficient: 0.7793 - loss: 0.1257 - val_binary_accuracy: 0.9355 - val_dice_coefficient: 0.8734 - val_iou_coefficient: 0.7795 - val_loss: 0.1311
  → Dice=0.8902  IoU=0.8050  (446s, 22 epochs)

============================================================
  loss=bce+dice  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8399 - dice_coefficient: 0.6100 - iou_coefficient: 0.4494 - loss: 0.7859 - val_binary_accuracy: 0.7201 - val_dice_coefficient: 0.2994 - val_iou_coefficient: 0.1767 - val_loss: 1.2673
Epoch 2/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.8920 - dice_coefficient: 0.7185 - iou_coefficient: 0.5636 - loss: 0.5729 - val_binary_accuracy: 0.7634 - val_dice_coefficient: 0.3573 - val_iou_coefficient: 0.2187 - val_loss: 1.1703
Epoch 3/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9068 - dice_coefficient: 0.7588 - iou_coefficient: 0.6137 - loss: 0.4890 - val_binary_accuracy: 0.8283 - val_dice_coefficient: 0.5925 - val_iou_coefficient: 0.4264 - val_loss: 0.9095
Epoch 4/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9113 - dice_coefficient: 0.7715 - iou_coefficient: 0.6319 - loss: 0.4616 - val_binary_accuracy: 0.8206 - val_dice_coefficient: 0.6627 - val_iou_coefficient: 0.4998 - val_loss: 0.8598
Epoch 5/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9220 - dice_coefficient: 0.7948 - iou_coefficient: 0.6624 - loss: 0.4174 - val_binary_accuracy: 0.8853 - val_dice_coefficient: 0.6995 - val_iou_coefficient: 0.5515 - val_loss: 0.6081
Epoch 6/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9205 - dice_coefficient: 0.7938 - iou_coefficient: 0.6625 - loss: 0.4146 - val_binary_accuracy: 0.9082 - val_dice_coefficient: 0.7423 - val_iou_coefficient: 0.5968 - val_loss: 0.5055
Epoch 7/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9253 - dice_coefficient: 0.8003 - iou_coefficient: 0.6713 - loss: 0.3970 - val_binary_accuracy: 0.9100 - val_dice_coefficient: 0.7447 - val_iou_coefficient: 0.6089 - val_loss: 0.4742
Epoch 8/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9274 - dice_coefficient: 0.8098 - iou_coefficient: 0.6838 - loss: 0.3860 - val_binary_accuracy: 0.9068 - val_dice_coefficient: 0.7746 - val_iou_coefficient: 0.6379 - val_loss: 0.4729
Epoch 9/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9337 - dice_coefficient: 0.8248 - iou_coefficient: 0.7062 - loss: 0.3469 - val_binary_accuracy: 0.9308 - val_dice_coefficient: 0.8106 - val_iou_coefficient: 0.6840 - val_loss: 0.3786
Epoch 10/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9308 - dice_coefficient: 0.8218 - iou_coefficient: 0.7022 - loss: 0.3641 - val_binary_accuracy: 0.9165 - val_dice_coefficient: 0.8289 - val_iou_coefficient: 0.7115 - val_loss: 0.4128
Epoch 11/30
40/40 - 19s - 480ms/step - binary_accuracy: 0.9309 - dice_coefficient: 0.8253 - iou_coefficient: 0.7055 - loss: 0.3582 - val_binary_accuracy: 0.9398 - val_dice_coefficient: 0.8626 - val_iou_coefficient: 0.7594 - val_loss: 0.3037
Epoch 12/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9394 - dice_coefficient: 0.8442 - iou_coefficient: 0.7328 - loss: 0.3165 - val_binary_accuracy: 0.9165 - val_dice_coefficient: 0.7974 - val_iou_coefficient: 0.6651 - val_loss: 0.4134
Epoch 13/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9372 - dice_coefficient: 0.8370 - iou_coefficient: 0.7227 - loss: 0.3290 - val_binary_accuracy: 0.9350 - val_dice_coefficient: 0.8307 - val_iou_coefficient: 0.7127 - val_loss: 0.3480
Epoch 14/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9377 - dice_coefficient: 0.8406 - iou_coefficient: 0.7272 - loss: 0.3212 - val_binary_accuracy: 0.9409 - val_dice_coefficient: 0.8494 - val_iou_coefficient: 0.7411 - val_loss: 0.2985
Epoch 15/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9406 - dice_coefficient: 0.8474 - iou_coefficient: 0.7374 - loss: 0.3151 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8542 - val_iou_coefficient: 0.7466 - val_loss: 0.3079
Epoch 16/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9469 - dice_coefficient: 0.8589 - iou_coefficient: 0.7547 - loss: 0.2820 - val_binary_accuracy: 0.9370 - val_dice_coefficient: 0.8593 - val_iou_coefficient: 0.7542 - val_loss: 0.3119
Epoch 17/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9395 - dice_coefficient: 0.8436 - iou_coefficient: 0.7336 - loss: 0.3192 - val_binary_accuracy: 0.9373 - val_dice_coefficient: 0.8609 - val_iou_coefficient: 0.7570 - val_loss: 0.3131
Epoch 18/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9344 - dice_coefficient: 0.8363 - iou_coefficient: 0.7216 - loss: 0.3379 - val_binary_accuracy: 0.9320 - val_dice_coefficient: 0.8625 - val_iou_coefficient: 0.7596 - val_loss: 0.3441
Epoch 19/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9440 - dice_coefficient: 0.8534 - iou_coefficient: 0.7460 - loss: 0.2971 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8452 - val_iou_coefficient: 0.7340 - val_loss: 0.3416
Epoch 20/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9430 - dice_coefficient: 0.8584 - iou_coefficient: 0.7542 - loss: 0.2952 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8677 - val_iou_coefficient: 0.7689 - val_loss: 0.3017
Epoch 21/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9472 - dice_coefficient: 0.8637 - iou_coefficient: 0.7615 - loss: 0.2809 - val_binary_accuracy: 0.9430 - val_dice_coefficient: 0.8809 - val_iou_coefficient: 0.7903 - val_loss: 0.2799
Epoch 22/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9496 - dice_coefficient: 0.8696 - iou_coefficient: 0.7714 - loss: 0.2628 - val_binary_accuracy: 0.9376 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7314 - val_loss: 0.3106
Epoch 23/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9518 - dice_coefficient: 0.8755 - iou_coefficient: 0.7806 - loss: 0.2527 - val_binary_accuracy: 0.9435 - val_dice_coefficient: 0.8738 - val_iou_coefficient: 0.7774 - val_loss: 0.2822
Epoch 24/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9451 - dice_coefficient: 0.8593 - iou_coefficient: 0.7563 - loss: 0.2927 - val_binary_accuracy: 0.9459 - val_dice_coefficient: 0.8697 - val_iou_coefficient: 0.7721 - val_loss: 0.2833
Epoch 25/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9489 - dice_coefficient: 0.8698 - iou_coefficient: 0.7716 - loss: 0.2716 - val_binary_accuracy: 0.9458 - val_dice_coefficient: 0.8766 - val_iou_coefficient: 0.7821 - val_loss: 0.2737
Epoch 26/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9504 - dice_coefficient: 0.8776 - iou_coefficient: 0.7839 - loss: 0.2617 - val_binary_accuracy: 0.9462 - val_dice_coefficient: 0.8701 - val_iou_coefficient: 0.7719 - val_loss: 0.2685
Epoch 27/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9548 - dice_coefficient: 0.8836 - iou_coefficient: 0.7928 - loss: 0.2355 - val_binary_accuracy: 0.9456 - val_dice_coefficient: 0.8770 - val_iou_coefficient: 0.7823 - val_loss: 0.2693
Epoch 28/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9536 - dice_coefficient: 0.8831 - iou_coefficient: 0.7928 - loss: 0.2447 - val_binary_accuracy: 0.9368 - val_dice_coefficient: 0.8508 - val_iou_coefficient: 0.7447 - val_loss: 0.3113
Epoch 29/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9505 - dice_coefficient: 0.8770 - iou_coefficient: 0.7833 - loss: 0.2547 - val_binary_accuracy: 0.9441 - val_dice_coefficient: 0.8689 - val_iou_coefficient: 0.7695 - val_loss: 0.2807
Epoch 30/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9537 - dice_coefficient: 0.8844 - iou_coefficient: 0.7941 - loss: 0.2399 - val_binary_accuracy: 0.9460 - val_dice_coefficient: 0.8735 - val_iou_coefficient: 0.7770 - val_loss: 0.2767
  → Dice=0.8809  IoU=0.7903  (598s, 30 epochs)

============================================================
  loss=iou  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8020 - dice_coefficient: 0.6250 - iou_coefficient: 0.4665 - loss: 0.5304 - val_binary_accuracy: 0.7327 - val_dice_coefficient: 0.3373 - val_iou_coefficient: 0.2033 - val_loss: 0.7955
Epoch 2/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.8801 - dice_coefficient: 0.7356 - iou_coefficient: 0.5890 - loss: 0.4129 - val_binary_accuracy: 0.7995 - val_dice_coefficient: 0.4838 - val_iou_coefficient: 0.3211 - val_loss: 0.6794
Epoch 3/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.8948 - dice_coefficient: 0.7691 - iou_coefficient: 0.6314 - loss: 0.3685 - val_binary_accuracy: 0.8023 - val_dice_coefficient: 0.4986 - val_iou_coefficient: 0.3438 - val_loss: 0.6732
Epoch 4/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.8950 - dice_coefficient: 0.7703 - iou_coefficient: 0.6326 - loss: 0.3644 - val_binary_accuracy: 0.7976 - val_dice_coefficient: 0.4595 - val_iou_coefficient: 0.3102 - val_loss: 0.6703
Epoch 5/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9088 - dice_coefficient: 0.7887 - iou_coefficient: 0.6579 - loss: 0.3370 - val_binary_accuracy: 0.8761 - val_dice_coefficient: 0.7127 - val_iou_coefficient: 0.5617 - val_loss: 0.4503
Epoch 6/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9115 - dice_coefficient: 0.7984 - iou_coefficient: 0.6689 - loss: 0.3321 - val_binary_accuracy: 0.8908 - val_dice_coefficient: 0.7636 - val_iou_coefficient: 0.6216 - val_loss: 0.3734
Epoch 7/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9216 - dice_coefficient: 0.8169 - iou_coefficient: 0.6963 - loss: 0.3026 - val_binary_accuracy: 0.9083 - val_dice_coefficient: 0.8016 - val_iou_coefficient: 0.6733 - val_loss: 0.3284
Epoch 8/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9237 - dice_coefficient: 0.8283 - iou_coefficient: 0.7132 - loss: 0.2869 - val_binary_accuracy: 0.8938 - val_dice_coefficient: 0.7914 - val_iou_coefficient: 0.6569 - val_loss: 0.3393
Epoch 9/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9222 - dice_coefficient: 0.8300 - iou_coefficient: 0.7129 - loss: 0.2887 - val_binary_accuracy: 0.9178 - val_dice_coefficient: 0.8233 - val_iou_coefficient: 0.7031 - val_loss: 0.2944
Epoch 10/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9195 - dice_coefficient: 0.8251 - iou_coefficient: 0.7076 - loss: 0.2930 - val_binary_accuracy: 0.9300 - val_dice_coefficient: 0.8515 - val_iou_coefficient: 0.7440 - val_loss: 0.2646
Epoch 11/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9319 - dice_coefficient: 0.8492 - iou_coefficient: 0.7417 - loss: 0.2588 - val_binary_accuracy: 0.9297 - val_dice_coefficient: 0.8513 - val_iou_coefficient: 0.7423 - val_loss: 0.2591
Epoch 12/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9293 - dice_coefficient: 0.8332 - iou_coefficient: 0.7247 - loss: 0.2674 - val_binary_accuracy: 0.9153 - val_dice_coefficient: 0.8407 - val_iou_coefficient: 0.7273 - val_loss: 0.2783
Epoch 13/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9277 - dice_coefficient: 0.8281 - iou_coefficient: 0.7178 - loss: 0.2736 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8689 - val_iou_coefficient: 0.7714 - val_loss: 0.2196
Epoch 14/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9361 - dice_coefficient: 0.8617 - iou_coefficient: 0.7594 - loss: 0.2414 - val_binary_accuracy: 0.9375 - val_dice_coefficient: 0.8727 - val_iou_coefficient: 0.7767 - val_loss: 0.2304
Epoch 15/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9372 - dice_coefficient: 0.8629 - iou_coefficient: 0.7620 - loss: 0.2359 - val_binary_accuracy: 0.9372 - val_dice_coefficient: 0.8658 - val_iou_coefficient: 0.7689 - val_loss: 0.2384
Epoch 16/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9403 - dice_coefficient: 0.8703 - iou_coefficient: 0.7736 - loss: 0.2278 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8737 - val_iou_coefficient: 0.7813 - val_loss: 0.2239
Epoch 17/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9413 - dice_coefficient: 0.8780 - iou_coefficient: 0.7848 - loss: 0.2165 - val_binary_accuracy: 0.9336 - val_dice_coefficient: 0.8573 - val_iou_coefficient: 0.7549 - val_loss: 0.2282
Epoch 18/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9409 - dice_coefficient: 0.8732 - iou_coefficient: 0.7782 - loss: 0.2222 - val_binary_accuracy: 0.9387 - val_dice_coefficient: 0.8714 - val_iou_coefficient: 0.7745 - val_loss: 0.2215
Epoch 19/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9384 - dice_coefficient: 0.8649 - iou_coefficient: 0.7677 - loss: 0.2306 - val_binary_accuracy: 0.9392 - val_dice_coefficient: 0.8601 - val_iou_coefficient: 0.7598 - val_loss: 0.2269
Epoch 20/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9405 - dice_coefficient: 0.8756 - iou_coefficient: 0.7814 - loss: 0.2189 - val_binary_accuracy: 0.9367 - val_dice_coefficient: 0.8827 - val_iou_coefficient: 0.7940 - val_loss: 0.2128
Epoch 21/30
40/40 - 19s - 480ms/step - binary_accuracy: 0.9403 - dice_coefficient: 0.8785 - iou_coefficient: 0.7857 - loss: 0.2155 - val_binary_accuracy: 0.9397 - val_dice_coefficient: 0.8790 - val_iou_coefficient: 0.7864 - val_loss: 0.2080
Epoch 22/30
40/40 - 19s - 483ms/step - binary_accuracy: 0.9374 - dice_coefficient: 0.8705 - iou_coefficient: 0.7735 - loss: 0.2257 - val_binary_accuracy: 0.9397 - val_dice_coefficient: 0.8819 - val_iou_coefficient: 0.7898 - val_loss: 0.2098
Epoch 23/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9433 - dice_coefficient: 0.8798 - iou_coefficient: 0.7879 - loss: 0.2115 - val_binary_accuracy: 0.9409 - val_dice_coefficient: 0.8897 - val_iou_coefficient: 0.8030 - val_loss: 0.2046
Epoch 24/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9438 - dice_coefficient: 0.8836 - iou_coefficient: 0.7929 - loss: 0.2082 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8901 - val_iou_coefficient: 0.8035 - val_loss: 0.2023
Epoch 25/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9452 - dice_coefficient: 0.8841 - iou_coefficient: 0.7956 - loss: 0.2055 - val_binary_accuracy: 0.9408 - val_dice_coefficient: 0.8769 - val_iou_coefficient: 0.7841 - val_loss: 0.2159
Epoch 26/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9476 - dice_coefficient: 0.8897 - iou_coefficient: 0.8036 - loss: 0.1938 - val_binary_accuracy: 0.9323 - val_dice_coefficient: 0.8769 - val_iou_coefficient: 0.7847 - val_loss: 0.2159
Epoch 27/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9464 - dice_coefficient: 0.8903 - iou_coefficient: 0.8042 - loss: 0.1951 - val_binary_accuracy: 0.9438 - val_dice_coefficient: 0.8870 - val_iou_coefficient: 0.7996 - val_loss: 0.2026
Epoch 28/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9479 - dice_coefficient: 0.8941 - iou_coefficient: 0.8098 - loss: 0.1886 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8864 - val_iou_coefficient: 0.7998 - val_loss: 0.1996
Epoch 29/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9485 - dice_coefficient: 0.8932 - iou_coefficient: 0.8102 - loss: 0.1848 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8835 - val_iou_coefficient: 0.7927 - val_loss: 0.2018
Epoch 30/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9466 - dice_coefficient: 0.8920 - iou_coefficient: 0.8072 - loss: 0.1933 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8897 - val_iou_coefficient: 0.8041 - val_loss: 0.1964
  → Dice=0.8901  IoU=0.8041  (600s, 30 epochs)

============================================================
</pre>


### Récap `loss`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `loss=bce` | 31,402,497 | 0.8600 | 0.7556 | 597s, 30 epochs |
| `loss=dice` | 31,402,497 | 0.8902 | 0.8050 | 446s, 22 epochs |
| `loss=bce+dice` | 31,402,497 | 0.8809 | 0.7903 | 598s, 30 epochs |
| `loss=iou` | 31,402,497 | 0.8901 | 0.8041 | 600s, 30 epochs |

**Meilleur:** `loss=dice` (Dice=0.8902, IoU=0.8050).


## 5) Expérience : stratégie d'augmentation

**Ce qu'on teste :** pas d'augmentation, flips simples, augmentation forte (`heavy`).

Objectif : évaluer le gain de généralisation apporté par des transformations plus riches sur l'entraînement.


In [ ]:
# ── Exp 5 : augmentation (none / flip / heavy) ──

aug_configs = {
    'aug=none':  None,
    'aug=flip':  augment_flip,
    'aug=heavy': augment_heavy,
}

exp_aug = []
for name, aug_fn in aug_configs.items():
    ds_train = make_ds(train_img, train_mask, augment_fn=aug_fn)
    r = run_experiment(
        name,
        build_unet(base_filters=64, depth=4, use_skip=True),
        ds_train, val_ds
    )
    exp_aug.append(r)

plot_compare(exp_aug, 'augmentation')

### Logs `augmentation`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  aug=none  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8278 - dice_coefficient: 0.6100 - iou_coefficient: 0.4456 - loss: 0.8132 - val_binary_accuracy: 0.7946 - val_dice_coefficient: 0.4216 - val_iou_coefficient: 0.2698 - val_loss: 1.0924
Epoch 2/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.8905 - dice_coefficient: 0.7054 - iou_coefficient: 0.5481 - loss: 0.5897 - val_binary_accuracy: 0.7327 - val_dice_coefficient: 0.4734 - val_iou_coefficient: 0.3141 - val_loss: 1.2327
Epoch 3/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.8957 - dice_coefficient: 0.7269 - iou_coefficient: 0.5746 - loss: 0.5477 - val_binary_accuracy: 0.8509 - val_dice_coefficient: 0.5949 - val_iou_coefficient: 0.4284 - val_loss: 0.8677
Epoch 4/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9120 - dice_coefficient: 0.7662 - iou_coefficient: 0.6267 - loss: 0.4761 - val_binary_accuracy: 0.8452 - val_dice_coefficient: 0.6405 - val_iou_coefficient: 0.4827 - val_loss: 0.7992
Epoch 5/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9149 - dice_coefficient: 0.7708 - iou_coefficient: 0.6304 - loss: 0.4526 - val_binary_accuracy: 0.8814 - val_dice_coefficient: 0.7036 - val_iou_coefficient: 0.5506 - val_loss: 0.6319
Epoch 6/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9284 - dice_coefficient: 0.8024 - iou_coefficient: 0.6724 - loss: 0.3897 - val_binary_accuracy: 0.9008 - val_dice_coefficient: 0.7685 - val_iou_coefficient: 0.6262 - val_loss: 0.5208
Epoch 7/30
40/40 - 18s - 459ms/step - binary_accuracy: 0.9241 - dice_coefficient: 0.7980 - iou_coefficient: 0.6689 - loss: 0.4062 - val_binary_accuracy: 0.8976 - val_dice_coefficient: 0.7853 - val_iou_coefficient: 0.6509 - val_loss: 0.5249
Epoch 8/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9339 - dice_coefficient: 0.8194 - iou_coefficient: 0.6973 - loss: 0.3601 - val_binary_accuracy: 0.9288 - val_dice_coefficient: 0.8111 - val_iou_coefficient: 0.6835 - val_loss: 0.3873
Epoch 9/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8345 - iou_coefficient: 0.7180 - loss: 0.3374 - val_binary_accuracy: 0.9232 - val_dice_coefficient: 0.8118 - val_iou_coefficient: 0.6900 - val_loss: 0.4101
Epoch 10/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9371 - dice_coefficient: 0.8289 - iou_coefficient: 0.7120 - loss: 0.3411 - val_binary_accuracy: 0.9314 - val_dice_coefficient: 0.8315 - val_iou_coefficient: 0.7133 - val_loss: 0.3650
Epoch 11/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9391 - dice_coefficient: 0.8323 - iou_coefficient: 0.7151 - loss: 0.3289 - val_binary_accuracy: 0.9383 - val_dice_coefficient: 0.8413 - val_iou_coefficient: 0.7274 - val_loss: 0.3342
Epoch 12/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9433 - dice_coefficient: 0.8441 - iou_coefficient: 0.7328 - loss: 0.3123 - val_binary_accuracy: 0.9351 - val_dice_coefficient: 0.8361 - val_iou_coefficient: 0.7205 - val_loss: 0.3392
Epoch 13/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9348 - dice_coefficient: 0.8248 - iou_coefficient: 0.7056 - loss: 0.3498 - val_binary_accuracy: 0.9290 - val_dice_coefficient: 0.8382 - val_iou_coefficient: 0.7235 - val_loss: 0.3663
Epoch 14/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8430 - iou_coefficient: 0.7316 - loss: 0.3125 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8388 - val_iou_coefficient: 0.7257 - val_loss: 0.3321
Epoch 15/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9451 - dice_coefficient: 0.8511 - iou_coefficient: 0.7436 - loss: 0.2954 - val_binary_accuracy: 0.9394 - val_dice_coefficient: 0.8609 - val_iou_coefficient: 0.7578 - val_loss: 0.3167
Epoch 16/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9423 - dice_coefficient: 0.8466 - iou_coefficient: 0.7370 - loss: 0.3126 - val_binary_accuracy: 0.9400 - val_dice_coefficient: 0.8707 - val_iou_coefficient: 0.7722 - val_loss: 0.3059
Epoch 17/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9497 - dice_coefficient: 0.8683 - iou_coefficient: 0.7693 - loss: 0.2705 - val_binary_accuracy: 0.9412 - val_dice_coefficient: 0.8676 - val_iou_coefficient: 0.7676 - val_loss: 0.2970
Epoch 18/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9515 - dice_coefficient: 0.8693 - iou_coefficient: 0.7710 - loss: 0.2632 - val_binary_accuracy: 0.9381 - val_dice_coefficient: 0.8424 - val_iou_coefficient: 0.7302 - val_loss: 0.3211
Epoch 19/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9572 - dice_coefficient: 0.8838 - iou_coefficient: 0.7936 - loss: 0.2376 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8548 - val_iou_coefficient: 0.7480 - val_loss: 0.3282
Epoch 20/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9515 - dice_coefficient: 0.8680 - iou_coefficient: 0.7694 - loss: 0.2672 - val_binary_accuracy: 0.9306 - val_dice_coefficient: 0.8323 - val_iou_coefficient: 0.7204 - val_loss: 0.3539
Epoch 21/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9512 - dice_coefficient: 0.8696 - iou_coefficient: 0.7716 - loss: 0.2654 - val_binary_accuracy: 0.9384 - val_dice_coefficient: 0.8594 - val_iou_coefficient: 0.7556 - val_loss: 0.3172
Epoch 22/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9559 - dice_coefficient: 0.8811 - iou_coefficient: 0.7891 - loss: 0.2420 - val_binary_accuracy: 0.9408 - val_dice_coefficient: 0.8597 - val_iou_coefficient: 0.7556 - val_loss: 0.3070
Epoch 23/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9554 - dice_coefficient: 0.8829 - iou_coefficient: 0.7919 - loss: 0.2412 - val_binary_accuracy: 0.9319 - val_dice_coefficient: 0.8178 - val_iou_coefficient: 0.7058 - val_loss: 0.3555
Epoch 24/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9549 - dice_coefficient: 0.8767 - iou_coefficient: 0.7832 - loss: 0.2465 - val_binary_accuracy: 0.9338 - val_dice_coefficient: 0.8560 - val_iou_coefficient: 0.7520 - val_loss: 0.3316
Epoch 25/30
40/40 - 18s - 459ms/step - binary_accuracy: 0.9605 - dice_coefficient: 0.8936 - iou_coefficient: 0.8093 - loss: 0.2115 - val_binary_accuracy: 0.9379 - val_dice_coefficient: 0.8543 - val_iou_coefficient: 0.7482 - val_loss: 0.3212
  → Dice=0.8707  IoU=0.7722  (503s, 25 epochs)

============================================================
  aug=flip  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8428 - dice_coefficient: 0.6136 - iou_coefficient: 0.4519 - loss: 0.7865 - val_binary_accuracy: 0.7793 - val_dice_coefficient: 0.3999 - val_iou_coefficient: 0.2509 - val_loss: 1.0839
Epoch 2/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.8815 - dice_coefficient: 0.7094 - iou_coefficient: 0.5539 - loss: 0.5941 - val_binary_accuracy: 0.7642 - val_dice_coefficient: 0.3675 - val_iou_coefficient: 0.2279 - val_loss: 1.2014
Epoch 3/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9078 - dice_coefficient: 0.7540 - iou_coefficient: 0.6110 - loss: 0.4904 - val_binary_accuracy: 0.8717 - val_dice_coefficient: 0.6549 - val_iou_coefficient: 0.4913 - val_loss: 0.6930
Epoch 4/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9080 - dice_coefficient: 0.7671 - iou_coefficient: 0.6262 - loss: 0.4750 - val_binary_accuracy: 0.8038 - val_dice_coefficient: 0.4875 - val_iou_coefficient: 0.3295 - val_loss: 1.0726
Epoch 5/30
40/40 - 19s - 481ms/step - binary_accuracy: 0.9157 - dice_coefficient: 0.7876 - iou_coefficient: 0.6536 - loss: 0.4397 - val_binary_accuracy: 0.8630 - val_dice_coefficient: 0.6496 - val_iou_coefficient: 0.4896 - val_loss: 0.7184
Epoch 6/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9165 - dice_coefficient: 0.7873 - iou_coefficient: 0.6535 - loss: 0.4362 - val_binary_accuracy: 0.8677 - val_dice_coefficient: 0.6870 - val_iou_coefficient: 0.5298 - val_loss: 0.7262
Epoch 7/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9296 - dice_coefficient: 0.8145 - iou_coefficient: 0.6898 - loss: 0.3697 - val_binary_accuracy: 0.8974 - val_dice_coefficient: 0.7711 - val_iou_coefficient: 0.6320 - val_loss: 0.5571
Epoch 8/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9295 - dice_coefficient: 0.8125 - iou_coefficient: 0.6907 - loss: 0.3761 - val_binary_accuracy: 0.9206 - val_dice_coefficient: 0.8169 - val_iou_coefficient: 0.6949 - val_loss: 0.4115
Epoch 9/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9292 - dice_coefficient: 0.8137 - iou_coefficient: 0.6892 - loss: 0.3781 - val_binary_accuracy: 0.9151 - val_dice_coefficient: 0.8143 - val_iou_coefficient: 0.6889 - val_loss: 0.4193
Epoch 10/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9332 - dice_coefficient: 0.8297 - iou_coefficient: 0.7126 - loss: 0.3485 - val_binary_accuracy: 0.9351 - val_dice_coefficient: 0.8460 - val_iou_coefficient: 0.7348 - val_loss: 0.3345
Epoch 11/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9340 - dice_coefficient: 0.8330 - iou_coefficient: 0.7162 - loss: 0.3413 - val_binary_accuracy: 0.9293 - val_dice_coefficient: 0.8409 - val_iou_coefficient: 0.7278 - val_loss: 0.3612
Epoch 12/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9379 - dice_coefficient: 0.8355 - iou_coefficient: 0.7207 - loss: 0.3317 - val_binary_accuracy: 0.9303 - val_dice_coefficient: 0.8403 - val_iou_coefficient: 0.7269 - val_loss: 0.3486
Epoch 13/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9384 - dice_coefficient: 0.8410 - iou_coefficient: 0.7276 - loss: 0.3246 - val_binary_accuracy: 0.9349 - val_dice_coefficient: 0.8451 - val_iou_coefficient: 0.7344 - val_loss: 0.3364
Epoch 14/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9380 - dice_coefficient: 0.8450 - iou_coefficient: 0.7343 - loss: 0.3246 - val_binary_accuracy: 0.9401 - val_dice_coefficient: 0.8454 - val_iou_coefficient: 0.7366 - val_loss: 0.3112
Epoch 15/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9393 - dice_coefficient: 0.8428 - iou_coefficient: 0.7306 - loss: 0.3194 - val_binary_accuracy: 0.9351 - val_dice_coefficient: 0.8535 - val_iou_coefficient: 0.7466 - val_loss: 0.3380
Epoch 16/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9417 - dice_coefficient: 0.8506 - iou_coefficient: 0.7421 - loss: 0.3065 - val_binary_accuracy: 0.9340 - val_dice_coefficient: 0.8566 - val_iou_coefficient: 0.7517 - val_loss: 0.3245
Epoch 17/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9440 - dice_coefficient: 0.8561 - iou_coefficient: 0.7501 - loss: 0.2941 - val_binary_accuracy: 0.9355 - val_dice_coefficient: 0.8602 - val_iou_coefficient: 0.7561 - val_loss: 0.3200
Epoch 18/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9445 - dice_coefficient: 0.8551 - iou_coefficient: 0.7496 - loss: 0.2887 - val_binary_accuracy: 0.9379 - val_dice_coefficient: 0.8490 - val_iou_coefficient: 0.7439 - val_loss: 0.3200
Epoch 19/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9470 - dice_coefficient: 0.8663 - iou_coefficient: 0.7664 - loss: 0.2762 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8511 - val_iou_coefficient: 0.7463 - val_loss: 0.3008
Epoch 20/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9463 - dice_coefficient: 0.8611 - iou_coefficient: 0.7589 - loss: 0.2835 - val_binary_accuracy: 0.9410 - val_dice_coefficient: 0.8663 - val_iou_coefficient: 0.7654 - val_loss: 0.2934
Epoch 21/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9477 - dice_coefficient: 0.8682 - iou_coefficient: 0.7687 - loss: 0.2760 - val_binary_accuracy: 0.9395 - val_dice_coefficient: 0.8588 - val_iou_coefficient: 0.7540 - val_loss: 0.3072
Epoch 22/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9493 - dice_coefficient: 0.8693 - iou_coefficient: 0.7705 - loss: 0.2701 - val_binary_accuracy: 0.9414 - val_dice_coefficient: 0.8590 - val_iou_coefficient: 0.7567 - val_loss: 0.2902
Epoch 23/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8721 - iou_coefficient: 0.7752 - loss: 0.2672 - val_binary_accuracy: 0.9416 - val_dice_coefficient: 0.8668 - val_iou_coefficient: 0.7670 - val_loss: 0.2937
Epoch 24/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9467 - dice_coefficient: 0.8625 - iou_coefficient: 0.7623 - loss: 0.2752 - val_binary_accuracy: 0.9456 - val_dice_coefficient: 0.8738 - val_iou_coefficient: 0.7779 - val_loss: 0.2714
Epoch 25/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9493 - dice_coefficient: 0.8636 - iou_coefficient: 0.7635 - loss: 0.2680 - val_binary_accuracy: 0.9409 - val_dice_coefficient: 0.8615 - val_iou_coefficient: 0.7596 - val_loss: 0.3007
Epoch 26/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9466 - dice_coefficient: 0.8679 - iou_coefficient: 0.7696 - loss: 0.2764 - val_binary_accuracy: 0.9300 - val_dice_coefficient: 0.8441 - val_iou_coefficient: 0.7364 - val_loss: 0.3575
Epoch 27/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9511 - dice_coefficient: 0.8743 - iou_coefficient: 0.7782 - loss: 0.2558 - val_binary_accuracy: 0.9445 - val_dice_coefficient: 0.8596 - val_iou_coefficient: 0.7587 - val_loss: 0.2904
Epoch 28/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9504 - dice_coefficient: 0.8807 - iou_coefficient: 0.7884 - loss: 0.2557 - val_binary_accuracy: 0.9225 - val_dice_coefficient: 0.8393 - val_iou_coefficient: 0.7290 - val_loss: 0.4047
Epoch 29/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9486 - dice_coefficient: 0.8715 - iou_coefficient: 0.7735 - loss: 0.2658 - val_binary_accuracy: 0.9493 - val_dice_coefficient: 0.8677 - val_iou_coefficient: 0.7693 - val_loss: 0.2672
Epoch 30/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9523 - dice_coefficient: 0.8801 - iou_coefficient: 0.7879 - loss: 0.2493 - val_binary_accuracy: 0.9462 - val_dice_coefficient: 0.8789 - val_iou_coefficient: 0.7852 - val_loss: 0.2679
  → Dice=0.8789  IoU=0.7852  (600s, 30 epochs)

============================================================
  aug=heavy  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8088 - dice_coefficient: 0.5802 - iou_coefficient: 0.4155 - loss: 0.8547 - val_binary_accuracy: 0.7394 - val_dice_coefficient: 0.3253 - val_iou_coefficient: 0.1957 - val_loss: 1.2729
Epoch 2/30
40/40 - 19s - 481ms/step - binary_accuracy: 0.8814 - dice_coefficient: 0.6848 - iou_coefficient: 0.5236 - loss: 0.6316 - val_binary_accuracy: 0.8327 - val_dice_coefficient: 0.4945 - val_iou_coefficient: 0.3306 - val_loss: 0.9502
Epoch 3/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.8971 - dice_coefficient: 0.7346 - iou_coefficient: 0.5831 - loss: 0.5430 - val_binary_accuracy: 0.7636 - val_dice_coefficient: 0.4667 - val_iou_coefficient: 0.3112 - val_loss: 1.1541
Epoch 4/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.8982 - dice_coefficient: 0.7401 - iou_coefficient: 0.5910 - loss: 0.5301 - val_binary_accuracy: 0.8423 - val_dice_coefficient: 0.5979 - val_iou_coefficient: 0.4354 - val_loss: 0.8931
Epoch 5/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9017 - dice_coefficient: 0.7520 - iou_coefficient: 0.6070 - loss: 0.5137 - val_binary_accuracy: 0.8863 - val_dice_coefficient: 0.7296 - val_iou_coefficient: 0.5789 - val_loss: 0.6042
Epoch 6/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9131 - dice_coefficient: 0.7804 - iou_coefficient: 0.6435 - loss: 0.4531 - val_binary_accuracy: 0.9025 - val_dice_coefficient: 0.7633 - val_iou_coefficient: 0.6212 - val_loss: 0.5152
Epoch 7/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9129 - dice_coefficient: 0.7762 - iou_coefficient: 0.6374 - loss: 0.4626 - val_binary_accuracy: 0.9029 - val_dice_coefficient: 0.7603 - val_iou_coefficient: 0.6199 - val_loss: 0.5033
Epoch 8/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9225 - dice_coefficient: 0.7947 - iou_coefficient: 0.6627 - loss: 0.4063 - val_binary_accuracy: 0.9112 - val_dice_coefficient: 0.7817 - val_iou_coefficient: 0.6429 - val_loss: 0.4892
Epoch 9/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9213 - dice_coefficient: 0.7987 - iou_coefficient: 0.6679 - loss: 0.4137 - val_binary_accuracy: 0.9145 - val_dice_coefficient: 0.7947 - val_iou_coefficient: 0.6610 - val_loss: 0.4413
Epoch 10/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9211 - dice_coefficient: 0.8043 - iou_coefficient: 0.6759 - loss: 0.4009 - val_binary_accuracy: 0.9204 - val_dice_coefficient: 0.7997 - val_iou_coefficient: 0.6705 - val_loss: 0.4138
Epoch 11/30
40/40 - 19s - 480ms/step - binary_accuracy: 0.9272 - dice_coefficient: 0.8140 - iou_coefficient: 0.6899 - loss: 0.3819 - val_binary_accuracy: 0.9293 - val_dice_coefficient: 0.8350 - val_iou_coefficient: 0.7187 - val_loss: 0.3662
Epoch 12/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9266 - dice_coefficient: 0.8148 - iou_coefficient: 0.6913 - loss: 0.3806 - val_binary_accuracy: 0.9383 - val_dice_coefficient: 0.8385 - val_iou_coefficient: 0.7255 - val_loss: 0.3282
Epoch 13/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9318 - dice_coefficient: 0.8242 - iou_coefficient: 0.7037 - loss: 0.3614 - val_binary_accuracy: 0.9295 - val_dice_coefficient: 0.8323 - val_iou_coefficient: 0.7168 - val_loss: 0.3559
Epoch 14/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9278 - dice_coefficient: 0.8214 - iou_coefficient: 0.6987 - loss: 0.3712 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8511 - val_iou_coefficient: 0.7425 - val_loss: 0.3118
Epoch 15/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9281 - dice_coefficient: 0.8229 - iou_coefficient: 0.7024 - loss: 0.3738 - val_binary_accuracy: 0.9389 - val_dice_coefficient: 0.8610 - val_iou_coefficient: 0.7569 - val_loss: 0.3076
Epoch 16/30
40/40 - 19s - 480ms/step - binary_accuracy: 0.9341 - dice_coefficient: 0.8312 - iou_coefficient: 0.7134 - loss: 0.3460 - val_binary_accuracy: 0.9385 - val_dice_coefficient: 0.8472 - val_iou_coefficient: 0.7361 - val_loss: 0.3172
Epoch 17/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9392 - dice_coefficient: 0.8461 - iou_coefficient: 0.7351 - loss: 0.3149 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8330 - val_iou_coefficient: 0.7187 - val_loss: 0.3366
Epoch 18/30
40/40 - 19s - 478ms/step - binary_accuracy: 0.9396 - dice_coefficient: 0.8496 - iou_coefficient: 0.7400 - loss: 0.3128 - val_binary_accuracy: 0.9384 - val_dice_coefficient: 0.8472 - val_iou_coefficient: 0.7370 - val_loss: 0.3086
Epoch 19/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9316 - dice_coefficient: 0.8264 - iou_coefficient: 0.7075 - loss: 0.3566 - val_binary_accuracy: 0.9437 - val_dice_coefficient: 0.8603 - val_iou_coefficient: 0.7567 - val_loss: 0.2861
Epoch 20/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9368 - dice_coefficient: 0.8426 - iou_coefficient: 0.7298 - loss: 0.3240 - val_binary_accuracy: 0.9410 - val_dice_coefficient: 0.8687 - val_iou_coefficient: 0.7698 - val_loss: 0.2960
Epoch 21/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9406 - dice_coefficient: 0.8451 - iou_coefficient: 0.7345 - loss: 0.3129 - val_binary_accuracy: 0.9315 - val_dice_coefficient: 0.8331 - val_iou_coefficient: 0.7170 - val_loss: 0.3543
Epoch 22/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9417 - dice_coefficient: 0.8518 - iou_coefficient: 0.7443 - loss: 0.3075 - val_binary_accuracy: 0.9380 - val_dice_coefficient: 0.8505 - val_iou_coefficient: 0.7418 - val_loss: 0.3190
Epoch 23/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9406 - dice_coefficient: 0.8515 - iou_coefficient: 0.7436 - loss: 0.3055 - val_binary_accuracy: 0.9355 - val_dice_coefficient: 0.8476 - val_iou_coefficient: 0.7380 - val_loss: 0.3202
Epoch 24/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9418 - dice_coefficient: 0.8514 - iou_coefficient: 0.7437 - loss: 0.3119 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8421 - val_iou_coefficient: 0.7319 - val_loss: 0.3026
Epoch 25/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9408 - dice_coefficient: 0.8551 - iou_coefficient: 0.7485 - loss: 0.3049 - val_binary_accuracy: 0.9406 - val_dice_coefficient: 0.8586 - val_iou_coefficient: 0.7532 - val_loss: 0.3060
Epoch 26/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9403 - dice_coefficient: 0.8465 - iou_coefficient: 0.7383 - loss: 0.3096 - val_binary_accuracy: 0.9390 - val_dice_coefficient: 0.8575 - val_iou_coefficient: 0.7535 - val_loss: 0.3056
Epoch 27/30
40/40 - 19s - 477ms/step - binary_accuracy: 0.9402 - dice_coefficient: 0.8531 - iou_coefficient: 0.7454 - loss: 0.3134 - val_binary_accuracy: 0.9392 - val_dice_coefficient: 0.8633 - val_iou_coefficient: 0.7610 - val_loss: 0.3127
  → Dice=0.8687  IoU=0.7698  (546s, 27 epochs)

============================================================
</pre>


### Récap `augmentation`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `aug=none` | 31,402,497 | 0.8707 | 0.7722 | 503s, 25 epochs |
| `aug=flip` | 31,402,497 | 0.8789 | 0.7852 | 600s, 30 epochs |
| `aug=heavy` | 31,402,497 | 0.8687 | 0.7698 | 546s, 27 epochs |

**Meilleur:** `aug=flip` (Dice=0.8789, IoU=0.7852).


## 6) Expérience : résolution d'entrée

**Ce qu'on teste :** tailles d'images `128x128`, `256x256`, `384x384`.

Objectif : analyser le compromis entre précision spatiale, coût mémoire et temps d'entraînement.


In [ ]:
# ── Exp 6 : image size (128 / 256 / 384) ──

exp_size = []
size_batch = {128: 8, 256: 8, 384: 2}
for sz in [128, 256, 384]:
    bs = size_batch[sz]
    ds_tr = make_ds(train_img, train_mask, img_size=(sz, sz), batch_size=bs, augment_fn=augment_flip)
    ds_vl = make_ds(val_img, val_mask, img_size=(sz, sz), batch_size=bs)
    r = run_experiment(
        f'size={sz} (bs={bs})',
        build_unet(input_shape=(sz, sz, 3), base_filters=64, depth=4, use_skip=True),
        ds_tr, ds_vl
    )
    exp_size.append(r)

plot_compare(exp_size, 'image size')

### Logs `image_size`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  size=128  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 48s - 1s/step - binary_accuracy: 0.8444 - dice_coefficient: 0.6247 - iou_coefficient: 0.4651 - loss: 0.7639 - val_binary_accuracy: 0.7253 - val_dice_coefficient: 0.2840 - val_iou_coefficient: 0.1665 - val_loss: 1.2735
Epoch 2/30
40/40 - 13s - 313ms/step - binary_accuracy: 0.9049 - dice_coefficient: 0.7376 - iou_coefficient: 0.5874 - loss: 0.5193 - val_binary_accuracy: 0.7873 - val_dice_coefficient: 0.4265 - val_iou_coefficient: 0.2722 - val_loss: 1.0510
Epoch 3/30
40/40 - 12s - 312ms/step - binary_accuracy: 0.9144 - dice_coefficient: 0.7633 - iou_coefficient: 0.6204 - loss: 0.4690 - val_binary_accuracy: 0.8399 - val_dice_coefficient: 0.5335 - val_iou_coefficient: 0.3738 - val_loss: 0.8810
Epoch 4/30
40/40 - 13s - 315ms/step - binary_accuracy: 0.9276 - dice_coefficient: 0.7949 - iou_coefficient: 0.6620 - loss: 0.4058 - val_binary_accuracy: 0.8644 - val_dice_coefficient: 0.6410 - val_iou_coefficient: 0.4775 - val_loss: 0.7181
Epoch 5/30
40/40 - 12s - 303ms/step - binary_accuracy: 0.9285 - dice_coefficient: 0.8064 - iou_coefficient: 0.6774 - loss: 0.3820 - val_binary_accuracy: 0.8964 - val_dice_coefficient: 0.7226 - val_iou_coefficient: 0.5688 - val_loss: 0.5669
Epoch 6/30
40/40 - 12s - 306ms/step - binary_accuracy: 0.9389 - dice_coefficient: 0.8241 - iou_coefficient: 0.7038 - loss: 0.3441 - val_binary_accuracy: 0.9026 - val_dice_coefficient: 0.7105 - val_iou_coefficient: 0.5599 - val_loss: 0.5312
Epoch 7/30
40/40 - 12s - 310ms/step - binary_accuracy: 0.9409 - dice_coefficient: 0.8325 - iou_coefficient: 0.7156 - loss: 0.3311 - val_binary_accuracy: 0.9125 - val_dice_coefficient: 0.7839 - val_iou_coefficient: 0.6495 - val_loss: 0.4595
Epoch 8/30
40/40 - 12s - 307ms/step - binary_accuracy: 0.9309 - dice_coefficient: 0.8081 - iou_coefficient: 0.6846 - loss: 0.3750 - val_binary_accuracy: 0.9320 - val_dice_coefficient: 0.8124 - val_iou_coefficient: 0.6889 - val_loss: 0.3620
Epoch 9/30
40/40 - 12s - 304ms/step - binary_accuracy: 0.9423 - dice_coefficient: 0.8376 - iou_coefficient: 0.7221 - loss: 0.3201 - val_binary_accuracy: 0.9395 - val_dice_coefficient: 0.8480 - val_iou_coefficient: 0.7374 - val_loss: 0.3165
Epoch 10/30
40/40 - 12s - 305ms/step - binary_accuracy: 0.9448 - dice_coefficient: 0.8496 - iou_coefficient: 0.7406 - loss: 0.3025 - val_binary_accuracy: 0.9431 - val_dice_coefficient: 0.8504 - val_iou_coefficient: 0.7412 - val_loss: 0.3018
Epoch 11/30
40/40 - 12s - 309ms/step - binary_accuracy: 0.9441 - dice_coefficient: 0.8472 - iou_coefficient: 0.7366 - loss: 0.3062 - val_binary_accuracy: 0.9405 - val_dice_coefficient: 0.8589 - val_iou_coefficient: 0.7554 - val_loss: 0.3085
Epoch 12/30
40/40 - 12s - 311ms/step - binary_accuracy: 0.9492 - dice_coefficient: 0.8507 - iou_coefficient: 0.7453 - loss: 0.2840 - val_binary_accuracy: 0.9274 - val_dice_coefficient: 0.8381 - val_iou_coefficient: 0.7244 - val_loss: 0.3627
Epoch 13/30
40/40 - 12s - 303ms/step - binary_accuracy: 0.9407 - dice_coefficient: 0.8424 - iou_coefficient: 0.7306 - loss: 0.3174 - val_binary_accuracy: 0.9297 - val_dice_coefficient: 0.8491 - val_iou_coefficient: 0.7395 - val_loss: 0.3742
Epoch 14/30
40/40 - 12s - 302ms/step - binary_accuracy: 0.9465 - dice_coefficient: 0.8547 - iou_coefficient: 0.7502 - loss: 0.2911 - val_binary_accuracy: 0.9390 - val_dice_coefficient: 0.8616 - val_iou_coefficient: 0.7579 - val_loss: 0.3120
Epoch 15/30
40/40 - 12s - 311ms/step - binary_accuracy: 0.9496 - dice_coefficient: 0.8606 - iou_coefficient: 0.7573 - loss: 0.2764 - val_binary_accuracy: 0.9409 - val_dice_coefficient: 0.8673 - val_iou_coefficient: 0.7677 - val_loss: 0.2910
Epoch 16/30
40/40 - 12s - 309ms/step - binary_accuracy: 0.9487 - dice_coefficient: 0.8638 - iou_coefficient: 0.7621 - loss: 0.2772 - val_binary_accuracy: 0.9467 - val_dice_coefficient: 0.8773 - val_iou_coefficient: 0.7830 - val_loss: 0.2687
Epoch 17/30
40/40 - 12s - 305ms/step - binary_accuracy: 0.9568 - dice_coefficient: 0.8804 - iou_coefficient: 0.7879 - loss: 0.2372 - val_binary_accuracy: 0.9432 - val_dice_coefficient: 0.8640 - val_iou_coefficient: 0.7614 - val_loss: 0.2839
Epoch 18/30
40/40 - 12s - 300ms/step - binary_accuracy: 0.9585 - dice_coefficient: 0.8852 - iou_coefficient: 0.7958 - loss: 0.2227 - val_binary_accuracy: 0.9478 - val_dice_coefficient: 0.8676 - val_iou_coefficient: 0.7669 - val_loss: 0.2690
Epoch 19/30
40/40 - 12s - 310ms/step - binary_accuracy: 0.9562 - dice_coefficient: 0.8727 - iou_coefficient: 0.7789 - loss: 0.2381 - val_binary_accuracy: 0.9483 - val_dice_coefficient: 0.8620 - val_iou_coefficient: 0.7592 - val_loss: 0.2741
Epoch 20/30
40/40 - 12s - 307ms/step - binary_accuracy: 0.9532 - dice_coefficient: 0.8738 - iou_coefficient: 0.7783 - loss: 0.2567 - val_binary_accuracy: 0.9422 - val_dice_coefficient: 0.8623 - val_iou_coefficient: 0.7611 - val_loss: 0.3098
Epoch 21/30
40/40 - 12s - 303ms/step - binary_accuracy: 0.9546 - dice_coefficient: 0.8766 - iou_coefficient: 0.7824 - loss: 0.2491 - val_binary_accuracy: 0.9463 - val_dice_coefficient: 0.8817 - val_iou_coefficient: 0.7892 - val_loss: 0.2661
Epoch 22/30
40/40 - 12s - 305ms/step - binary_accuracy: 0.9546 - dice_coefficient: 0.8766 - iou_coefficient: 0.7841 - loss: 0.2457 - val_binary_accuracy: 0.9386 - val_dice_coefficient: 0.8669 - val_iou_coefficient: 0.7665 - val_loss: 0.3069
Epoch 23/30
40/40 - 12s - 307ms/step - binary_accuracy: 0.9580 - dice_coefficient: 0.8916 - iou_coefficient: 0.8054 - loss: 0.2229 - val_binary_accuracy: 0.9412 - val_dice_coefficient: 0.8724 - val_iou_coefficient: 0.7748 - val_loss: 0.2892
Epoch 24/30
40/40 - 12s - 304ms/step - binary_accuracy: 0.9602 - dice_coefficient: 0.8936 - iou_coefficient: 0.8085 - loss: 0.2104 - val_binary_accuracy: 0.9455 - val_dice_coefficient: 0.8675 - val_iou_coefficient: 0.7679 - val_loss: 0.2840
Epoch 25/30
40/40 - 12s - 304ms/step - binary_accuracy: 0.9567 - dice_coefficient: 0.8869 - iou_coefficient: 0.7987 - loss: 0.2314 - val_binary_accuracy: 0.9480 - val_dice_coefficient: 0.8780 - val_iou_coefficient: 0.7842 - val_loss: 0.2572
Epoch 26/30
40/40 - 12s - 304ms/step - binary_accuracy: 0.9598 - dice_coefficient: 0.8958 - iou_coefficient: 0.8125 - loss: 0.2105 - val_binary_accuracy: 0.9430 - val_dice_coefficient: 0.8738 - val_iou_coefficient: 0.7785 - val_loss: 0.2783
Epoch 27/30
40/40 - 12s - 311ms/step - binary_accuracy: 0.9631 - dice_coefficient: 0.9001 - iou_coefficient: 0.8193 - loss: 0.1990 - val_binary_accuracy: 0.9476 - val_dice_coefficient: 0.8764 - val_iou_coefficient: 0.7816 - val_loss: 0.2595
Epoch 28/30
40/40 - 12s - 307ms/step - binary_accuracy: 0.9643 - dice_coefficient: 0.9038 - iou_coefficient: 0.8252 - loss: 0.1925 - val_binary_accuracy: 0.9451 - val_dice_coefficient: 0.8693 - val_iou_coefficient: 0.7696 - val_loss: 0.2804
Epoch 29/30
40/40 - 12s - 298ms/step - binary_accuracy: 0.9598 - dice_coefficient: 0.8961 - iou_coefficient: 0.8129 - loss: 0.2128 - val_binary_accuracy: 0.9467 - val_dice_coefficient: 0.8741 - val_iou_coefficient: 0.7782 - val_loss: 0.2632
Epoch 30/30
40/40 - 12s - 302ms/step - binary_accuracy: 0.9636 - dice_coefficient: 0.9050 - iou_coefficient: 0.8273 - loss: 0.1927 - val_binary_accuracy: 0.9482 - val_dice_coefficient: 0.8844 - val_iou_coefficient: 0.7940 - val_loss: 0.2616
  → Dice=0.8844  IoU=0.7940  (404s, 30 epochs)

============================================================
  size=256  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 52s - 1s/step - binary_accuracy: 0.8034 - dice_coefficient: 0.5863 - iou_coefficient: 0.4234 - loss: 0.8735 - val_binary_accuracy: 0.7171 - val_dice_coefficient: 0.3248 - val_iou_coefficient: 0.1946 - val_loss: 1.2375
Epoch 2/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.8860 - dice_coefficient: 0.6799 - iou_coefficient: 0.5195 - loss: 0.6318 - val_binary_accuracy: 0.7676 - val_dice_coefficient: 0.3722 - val_iou_coefficient: 0.2307 - val_loss: 1.1389
Epoch 3/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9025 - dice_coefficient: 0.7237 - iou_coefficient: 0.5716 - loss: 0.5373 - val_binary_accuracy: 0.8511 - val_dice_coefficient: 0.5739 - val_iou_coefficient: 0.4071 - val_loss: 0.8388
Epoch 4/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9136 - dice_coefficient: 0.7546 - iou_coefficient: 0.6093 - loss: 0.4863 - val_binary_accuracy: 0.8588 - val_dice_coefficient: 0.6536 - val_iou_coefficient: 0.4920 - val_loss: 0.7517
Epoch 5/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9107 - dice_coefficient: 0.7517 - iou_coefficient: 0.6059 - loss: 0.4895 - val_binary_accuracy: 0.8728 - val_dice_coefficient: 0.7248 - val_iou_coefficient: 0.5749 - val_loss: 0.7037
Epoch 6/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9168 - dice_coefficient: 0.7645 - iou_coefficient: 0.6248 - loss: 0.4653 - val_binary_accuracy: 0.9016 - val_dice_coefficient: 0.7442 - val_iou_coefficient: 0.5950 - val_loss: 0.5285
Epoch 7/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9256 - dice_coefficient: 0.7860 - iou_coefficient: 0.6513 - loss: 0.4175 - val_binary_accuracy: 0.8956 - val_dice_coefficient: 0.7583 - val_iou_coefficient: 0.6168 - val_loss: 0.5280
Epoch 8/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9245 - dice_coefficient: 0.7928 - iou_coefficient: 0.6595 - loss: 0.4106 - val_binary_accuracy: 0.9316 - val_dice_coefficient: 0.7953 - val_iou_coefficient: 0.6660 - val_loss: 0.3810
Epoch 9/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9283 - dice_coefficient: 0.8039 - iou_coefficient: 0.6751 - loss: 0.3940 - val_binary_accuracy: 0.9074 - val_dice_coefficient: 0.7386 - val_iou_coefficient: 0.5913 - val_loss: 0.4837
Epoch 10/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9289 - dice_coefficient: 0.7995 - iou_coefficient: 0.6690 - loss: 0.3956 - val_binary_accuracy: 0.9212 - val_dice_coefficient: 0.7950 - val_iou_coefficient: 0.6671 - val_loss: 0.4291
Epoch 11/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9350 - dice_coefficient: 0.8195 - iou_coefficient: 0.6963 - loss: 0.3552 - val_binary_accuracy: 0.9359 - val_dice_coefficient: 0.8343 - val_iou_coefficient: 0.7188 - val_loss: 0.3615
Epoch 12/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9303 - dice_coefficient: 0.8147 - iou_coefficient: 0.6899 - loss: 0.3736 - val_binary_accuracy: 0.9326 - val_dice_coefficient: 0.8460 - val_iou_coefficient: 0.7352 - val_loss: 0.3489
Epoch 13/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9384 - dice_coefficient: 0.8306 - iou_coefficient: 0.7134 - loss: 0.3381 - val_binary_accuracy: 0.9330 - val_dice_coefficient: 0.8221 - val_iou_coefficient: 0.7003 - val_loss: 0.3558
Epoch 14/30
40/40 - 19s - 480ms/step - binary_accuracy: 0.9403 - dice_coefficient: 0.8384 - iou_coefficient: 0.7231 - loss: 0.3234 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8425 - val_iou_coefficient: 0.7298 - val_loss: 0.3312
Epoch 15/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9372 - dice_coefficient: 0.8335 - iou_coefficient: 0.7171 - loss: 0.3401 - val_binary_accuracy: 0.9285 - val_dice_coefficient: 0.8239 - val_iou_coefficient: 0.7050 - val_loss: 0.3762
Epoch 16/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9392 - dice_coefficient: 0.8378 - iou_coefficient: 0.7231 - loss: 0.3277 - val_binary_accuracy: 0.9356 - val_dice_coefficient: 0.8319 - val_iou_coefficient: 0.7160 - val_loss: 0.3357
Epoch 17/30
40/40 - 19s - 482ms/step - binary_accuracy: 0.9385 - dice_coefficient: 0.8390 - iou_coefficient: 0.7245 - loss: 0.3325 - val_binary_accuracy: 0.9275 - val_dice_coefficient: 0.8267 - val_iou_coefficient: 0.7076 - val_loss: 0.3792
Epoch 18/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9377 - dice_coefficient: 0.8342 - iou_coefficient: 0.7186 - loss: 0.3407 - val_binary_accuracy: 0.9429 - val_dice_coefficient: 0.8557 - val_iou_coefficient: 0.7499 - val_loss: 0.3044
Epoch 19/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9407 - dice_coefficient: 0.8458 - iou_coefficient: 0.7352 - loss: 0.3110 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8520 - val_iou_coefficient: 0.7454 - val_loss: 0.3173
Epoch 20/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9463 - dice_coefficient: 0.8544 - iou_coefficient: 0.7479 - loss: 0.2984 - val_binary_accuracy: 0.9405 - val_dice_coefficient: 0.8243 - val_iou_coefficient: 0.7207 - val_loss: 0.3082
Epoch 21/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9498 - dice_coefficient: 0.8654 - iou_coefficient: 0.7647 - loss: 0.2715 - val_binary_accuracy: 0.9361 - val_dice_coefficient: 0.8548 - val_iou_coefficient: 0.7491 - val_loss: 0.3347
Epoch 22/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9404 - dice_coefficient: 0.8487 - iou_coefficient: 0.7397 - loss: 0.3174 - val_binary_accuracy: 0.9423 - val_dice_coefficient: 0.8683 - val_iou_coefficient: 0.7685 - val_loss: 0.3115
Epoch 23/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9480 - dice_coefficient: 0.8618 - iou_coefficient: 0.7587 - loss: 0.2793 - val_binary_accuracy: 0.9411 - val_dice_coefficient: 0.8579 - val_iou_coefficient: 0.7524 - val_loss: 0.3094
Epoch 24/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9438 - dice_coefficient: 0.8540 - iou_coefficient: 0.7483 - loss: 0.3007 - val_binary_accuracy: 0.9399 - val_dice_coefficient: 0.8735 - val_iou_coefficient: 0.7765 - val_loss: 0.2986
Epoch 25/30
40/40 - 19s - 478ms/step - binary_accuracy: 0.9473 - dice_coefficient: 0.8591 - iou_coefficient: 0.7564 - loss: 0.2801 - val_binary_accuracy: 0.9302 - val_dice_coefficient: 0.8361 - val_iou_coefficient: 0.7219 - val_loss: 0.3647
Epoch 26/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9512 - dice_coefficient: 0.8733 - iou_coefficient: 0.7762 - loss: 0.2595 - val_binary_accuracy: 0.9286 - val_dice_coefficient: 0.8473 - val_iou_coefficient: 0.7405 - val_loss: 0.3462
Epoch 27/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9515 - dice_coefficient: 0.8727 - iou_coefficient: 0.7754 - loss: 0.2574 - val_binary_accuracy: 0.9440 - val_dice_coefficient: 0.8604 - val_iou_coefficient: 0.7588 - val_loss: 0.2914
Epoch 28/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9545 - dice_coefficient: 0.8805 - iou_coefficient: 0.7877 - loss: 0.2439 - val_binary_accuracy: 0.9396 - val_dice_coefficient: 0.8634 - val_iou_coefficient: 0.7610 - val_loss: 0.3014
Epoch 29/30
40/40 - 19s - 479ms/step - binary_accuracy: 0.9488 - dice_coefficient: 0.8662 - iou_coefficient: 0.7663 - loss: 0.2740 - val_binary_accuracy: 0.9310 - val_dice_coefficient: 0.8340 - val_iou_coefficient: 0.7211 - val_loss: 0.3658
Epoch 30/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9527 - dice_coefficient: 0.8772 - iou_coefficient: 0.7828 - loss: 0.2500 - val_binary_accuracy: 0.9420 - val_dice_coefficient: 0.8628 - val_iou_coefficient: 0.7602 - val_loss: 0.3015
  → Dice=0.8735  IoU=0.7765  (600s, 30 epochs)

============================================================
  size=384  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 126s - 3s/step - binary_accuracy: 0.8239 - dice_coefficient: 0.5919 - iou_coefficient: 0.4234 - loss: 0.8351 - val_binary_accuracy: 0.7636 - val_dice_coefficient: 0.3571 - val_iou_coefficient: 0.2185 - val_loss: 1.2645
Epoch 2/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.8664 - dice_coefficient: 0.6566 - iou_coefficient: 0.4950 - loss: 0.7007 - val_binary_accuracy: 0.7195 - val_dice_coefficient: 0.3529 - val_iou_coefficient: 0.2154 - val_loss: 1.1987
Epoch 3/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8761 - dice_coefficient: 0.6913 - iou_coefficient: 0.5337 - loss: 0.6283 - val_binary_accuracy: 0.7398 - val_dice_coefficient: 0.2814 - val_iou_coefficient: 0.1645 - val_loss: 1.2902
Epoch 4/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8958 - dice_coefficient: 0.7290 - iou_coefficient: 0.5770 - loss: 0.5407 - val_binary_accuracy: 0.7977 - val_dice_coefficient: 0.4630 - val_iou_coefficient: 0.3104 - val_loss: 1.0636
Epoch 5/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8979 - dice_coefficient: 0.7347 - iou_coefficient: 0.5865 - loss: 0.5451 - val_binary_accuracy: 0.8637 - val_dice_coefficient: 0.6740 - val_iou_coefficient: 0.5157 - val_loss: 0.7217
Epoch 6/30
40/40 - 30s - 759ms/step - binary_accuracy: 0.9004 - dice_coefficient: 0.7453 - iou_coefficient: 0.6000 - loss: 0.5204 - val_binary_accuracy: 0.8090 - val_dice_coefficient: 0.6466 - val_iou_coefficient: 0.4909 - val_loss: 0.8789
Epoch 7/30
40/40 - 30s - 757ms/step - binary_accuracy: 0.9067 - dice_coefficient: 0.7592 - iou_coefficient: 0.6170 - loss: 0.4879 - val_binary_accuracy: 0.8684 - val_dice_coefficient: 0.7155 - val_iou_coefficient: 0.5656 - val_loss: 0.6512
Epoch 8/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9229 - dice_coefficient: 0.7985 - iou_coefficient: 0.6662 - loss: 0.4114 - val_binary_accuracy: 0.9034 - val_dice_coefficient: 0.7834 - val_iou_coefficient: 0.6470 - val_loss: 0.4987
Epoch 9/30
40/40 - 31s - 763ms/step - binary_accuracy: 0.9198 - dice_coefficient: 0.7974 - iou_coefficient: 0.6657 - loss: 0.4182 - val_binary_accuracy: 0.9103 - val_dice_coefficient: 0.7941 - val_iou_coefficient: 0.6627 - val_loss: 0.4624
Epoch 10/30
40/40 - 30s - 762ms/step - binary_accuracy: 0.9228 - dice_coefficient: 0.8013 - iou_coefficient: 0.6726 - loss: 0.4042 - val_binary_accuracy: 0.9120 - val_dice_coefficient: 0.7808 - val_iou_coefficient: 0.6432 - val_loss: 0.4622
Epoch 11/30
40/40 - 30s - 754ms/step - binary_accuracy: 0.9215 - dice_coefficient: 0.8042 - iou_coefficient: 0.6743 - loss: 0.4006 - val_binary_accuracy: 0.9184 - val_dice_coefficient: 0.8141 - val_iou_coefficient: 0.6894 - val_loss: 0.4242
Epoch 12/30
40/40 - 30s - 757ms/step - binary_accuracy: 0.9280 - dice_coefficient: 0.8214 - iou_coefficient: 0.6995 - loss: 0.3745 - val_binary_accuracy: 0.9278 - val_dice_coefficient: 0.7924 - val_iou_coefficient: 0.6787 - val_loss: 0.3709
Epoch 13/30
40/40 - 30s - 760ms/step - binary_accuracy: 0.9226 - dice_coefficient: 0.8014 - iou_coefficient: 0.6742 - loss: 0.4071 - val_binary_accuracy: 0.9285 - val_dice_coefficient: 0.8157 - val_iou_coefficient: 0.6941 - val_loss: 0.3663
Epoch 14/30
40/40 - 30s - 758ms/step - binary_accuracy: 0.9302 - dice_coefficient: 0.8218 - iou_coefficient: 0.7017 - loss: 0.3606 - val_binary_accuracy: 0.9232 - val_dice_coefficient: 0.8204 - val_iou_coefficient: 0.7019 - val_loss: 0.3880
Epoch 15/30
40/40 - 30s - 752ms/step - binary_accuracy: 0.9257 - dice_coefficient: 0.8112 - iou_coefficient: 0.6855 - loss: 0.3829 - val_binary_accuracy: 0.9211 - val_dice_coefficient: 0.8244 - val_iou_coefficient: 0.7049 - val_loss: 0.3908
Epoch 16/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.9284 - dice_coefficient: 0.8207 - iou_coefficient: 0.7003 - loss: 0.3707 - val_binary_accuracy: 0.9310 - val_dice_coefficient: 0.8185 - val_iou_coefficient: 0.6982 - val_loss: 0.3629
Epoch 17/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.9321 - dice_coefficient: 0.8292 - iou_coefficient: 0.7111 - loss: 0.3507 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8452 - val_iou_coefficient: 0.7356 - val_loss: 0.3298
Epoch 18/30
40/40 - 30s - 748ms/step - binary_accuracy: 0.9383 - dice_coefficient: 0.8441 - iou_coefficient: 0.7325 - loss: 0.3248 - val_binary_accuracy: 0.9360 - val_dice_coefficient: 0.8520 - val_iou_coefficient: 0.7440 - val_loss: 0.3224
Epoch 19/30
40/40 - 30s - 754ms/step - binary_accuracy: 0.9304 - dice_coefficient: 0.8268 - iou_coefficient: 0.7080 - loss: 0.3634 - val_binary_accuracy: 0.9339 - val_dice_coefficient: 0.8572 - val_iou_coefficient: 0.7527 - val_loss: 0.3423
Epoch 20/30
40/40 - 30s - 745ms/step - binary_accuracy: 0.9313 - dice_coefficient: 0.8303 - iou_coefficient: 0.7128 - loss: 0.3545 - val_binary_accuracy: 0.9282 - val_dice_coefficient: 0.8417 - val_iou_coefficient: 0.7294 - val_loss: 0.3564
Epoch 21/30
40/40 - 30s - 755ms/step - binary_accuracy: 0.9321 - dice_coefficient: 0.8331 - iou_coefficient: 0.7164 - loss: 0.3510 - val_binary_accuracy: 0.9271 - val_dice_coefficient: 0.8213 - val_iou_coefficient: 0.6989 - val_loss: 0.3725
Epoch 22/30
40/40 - 30s - 748ms/step - binary_accuracy: 0.9357 - dice_coefficient: 0.8414 - iou_coefficient: 0.7281 - loss: 0.3354 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7296 - val_loss: 0.3136
Epoch 23/30
40/40 - 30s - 752ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8348 - iou_coefficient: 0.7212 - loss: 0.3312 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8440 - val_iou_coefficient: 0.7317 - val_loss: 0.3242
Epoch 24/30
40/40 - 30s - 758ms/step - binary_accuracy: 0.9420 - dice_coefficient: 0.8462 - iou_coefficient: 0.7383 - loss: 0.3026 - val_binary_accuracy: 0.9418 - val_dice_coefficient: 0.8544 - val_iou_coefficient: 0.7479 - val_loss: 0.3045
Epoch 25/30
40/40 - 31s - 763ms/step - binary_accuracy: 0.9386 - dice_coefficient: 0.8439 - iou_coefficient: 0.7329 - loss: 0.3287 - val_binary_accuracy: 0.9379 - val_dice_coefficient: 0.8492 - val_iou_coefficient: 0.7411 - val_loss: 0.3282
Epoch 26/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8431 - iou_coefficient: 0.7317 - loss: 0.3282 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8591 - val_iou_coefficient: 0.7541 - val_loss: 0.3148
Epoch 27/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9381 - dice_coefficient: 0.8462 - iou_coefficient: 0.7354 - loss: 0.3226 - val_binary_accuracy: 0.9332 - val_dice_coefficient: 0.8511 - val_iou_coefficient: 0.7421 - val_loss: 0.3273
Epoch 28/30
40/40 - 30s - 755ms/step - binary_accuracy: 0.9357 - dice_coefficient: 0.8333 - iou_coefficient: 0.7190 - loss: 0.3422 - val_binary_accuracy: 0.9407 - val_dice_coefficient: 0.8429 - val_iou_coefficient: 0.7304 - val_loss: 0.3257
Epoch 29/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.9395 - dice_coefficient: 0.8515 - iou_coefficient: 0.7438 - loss: 0.3155 - val_binary_accuracy: 0.9425 - val_dice_coefficient: 0.8706 - val_iou_coefficient: 0.7729 - val_loss: 0.3006
Epoch 30/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9401 - dice_coefficient: 0.8526 - iou_coefficient: 0.7461 - loss: 0.3098 - val_binary_accuracy: 0.9363 - val_dice_coefficient: 0.8531 - val_iou_coefficient: 0.7456 - val_loss: 0.3438
  → Dice=0.8706  IoU=0.7729  (1002s, 30 epochs)

  ============================================================
  size=384  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 126s - 3s/step - binary_accuracy: 0.8239 - dice_coefficient: 0.5919 - iou_coefficient: 0.4234 - loss: 0.8351 - val_binary_accuracy: 0.7636 - val_dice_coefficient: 0.3571 - val_iou_coefficient: 0.2185 - val_loss: 1.2645
Epoch 2/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.8664 - dice_coefficient: 0.6566 - iou_coefficient: 0.4950 - loss: 0.7007 - val_binary_accuracy: 0.7195 - val_dice_coefficient: 0.3529 - val_iou_coefficient: 0.2154 - val_loss: 1.1987
Epoch 3/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8761 - dice_coefficient: 0.6913 - iou_coefficient: 0.5337 - loss: 0.6283 - val_binary_accuracy: 0.7398 - val_dice_coefficient: 0.2814 - val_iou_coefficient: 0.1645 - val_loss: 1.2902
Epoch 4/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8958 - dice_coefficient: 0.7290 - iou_coefficient: 0.5770 - loss: 0.5407 - val_binary_accuracy: 0.7977 - val_dice_coefficient: 0.4630 - val_iou_coefficient: 0.3104 - val_loss: 1.0636
Epoch 5/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.8979 - dice_coefficient: 0.7347 - iou_coefficient: 0.5865 - loss: 0.5451 - val_binary_accuracy: 0.8637 - val_dice_coefficient: 0.6740 - val_iou_coefficient: 0.5157 - val_loss: 0.7217
Epoch 6/30
40/40 - 30s - 759ms/step - binary_accuracy: 0.9004 - dice_coefficient: 0.7453 - iou_coefficient: 0.6000 - loss: 0.5204 - val_binary_accuracy: 0.8090 - val_dice_coefficient: 0.6466 - val_iou_coefficient: 0.4909 - val_loss: 0.8789
Epoch 7/30
40/40 - 30s - 757ms/step - binary_accuracy: 0.9067 - dice_coefficient: 0.7592 - iou_coefficient: 0.6170 - loss: 0.4879 - val_binary_accuracy: 0.8684 - val_dice_coefficient: 0.7155 - val_iou_coefficient: 0.5656 - val_loss: 0.6512
Epoch 8/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9229 - dice_coefficient: 0.7985 - iou_coefficient: 0.6662 - loss: 0.4114 - val_binary_accuracy: 0.9034 - val_dice_coefficient: 0.7834 - val_iou_coefficient: 0.6470 - val_loss: 0.4987
Epoch 9/30
40/40 - 31s - 763ms/step - binary_accuracy: 0.9198 - dice_coefficient: 0.7974 - iou_coefficient: 0.6657 - loss: 0.4182 - val_binary_accuracy: 0.9103 - val_dice_coefficient: 0.7941 - val_iou_coefficient: 0.6627 - val_loss: 0.4624
Epoch 10/30
40/40 - 30s - 762ms/step - binary_accuracy: 0.9228 - dice_coefficient: 0.8013 - iou_coefficient: 0.6726 - loss: 0.4042 - val_binary_accuracy: 0.9120 - val_dice_coefficient: 0.7808 - val_iou_coefficient: 0.6432 - val_loss: 0.4622
Epoch 11/30
40/40 - 30s - 754ms/step - binary_accuracy: 0.9215 - dice_coefficient: 0.8042 - iou_coefficient: 0.6743 - loss: 0.4006 - val_binary_accuracy: 0.9184 - val_dice_coefficient: 0.8141 - val_iou_coefficient: 0.6894 - val_loss: 0.4242
Epoch 12/30
40/40 - 30s - 757ms/step - binary_accuracy: 0.9280 - dice_coefficient: 0.8214 - iou_coefficient: 0.6995 - loss: 0.3745 - val_binary_accuracy: 0.9278 - val_dice_coefficient: 0.7924 - val_iou_coefficient: 0.6787 - val_loss: 0.3709
Epoch 13/30
40/40 - 30s - 760ms/step - binary_accuracy: 0.9226 - dice_coefficient: 0.8014 - iou_coefficient: 0.6742 - loss: 0.4071 - val_binary_accuracy: 0.9285 - val_dice_coefficient: 0.8157 - val_iou_coefficient: 0.6941 - val_loss: 0.3663
Epoch 14/30
40/40 - 30s - 758ms/step - binary_accuracy: 0.9302 - dice_coefficient: 0.8218 - iou_coefficient: 0.7017 - loss: 0.3606 - val_binary_accuracy: 0.9232 - val_dice_coefficient: 0.8204 - val_iou_coefficient: 0.7019 - val_loss: 0.3880
Epoch 15/30
40/40 - 30s - 752ms/step - binary_accuracy: 0.9257 - dice_coefficient: 0.8112 - iou_coefficient: 0.6855 - loss: 0.3829 - val_binary_accuracy: 0.9211 - val_dice_coefficient: 0.8244 - val_iou_coefficient: 0.7049 - val_loss: 0.3908
Epoch 16/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.9284 - dice_coefficient: 0.8207 - iou_coefficient: 0.7003 - loss: 0.3707 - val_binary_accuracy: 0.9310 - val_dice_coefficient: 0.8185 - val_iou_coefficient: 0.6982 - val_loss: 0.3629
Epoch 17/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.9321 - dice_coefficient: 0.8292 - iou_coefficient: 0.7111 - loss: 0.3507 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8452 - val_iou_coefficient: 0.7356 - val_loss: 0.3298
Epoch 18/30
40/40 - 30s - 748ms/step - binary_accuracy: 0.9383 - dice_coefficient: 0.8441 - iou_coefficient: 0.7325 - loss: 0.3248 - val_binary_accuracy: 0.9360 - val_dice_coefficient: 0.8520 - val_iou_coefficient: 0.7440 - val_loss: 0.3224
Epoch 19/30
40/40 - 30s - 754ms/step - binary_accuracy: 0.9304 - dice_coefficient: 0.8268 - iou_coefficient: 0.7080 - loss: 0.3634 - val_binary_accuracy: 0.9339 - val_dice_coefficient: 0.8572 - val_iou_coefficient: 0.7527 - val_loss: 0.3423
Epoch 20/30
40/40 - 30s - 745ms/step - binary_accuracy: 0.9313 - dice_coefficient: 0.8303 - iou_coefficient: 0.7128 - loss: 0.3545 - val_binary_accuracy: 0.9282 - val_dice_coefficient: 0.8417 - val_iou_coefficient: 0.7294 - val_loss: 0.3564
Epoch 21/30
40/40 - 30s - 755ms/step - binary_accuracy: 0.9321 - dice_coefficient: 0.8331 - iou_coefficient: 0.7164 - loss: 0.3510 - val_binary_accuracy: 0.9271 - val_dice_coefficient: 0.8213 - val_iou_coefficient: 0.6989 - val_loss: 0.3725
Epoch 22/30
40/40 - 30s - 748ms/step - binary_accuracy: 0.9357 - dice_coefficient: 0.8414 - iou_coefficient: 0.7281 - loss: 0.3354 - val_binary_accuracy: 0.9369 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7296 - val_loss: 0.3136
Epoch 23/30
40/40 - 30s - 752ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8348 - iou_coefficient: 0.7212 - loss: 0.3312 - val_binary_accuracy: 0.9365 - val_dice_coefficient: 0.8440 - val_iou_coefficient: 0.7317 - val_loss: 0.3242
Epoch 24/30
40/40 - 30s - 758ms/step - binary_accuracy: 0.9420 - dice_coefficient: 0.8462 - iou_coefficient: 0.7383 - loss: 0.3026 - val_binary_accuracy: 0.9418 - val_dice_coefficient: 0.8544 - val_iou_coefficient: 0.7479 - val_loss: 0.3045
Epoch 25/30
40/40 - 31s - 763ms/step - binary_accuracy: 0.9386 - dice_coefficient: 0.8439 - iou_coefficient: 0.7329 - loss: 0.3287 - val_binary_accuracy: 0.9379 - val_dice_coefficient: 0.8492 - val_iou_coefficient: 0.7411 - val_loss: 0.3282
Epoch 26/30
40/40 - 30s - 753ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8431 - iou_coefficient: 0.7317 - loss: 0.3282 - val_binary_accuracy: 0.9391 - val_dice_coefficient: 0.8591 - val_iou_coefficient: 0.7541 - val_loss: 0.3148
Epoch 27/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9381 - dice_coefficient: 0.8462 - iou_coefficient: 0.7354 - loss: 0.3226 - val_binary_accuracy: 0.9332 - val_dice_coefficient: 0.8511 - val_iou_coefficient: 0.7421 - val_loss: 0.3273
Epoch 28/30
40/40 - 30s - 755ms/step - binary_accuracy: 0.9357 - dice_coefficient: 0.8333 - iou_coefficient: 0.7190 - loss: 0.3422 - val_binary_accuracy: 0.9407 - val_dice_coefficient: 0.8429 - val_iou_coefficient: 0.7304 - val_loss: 0.3257
Epoch 29/30
40/40 - 30s - 756ms/step - binary_accuracy: 0.9395 - dice_coefficient: 0.8515 - iou_coefficient: 0.7438 - loss: 0.3155 - val_binary_accuracy: 0.9425 - val_dice_coefficient: 0.8706 - val_iou_coefficient: 0.7729 - val_loss: 0.3006
Epoch 30/30
40/40 - 30s - 750ms/step - binary_accuracy: 0.9401 - dice_coefficient: 0.8526 - iou_coefficient: 0.7461 - loss: 0.3098 - val_binary_accuracy: 0.9363 - val_dice_coefficient: 0.8531 - val_iou_coefficient: 0.7456 - val_loss: 0.3438
  → Dice=0.8706  IoU=0.7729  (1002s, 30 epochs)

============================================================
</pre>


### Récap `image_size`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `size=128` | 31,402,497 | 0.8844 | 0.7940 | 404s, 30 epochs |
| `size=256` | 31,402,497 | 0.8735 | 0.7765 | 600s, 30 epochs |
| `size=384` | 31,402,497 | 0.8706 | 0.7729 | 1002s, 30 epochs (best of 2) |

**Meilleur:** `size=128` (Dice=0.8844, IoU=0.7940).


### Figure — Courbes de validation pour `image size`

![Figure — Courbes de validation pour `image size`](Screenshot%202026-02-19%20at%2012.51.19.png)


## 7) Expérience : régularisation par dropout

**Ce qu'on teste :** `dropout_rate = 0.0, 0.2, 0.5` au bottleneck.

Objectif : vérifier si la régularisation réduit l'overfitting et améliore les scores de validation.


In [ ]:
# ── Exp 7 : dropout au bottleneck (0.0 / 0.2 / 0.5) ──

exp_drop = []
for dr in [0.0, 0.2, 0.5]:
    r = run_experiment(
        f'dropout={dr}',
        build_unet(base_filters=64, depth=4, use_skip=True, dropout_rate=dr),
        train_ds, val_ds
    )
    exp_drop.append(r)

plot_compare(exp_drop, 'dropout')

### Logs `dropout`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  dropout=0.0  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8478 - dice_coefficient: 0.6201 - iou_coefficient: 0.4587 - loss: 0.7646 - val_binary_accuracy: 0.7344 - val_dice_coefficient: 0.3527 - val_iou_coefficient: 0.2150 - val_loss: 1.1903
Epoch 2/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.8855 - dice_coefficient: 0.6925 - iou_coefficient: 0.5337 - loss: 0.6169 - val_binary_accuracy: 0.7931 - val_dice_coefficient: 0.4358 - val_iou_coefficient: 0.2812 - val_loss: 1.0103
Epoch 3/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9070 - dice_coefficient: 0.7483 - iou_coefficient: 0.6001 - loss: 0.4990 - val_binary_accuracy: 0.8204 - val_dice_coefficient: 0.5801 - val_iou_coefficient: 0.4155 - val_loss: 0.8885
Epoch 4/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9073 - dice_coefficient: 0.7545 - iou_coefficient: 0.6097 - loss: 0.5000 - val_binary_accuracy: 0.8660 - val_dice_coefficient: 0.6832 - val_iou_coefficient: 0.5262 - val_loss: 0.7422
Epoch 5/30
40/40 - 18s - 460ms/step - binary_accuracy: 0.9161 - dice_coefficient: 0.7769 - iou_coefficient: 0.6393 - loss: 0.4549 - val_binary_accuracy: 0.8598 - val_dice_coefficient: 0.6335 - val_iou_coefficient: 0.4732 - val_loss: 0.7574
Epoch 6/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9171 - dice_coefficient: 0.7860 - iou_coefficient: 0.6495 - loss: 0.4378 - val_binary_accuracy: 0.8996 - val_dice_coefficient: 0.7807 - val_iou_coefficient: 0.6447 - val_loss: 0.5299
Epoch 7/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9242 - dice_coefficient: 0.7950 - iou_coefficient: 0.6640 - loss: 0.4103 - val_binary_accuracy: 0.8927 - val_dice_coefficient: 0.7770 - val_iou_coefficient: 0.6427 - val_loss: 0.5816
Epoch 8/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9239 - dice_coefficient: 0.8029 - iou_coefficient: 0.6731 - loss: 0.3959 - val_binary_accuracy: 0.9267 - val_dice_coefficient: 0.8195 - val_iou_coefficient: 0.6972 - val_loss: 0.3939
Epoch 9/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9278 - dice_coefficient: 0.8065 - iou_coefficient: 0.6805 - loss: 0.3897 - val_binary_accuracy: 0.9283 - val_dice_coefficient: 0.8128 - val_iou_coefficient: 0.6938 - val_loss: 0.3642
Epoch 10/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9329 - dice_coefficient: 0.8232 - iou_coefficient: 0.7022 - loss: 0.3549 - val_binary_accuracy: 0.9247 - val_dice_coefficient: 0.8248 - val_iou_coefficient: 0.7048 - val_loss: 0.3811
Epoch 11/30
40/40 - 18s - 459ms/step - binary_accuracy: 0.9387 - dice_coefficient: 0.8382 - iou_coefficient: 0.7242 - loss: 0.3291 - val_binary_accuracy: 0.9380 - val_dice_coefficient: 0.8292 - val_iou_coefficient: 0.7111 - val_loss: 0.3398
Epoch 12/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9388 - dice_coefficient: 0.8429 - iou_coefficient: 0.7296 - loss: 0.3214 - val_binary_accuracy: 0.9364 - val_dice_coefficient: 0.8346 - val_iou_coefficient: 0.7212 - val_loss: 0.3402
Epoch 13/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9408 - dice_coefficient: 0.8386 - iou_coefficient: 0.7257 - loss: 0.3209 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8449 - val_iou_coefficient: 0.7321 - val_loss: 0.3221
Epoch 14/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9372 - dice_coefficient: 0.8339 - iou_coefficient: 0.7176 - loss: 0.3419 - val_binary_accuracy: 0.9260 - val_dice_coefficient: 0.8329 - val_iou_coefficient: 0.7182 - val_loss: 0.3716
Epoch 15/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9407 - dice_coefficient: 0.8439 - iou_coefficient: 0.7323 - loss: 0.3136 - val_binary_accuracy: 0.9332 - val_dice_coefficient: 0.8353 - val_iou_coefficient: 0.7221 - val_loss: 0.3237
Epoch 16/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9417 - dice_coefficient: 0.8549 - iou_coefficient: 0.7475 - loss: 0.3051 - val_binary_accuracy: 0.9308 - val_dice_coefficient: 0.8430 - val_iou_coefficient: 0.7319 - val_loss: 0.3424
Epoch 17/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9381 - dice_coefficient: 0.8342 - iou_coefficient: 0.7222 - loss: 0.3261 - val_binary_accuracy: 0.9364 - val_dice_coefficient: 0.8573 - val_iou_coefficient: 0.7532 - val_loss: 0.3157
Epoch 18/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9444 - dice_coefficient: 0.8554 - iou_coefficient: 0.7491 - loss: 0.2947 - val_binary_accuracy: 0.9272 - val_dice_coefficient: 0.8430 - val_iou_coefficient: 0.7329 - val_loss: 0.3599
Epoch 19/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9447 - dice_coefficient: 0.8539 - iou_coefficient: 0.7476 - loss: 0.2959 - val_binary_accuracy: 0.9432 - val_dice_coefficient: 0.8598 - val_iou_coefficient: 0.7561 - val_loss: 0.2957
Epoch 20/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9470 - dice_coefficient: 0.8635 - iou_coefficient: 0.7618 - loss: 0.2801 - val_binary_accuracy: 0.9284 - val_dice_coefficient: 0.8335 - val_iou_coefficient: 0.7172 - val_loss: 0.3788
Epoch 21/30
40/40 - 19s - 483ms/step - binary_accuracy: 0.9465 - dice_coefficient: 0.8578 - iou_coefficient: 0.7537 - loss: 0.2802 - val_binary_accuracy: 0.9443 - val_dice_coefficient: 0.8589 - val_iou_coefficient: 0.7566 - val_loss: 0.2900
Epoch 22/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9500 - dice_coefficient: 0.8696 - iou_coefficient: 0.7705 - loss: 0.2631 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8610 - val_iou_coefficient: 0.7578 - val_loss: 0.3330
Epoch 23/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9454 - dice_coefficient: 0.8606 - iou_coefficient: 0.7582 - loss: 0.2901 - val_binary_accuracy: 0.9389 - val_dice_coefficient: 0.8557 - val_iou_coefficient: 0.7505 - val_loss: 0.3016
Epoch 24/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9474 - dice_coefficient: 0.8689 - iou_coefficient: 0.7701 - loss: 0.2724 - val_binary_accuracy: 0.9413 - val_dice_coefficient: 0.8603 - val_iou_coefficient: 0.7567 - val_loss: 0.2943
Epoch 25/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9456 - dice_coefficient: 0.8624 - iou_coefficient: 0.7603 - loss: 0.2880 - val_binary_accuracy: 0.9350 - val_dice_coefficient: 0.8644 - val_iou_coefficient: 0.7626 - val_loss: 0.3046
Epoch 26/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9457 - dice_coefficient: 0.8619 - iou_coefficient: 0.7593 - loss: 0.2802 - val_binary_accuracy: 0.9411 - val_dice_coefficient: 0.8650 - val_iou_coefficient: 0.7632 - val_loss: 0.2928
Epoch 27/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9502 - dice_coefficient: 0.8743 - iou_coefficient: 0.7781 - loss: 0.2599 - val_binary_accuracy: 0.9378 - val_dice_coefficient: 0.8380 - val_iou_coefficient: 0.7309 - val_loss: 0.3104
Epoch 28/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9500 - dice_coefficient: 0.8718 - iou_coefficient: 0.7745 - loss: 0.2621 - val_binary_accuracy: 0.9412 - val_dice_coefficient: 0.8550 - val_iou_coefficient: 0.7496 - val_loss: 0.2975
Epoch 29/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9538 - dice_coefficient: 0.8844 - iou_coefficient: 0.7940 - loss: 0.2417 - val_binary_accuracy: 0.9462 - val_dice_coefficient: 0.8754 - val_iou_coefficient: 0.7799 - val_loss: 0.2726
Epoch 30/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9544 - dice_coefficient: 0.8794 - iou_coefficient: 0.7873 - loss: 0.2428 - val_binary_accuracy: 0.9413 - val_dice_coefficient: 0.8636 - val_iou_coefficient: 0.7622 - val_loss: 0.2920
  → Dice=0.8754  IoU=0.7799  (593s, 30 epochs)

============================================================
  dropout=0.2  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 53s - 1s/step - binary_accuracy: 0.8492 - dice_coefficient: 0.6243 - iou_coefficient: 0.4614 - loss: 0.7591 - val_binary_accuracy: 0.6486 - val_dice_coefficient: 0.4637 - val_iou_coefficient: 0.3061 - val_loss: 1.2760
Epoch 2/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.8849 - dice_coefficient: 0.7054 - iou_coefficient: 0.5496 - loss: 0.5974 - val_binary_accuracy: 0.7528 - val_dice_coefficient: 0.3765 - val_iou_coefficient: 0.2353 - val_loss: 1.1832
Epoch 3/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9004 - dice_coefficient: 0.7352 - iou_coefficient: 0.5855 - loss: 0.5324 - val_binary_accuracy: 0.8185 - val_dice_coefficient: 0.6408 - val_iou_coefficient: 0.4753 - val_loss: 0.9496
Epoch 4/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9050 - dice_coefficient: 0.7499 - iou_coefficient: 0.6052 - loss: 0.4952 - val_binary_accuracy: 0.8762 - val_dice_coefficient: 0.7133 - val_iou_coefficient: 0.5558 - val_loss: 0.6394
Epoch 5/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9147 - dice_coefficient: 0.7793 - iou_coefficient: 0.6406 - loss: 0.4390 - val_binary_accuracy: 0.8834 - val_dice_coefficient: 0.7194 - val_iou_coefficient: 0.5642 - val_loss: 0.6260
Epoch 6/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9260 - dice_coefficient: 0.8040 - iou_coefficient: 0.6754 - loss: 0.3942 - val_binary_accuracy: 0.8934 - val_dice_coefficient: 0.7386 - val_iou_coefficient: 0.5936 - val_loss: 0.5688
Epoch 7/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9243 - dice_coefficient: 0.8037 - iou_coefficient: 0.6739 - loss: 0.3976 - val_binary_accuracy: 0.9053 - val_dice_coefficient: 0.7881 - val_iou_coefficient: 0.6530 - val_loss: 0.4808
Epoch 8/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9229 - dice_coefficient: 0.8029 - iou_coefficient: 0.6733 - loss: 0.3967 - val_binary_accuracy: 0.9148 - val_dice_coefficient: 0.7944 - val_iou_coefficient: 0.6616 - val_loss: 0.4815
Epoch 9/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9308 - dice_coefficient: 0.8142 - iou_coefficient: 0.6903 - loss: 0.3680 - val_binary_accuracy: 0.9275 - val_dice_coefficient: 0.8234 - val_iou_coefficient: 0.7030 - val_loss: 0.3881
Epoch 10/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9326 - dice_coefficient: 0.8249 - iou_coefficient: 0.7036 - loss: 0.3556 - val_binary_accuracy: 0.9372 - val_dice_coefficient: 0.8441 - val_iou_coefficient: 0.7326 - val_loss: 0.3294
Epoch 11/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9323 - dice_coefficient: 0.8223 - iou_coefficient: 0.7013 - loss: 0.3575 - val_binary_accuracy: 0.9001 - val_dice_coefficient: 0.7882 - val_iou_coefficient: 0.6548 - val_loss: 0.5225
Epoch 12/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9337 - dice_coefficient: 0.8316 - iou_coefficient: 0.7135 - loss: 0.3460 - val_binary_accuracy: 0.9297 - val_dice_coefficient: 0.8055 - val_iou_coefficient: 0.6786 - val_loss: 0.3967
Epoch 13/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9334 - dice_coefficient: 0.8317 - iou_coefficient: 0.7142 - loss: 0.3422 - val_binary_accuracy: 0.9325 - val_dice_coefficient: 0.8496 - val_iou_coefficient: 0.7405 - val_loss: 0.3336
Epoch 14/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9351 - dice_coefficient: 0.8333 - iou_coefficient: 0.7171 - loss: 0.3466 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8506 - val_iou_coefficient: 0.7418 - val_loss: 0.3333
Epoch 15/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9379 - dice_coefficient: 0.8366 - iou_coefficient: 0.7229 - loss: 0.3287 - val_binary_accuracy: 0.9280 - val_dice_coefficient: 0.8298 - val_iou_coefficient: 0.7151 - val_loss: 0.3737
Epoch 16/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9425 - dice_coefficient: 0.8488 - iou_coefficient: 0.7397 - loss: 0.3066 - val_binary_accuracy: 0.9374 - val_dice_coefficient: 0.8370 - val_iou_coefficient: 0.7242 - val_loss: 0.3198
Epoch 17/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8544 - iou_coefficient: 0.7487 - loss: 0.2971 - val_binary_accuracy: 0.9333 - val_dice_coefficient: 0.8506 - val_iou_coefficient: 0.7417 - val_loss: 0.3330
Epoch 18/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9428 - dice_coefficient: 0.8547 - iou_coefficient: 0.7487 - loss: 0.2995 - val_binary_accuracy: 0.9416 - val_dice_coefficient: 0.8600 - val_iou_coefficient: 0.7560 - val_loss: 0.3027
Epoch 19/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9440 - dice_coefficient: 0.8526 - iou_coefficient: 0.7455 - loss: 0.2982 - val_binary_accuracy: 0.9383 - val_dice_coefficient: 0.8518 - val_iou_coefficient: 0.7437 - val_loss: 0.3156
Epoch 20/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8533 - iou_coefficient: 0.7468 - loss: 0.2981 - val_binary_accuracy: 0.9388 - val_dice_coefficient: 0.8699 - val_iou_coefficient: 0.7737 - val_loss: 0.3056
Epoch 21/30
40/40 - 18s - 460ms/step - binary_accuracy: 0.9440 - dice_coefficient: 0.8537 - iou_coefficient: 0.7484 - loss: 0.3001 - val_binary_accuracy: 0.9270 - val_dice_coefficient: 0.8379 - val_iou_coefficient: 0.7239 - val_loss: 0.3640
Epoch 22/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9485 - dice_coefficient: 0.8650 - iou_coefficient: 0.7638 - loss: 0.2735 - val_binary_accuracy: 0.9423 - val_dice_coefficient: 0.8613 - val_iou_coefficient: 0.7600 - val_loss: 0.2919
Epoch 23/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9522 - dice_coefficient: 0.8781 - iou_coefficient: 0.7842 - loss: 0.2515 - val_binary_accuracy: 0.9443 - val_dice_coefficient: 0.8614 - val_iou_coefficient: 0.7608 - val_loss: 0.2770
Epoch 24/30
40/40 - 18s - 460ms/step - binary_accuracy: 0.9518 - dice_coefficient: 0.8775 - iou_coefficient: 0.7833 - loss: 0.2525 - val_binary_accuracy: 0.9421 - val_dice_coefficient: 0.8624 - val_iou_coefficient: 0.7603 - val_loss: 0.2967
Epoch 25/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9437 - dice_coefficient: 0.8599 - iou_coefficient: 0.7566 - loss: 0.2892 - val_binary_accuracy: 0.9295 - val_dice_coefficient: 0.8538 - val_iou_coefficient: 0.7475 - val_loss: 0.3405
Epoch 26/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9511 - dice_coefficient: 0.8755 - iou_coefficient: 0.7797 - loss: 0.2589 - val_binary_accuracy: 0.9451 - val_dice_coefficient: 0.8767 - val_iou_coefficient: 0.7823 - val_loss: 0.2749
Epoch 27/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9491 - dice_coefficient: 0.8723 - iou_coefficient: 0.7745 - loss: 0.2686 - val_binary_accuracy: 0.9425 - val_dice_coefficient: 0.8508 - val_iou_coefficient: 0.7448 - val_loss: 0.2984
Epoch 28/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9529 - dice_coefficient: 0.8754 - iou_coefficient: 0.7811 - loss: 0.2501 - val_binary_accuracy: 0.9381 - val_dice_coefficient: 0.8566 - val_iou_coefficient: 0.7532 - val_loss: 0.3131
Epoch 29/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8725 - iou_coefficient: 0.7762 - loss: 0.2709 - val_binary_accuracy: 0.9337 - val_dice_coefficient: 0.8670 - val_iou_coefficient: 0.7665 - val_loss: 0.3180
Epoch 30/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9485 - dice_coefficient: 0.8692 - iou_coefficient: 0.7718 - loss: 0.2698 - val_binary_accuracy: 0.9425 - val_dice_coefficient: 0.8678 - val_iou_coefficient: 0.7683 - val_loss: 0.2924
  → Dice=0.8767  IoU=0.7823  (596s, 30 epochs)

============================================================
  dropout=0.5  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 53s - 1s/step - binary_accuracy: 0.8369 - dice_coefficient: 0.6089 - iou_coefficient: 0.4447 - loss: 0.7910 - val_binary_accuracy: 0.7153 - val_dice_coefficient: 0.2942 - val_iou_coefficient: 0.1737 - val_loss: 1.2718
Epoch 2/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.8871 - dice_coefficient: 0.7076 - iou_coefficient: 0.5524 - loss: 0.5884 - val_binary_accuracy: 0.7531 - val_dice_coefficient: 0.3310 - val_iou_coefficient: 0.2020 - val_loss: 1.2339
Epoch 3/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.8939 - dice_coefficient: 0.7291 - iou_coefficient: 0.5770 - loss: 0.5453 - val_binary_accuracy: 0.8465 - val_dice_coefficient: 0.5810 - val_iou_coefficient: 0.4114 - val_loss: 0.8454
Epoch 4/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9119 - dice_coefficient: 0.7646 - iou_coefficient: 0.6250 - loss: 0.4747 - val_binary_accuracy: 0.8505 - val_dice_coefficient: 0.6295 - val_iou_coefficient: 0.4633 - val_loss: 0.8150
Epoch 5/30
40/40 - 19s - 473ms/step - binary_accuracy: 0.9151 - dice_coefficient: 0.7789 - iou_coefficient: 0.6412 - loss: 0.4549 - val_binary_accuracy: 0.8725 - val_dice_coefficient: 0.6928 - val_iou_coefficient: 0.5422 - val_loss: 0.7065
Epoch 6/30
40/40 - 19s - 486ms/step - binary_accuracy: 0.9173 - dice_coefficient: 0.7829 - iou_coefficient: 0.6469 - loss: 0.4374 - val_binary_accuracy: 0.8685 - val_dice_coefficient: 0.7013 - val_iou_coefficient: 0.5497 - val_loss: 0.6578
Epoch 7/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9245 - dice_coefficient: 0.8009 - iou_coefficient: 0.6700 - loss: 0.4020 - val_binary_accuracy: 0.9111 - val_dice_coefficient: 0.7840 - val_iou_coefficient: 0.6511 - val_loss: 0.4722
Epoch 8/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9296 - dice_coefficient: 0.8169 - iou_coefficient: 0.6933 - loss: 0.3727 - val_binary_accuracy: 0.8887 - val_dice_coefficient: 0.7524 - val_iou_coefficient: 0.6078 - val_loss: 0.5610
Epoch 9/30
40/40 - 19s - 475ms/step - binary_accuracy: 0.9303 - dice_coefficient: 0.8183 - iou_coefficient: 0.6950 - loss: 0.3681 - val_binary_accuracy: 0.9108 - val_dice_coefficient: 0.8197 - val_iou_coefficient: 0.6953 - val_loss: 0.4391
Epoch 10/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9277 - dice_coefficient: 0.8096 - iou_coefficient: 0.6852 - loss: 0.3797 - val_binary_accuracy: 0.9307 - val_dice_coefficient: 0.8415 - val_iou_coefficient: 0.7289 - val_loss: 0.3494
Epoch 11/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9344 - dice_coefficient: 0.8281 - iou_coefficient: 0.7104 - loss: 0.3519 - val_binary_accuracy: 0.9367 - val_dice_coefficient: 0.8341 - val_iou_coefficient: 0.7184 - val_loss: 0.3346
Epoch 12/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9400 - dice_coefficient: 0.8421 - iou_coefficient: 0.7299 - loss: 0.3233 - val_binary_accuracy: 0.9271 - val_dice_coefficient: 0.8270 - val_iou_coefficient: 0.7082 - val_loss: 0.3809
Epoch 13/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9358 - dice_coefficient: 0.8358 - iou_coefficient: 0.7202 - loss: 0.3371 - val_binary_accuracy: 0.9407 - val_dice_coefficient: 0.8468 - val_iou_coefficient: 0.7370 - val_loss: 0.3191
Epoch 14/30
40/40 - 19s - 472ms/step - binary_accuracy: 0.9341 - dice_coefficient: 0.8280 - iou_coefficient: 0.7108 - loss: 0.3530 - val_binary_accuracy: 0.9421 - val_dice_coefficient: 0.8527 - val_iou_coefficient: 0.7444 - val_loss: 0.3098
Epoch 15/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9385 - dice_coefficient: 0.8373 - iou_coefficient: 0.7241 - loss: 0.3270 - val_binary_accuracy: 0.9108 - val_dice_coefficient: 0.8005 - val_iou_coefficient: 0.6739 - val_loss: 0.4327
Epoch 16/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9422 - dice_coefficient: 0.8473 - iou_coefficient: 0.7372 - loss: 0.3159 - val_binary_accuracy: 0.9436 - val_dice_coefficient: 0.8487 - val_iou_coefficient: 0.7383 - val_loss: 0.3007
Epoch 17/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9445 - dice_coefficient: 0.8567 - iou_coefficient: 0.7510 - loss: 0.2973 - val_binary_accuracy: 0.9226 - val_dice_coefficient: 0.8189 - val_iou_coefficient: 0.6954 - val_loss: 0.3921
Epoch 18/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9430 - dice_coefficient: 0.8535 - iou_coefficient: 0.7463 - loss: 0.3010 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8481 - val_iou_coefficient: 0.7381 - val_loss: 0.3079
Epoch 19/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9418 - dice_coefficient: 0.8508 - iou_coefficient: 0.7428 - loss: 0.3032 - val_binary_accuracy: 0.9386 - val_dice_coefficient: 0.8659 - val_iou_coefficient: 0.7650 - val_loss: 0.2993
Epoch 20/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9406 - dice_coefficient: 0.8538 - iou_coefficient: 0.7469 - loss: 0.3038 - val_binary_accuracy: 0.9376 - val_dice_coefficient: 0.8679 - val_iou_coefficient: 0.7683 - val_loss: 0.3059
Epoch 21/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9503 - dice_coefficient: 0.8726 - iou_coefficient: 0.7753 - loss: 0.2631 - val_binary_accuracy: 0.9454 - val_dice_coefficient: 0.8716 - val_iou_coefficient: 0.7734 - val_loss: 0.2736
Epoch 22/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8680 - iou_coefficient: 0.7684 - loss: 0.2710 - val_binary_accuracy: 0.9424 - val_dice_coefficient: 0.8490 - val_iou_coefficient: 0.7415 - val_loss: 0.2903
Epoch 23/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8690 - iou_coefficient: 0.7702 - loss: 0.2684 - val_binary_accuracy: 0.9452 - val_dice_coefficient: 0.8567 - val_iou_coefficient: 0.7563 - val_loss: 0.2751
Epoch 24/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9415 - dice_coefficient: 0.8549 - iou_coefficient: 0.7489 - loss: 0.2992 - val_binary_accuracy: 0.9388 - val_dice_coefficient: 0.8633 - val_iou_coefficient: 0.7619 - val_loss: 0.2977
Epoch 25/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9429 - dice_coefficient: 0.8571 - iou_coefficient: 0.7524 - loss: 0.2934 - val_binary_accuracy: 0.9351 - val_dice_coefficient: 0.8537 - val_iou_coefficient: 0.7469 - val_loss: 0.3185
Epoch 26/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9501 - dice_coefficient: 0.8692 - iou_coefficient: 0.7707 - loss: 0.2644 - val_binary_accuracy: 0.9443 - val_dice_coefficient: 0.8680 - val_iou_coefficient: 0.7699 - val_loss: 0.2796
Epoch 27/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9510 - dice_coefficient: 0.8727 - iou_coefficient: 0.7769 - loss: 0.2599 - val_binary_accuracy: 0.9401 - val_dice_coefficient: 0.8627 - val_iou_coefficient: 0.7600 - val_loss: 0.3065
Epoch 28/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9491 - dice_coefficient: 0.8700 - iou_coefficient: 0.7721 - loss: 0.2675 - val_binary_accuracy: 0.9326 - val_dice_coefficient: 0.8486 - val_iou_coefficient: 0.7404 - val_loss: 0.3383
Epoch 29/30
40/40 - 19s - 476ms/step - binary_accuracy: 0.9488 - dice_coefficient: 0.8747 - iou_coefficient: 0.7795 - loss: 0.2663 - val_binary_accuracy: 0.9417 - val_dice_coefficient: 0.8699 - val_iou_coefficient: 0.7720 - val_loss: 0.2994
  → Dice=0.8716  IoU=0.7734  (579s, 29 epochs)

============================================================
</pre>


### Récap `dropout`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `dropout=0.0` | 31,402,497 | 0.8754 | 0.7799 | 593s, 30 epochs |
| `dropout=0.2` | 31,402,497 | 0.8767 | 0.7823 | 596s, 30 epochs |
| `dropout=0.5` | 31,402,497 | 0.8716 | 0.7734 | 579s, 29 epochs |

**Meilleur:** `dropout=0.2` (Dice=0.8767, IoU=0.7823).


### Figure — Courbes de validation pour `dropout`

![Figure — Courbes de validation pour `dropout`](Screenshot%202026-02-19%20at%2012.52.09.png)


## 8) Expérience : taille de batch

**Ce qu'on teste :** `batch_size = 4, 8, 16`.

Objectif : mesurer l'effet du batch sur la stabilité de l'entraînement,
la qualité finale et le temps de calcul.


In [ ]:
# ── Exp 8 : batch size (4 / 8 / 16) ──

exp_bs = []
for bs in [4, 8, 16]:
    ds_tr = make_ds(train_img, train_mask, batch_size=bs, augment_fn=augment_flip)
    ds_vl = make_ds(val_img, val_mask, batch_size=bs)
    r = run_experiment(
        f'batch={bs}',
        build_unet(base_filters=64, depth=4, use_skip=True),
        ds_tr, ds_vl
    )
    exp_bs.append(r)

plot_compare(exp_bs, 'batch size')

### Logs `batch_size`

<pre style="font-size: 11px; line-height: 1.25; overflow-x: auto;">
  batch=4  |  params: 31,402,497
============================================================
Epoch 1/30
79/79 - 62s - 788ms/step - binary_accuracy: 0.8357 - dice_coefficient: 0.6117 - iou_coefficient: 0.4498 - loss: 0.7981 - val_binary_accuracy: 0.7651 - val_dice_coefficient: 0.3577 - val_iou_coefficient: 0.2221 - val_loss: 1.1559
Epoch 2/30
79/79 - 20s - 251ms/step - binary_accuracy: 0.8946 - dice_coefficient: 0.7000 - iou_coefficient: 0.5471 - loss: 0.5887 - val_binary_accuracy: 0.8627 - val_dice_coefficient: 0.5909 - val_iou_coefficient: 0.4320 - val_loss: 0.7890
Epoch 3/30
79/79 - 20s - 252ms/step - binary_accuracy: 0.9056 - dice_coefficient: 0.7362 - iou_coefficient: 0.5920 - loss: 0.5279 - val_binary_accuracy: 0.9015 - val_dice_coefficient: 0.7125 - val_iou_coefficient: 0.5656 - val_loss: 0.5816
Epoch 4/30
79/79 - 20s - 251ms/step - binary_accuracy: 0.9232 - dice_coefficient: 0.7787 - iou_coefficient: 0.6424 - loss: 0.4382 - val_binary_accuracy: 0.9151 - val_dice_coefficient: 0.7477 - val_iou_coefficient: 0.6082 - val_loss: 0.4855
Epoch 5/30
79/79 - 20s - 248ms/step - binary_accuracy: 0.9167 - dice_coefficient: 0.7639 - iou_coefficient: 0.6252 - loss: 0.4648 - val_binary_accuracy: 0.9266 - val_dice_coefficient: 0.7871 - val_iou_coefficient: 0.6567 - val_loss: 0.4225
Epoch 6/30
79/79 - 19s - 245ms/step - binary_accuracy: 0.9242 - dice_coefficient: 0.7858 - iou_coefficient: 0.6541 - loss: 0.4243 - val_binary_accuracy: 0.9137 - val_dice_coefficient: 0.7815 - val_iou_coefficient: 0.6465 - val_loss: 0.4429
Epoch 7/30
79/79 - 20s - 254ms/step - binary_accuracy: 0.9265 - dice_coefficient: 0.7942 - iou_coefficient: 0.6653 - loss: 0.4077 - val_binary_accuracy: 0.9290 - val_dice_coefficient: 0.8306 - val_iou_coefficient: 0.7157 - val_loss: 0.3679
Epoch 8/30
79/79 - 20s - 248ms/step - binary_accuracy: 0.9261 - dice_coefficient: 0.7928 - iou_coefficient: 0.6638 - loss: 0.4094 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8451 - val_iou_coefficient: 0.7372 - val_loss: 0.3467
Epoch 9/30
79/79 - 20s - 248ms/step - binary_accuracy: 0.9285 - dice_coefficient: 0.8018 - iou_coefficient: 0.6758 - loss: 0.3978 - val_binary_accuracy: 0.9099 - val_dice_coefficient: 0.7892 - val_iou_coefficient: 0.6627 - val_loss: 0.4843
Epoch 10/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9340 - dice_coefficient: 0.8203 - iou_coefficient: 0.7007 - loss: 0.3570 - val_binary_accuracy: 0.9346 - val_dice_coefficient: 0.8346 - val_iou_coefficient: 0.7211 - val_loss: 0.3467
Epoch 11/30
79/79 - 19s - 245ms/step - binary_accuracy: 0.9369 - dice_coefficient: 0.8282 - iou_coefficient: 0.7114 - loss: 0.3420 - val_binary_accuracy: 0.9287 - val_dice_coefficient: 0.8035 - val_iou_coefficient: 0.6802 - val_loss: 0.3909
Epoch 12/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9330 - dice_coefficient: 0.8209 - iou_coefficient: 0.7007 - loss: 0.3657 - val_binary_accuracy: 0.9177 - val_dice_coefficient: 0.8085 - val_iou_coefficient: 0.6856 - val_loss: 0.3939
Epoch 13/30
79/79 - 20s - 250ms/step - binary_accuracy: 0.9356 - dice_coefficient: 0.8267 - iou_coefficient: 0.7100 - loss: 0.3484 - val_binary_accuracy: 0.9276 - val_dice_coefficient: 0.8266 - val_iou_coefficient: 0.7115 - val_loss: 0.3516
Epoch 14/30
79/79 - 19s - 246ms/step - binary_accuracy: 0.9413 - dice_coefficient: 0.8408 - iou_coefficient: 0.7300 - loss: 0.3201 - val_binary_accuracy: 0.9311 - val_dice_coefficient: 0.8233 - val_iou_coefficient: 0.7061 - val_loss: 0.3556
Epoch 15/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9431 - dice_coefficient: 0.8456 - iou_coefficient: 0.7379 - loss: 0.3107 - val_binary_accuracy: 0.9362 - val_dice_coefficient: 0.8439 - val_iou_coefficient: 0.7345 - val_loss: 0.3298
Epoch 16/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9424 - dice_coefficient: 0.8429 - iou_coefficient: 0.7338 - loss: 0.3125 - val_binary_accuracy: 0.9434 - val_dice_coefficient: 0.8471 - val_iou_coefficient: 0.7391 - val_loss: 0.3014
Epoch 17/30
79/79 - 19s - 245ms/step - binary_accuracy: 0.9457 - dice_coefficient: 0.8548 - iou_coefficient: 0.7503 - loss: 0.2939 - val_binary_accuracy: 0.9357 - val_dice_coefficient: 0.8578 - val_iou_coefficient: 0.7546 - val_loss: 0.3299
Epoch 18/30
79/79 - 19s - 246ms/step - binary_accuracy: 0.9392 - dice_coefficient: 0.8408 - iou_coefficient: 0.7304 - loss: 0.3286 - val_binary_accuracy: 0.9351 - val_dice_coefficient: 0.8506 - val_iou_coefficient: 0.7455 - val_loss: 0.3261
Epoch 19/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9435 - dice_coefficient: 0.8512 - iou_coefficient: 0.7446 - loss: 0.2986 - val_binary_accuracy: 0.9336 - val_dice_coefficient: 0.8416 - val_iou_coefficient: 0.7362 - val_loss: 0.3336
Epoch 20/30
79/79 - 19s - 242ms/step - binary_accuracy: 0.9445 - dice_coefficient: 0.8607 - iou_coefficient: 0.7581 - loss: 0.2903 - val_binary_accuracy: 0.9428 - val_dice_coefficient: 0.8644 - val_iou_coefficient: 0.7664 - val_loss: 0.2992
Epoch 21/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9477 - dice_coefficient: 0.8579 - iou_coefficient: 0.7572 - loss: 0.2845 - val_binary_accuracy: 0.9413 - val_dice_coefficient: 0.8564 - val_iou_coefficient: 0.7528 - val_loss: 0.3089
Epoch 22/30
79/79 - 19s - 247ms/step - binary_accuracy: 0.9490 - dice_coefficient: 0.8684 - iou_coefficient: 0.7705 - loss: 0.2707 - val_binary_accuracy: 0.9376 - val_dice_coefficient: 0.8518 - val_iou_coefficient: 0.7502 - val_loss: 0.3204
Epoch 23/30
79/79 - 19s - 244ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8670 - iou_coefficient: 0.7688 - loss: 0.2743 - val_binary_accuracy: 0.9423 - val_dice_coefficient: 0.8579 - val_iou_coefficient: 0.7563 - val_loss: 0.3188
Epoch 24/30
79/79 - 20s - 250ms/step - binary_accuracy: 0.9466 - dice_coefficient: 0.8663 - iou_coefficient: 0.7672 - loss: 0.2752 - val_binary_accuracy: 0.9323 - val_dice_coefficient: 0.8488 - val_iou_coefficient: 0.7432 - val_loss: 0.3419
Epoch 25/30
79/79 - 19s - 246ms/step - binary_accuracy: 0.9452 - dice_coefficient: 0.8656 - iou_coefficient: 0.7666 - loss: 0.2880 - val_binary_accuracy: 0.8898 - val_dice_coefficient: 0.7475 - val_iou_coefficient: 0.6089 - val_loss: 0.5736
Epoch 26/30
79/79 - 20s - 252ms/step - binary_accuracy: 0.9488 - dice_coefficient: 0.8673 - iou_coefficient: 0.7691 - loss: 0.2683 - val_binary_accuracy: 0.9420 - val_dice_coefficient: 0.8484 - val_iou_coefficient: 0.7433 - val_loss: 0.3018
Epoch 27/30
79/79 - 20s - 253ms/step - binary_accuracy: 0.9485 - dice_coefficient: 0.8688 - iou_coefficient: 0.7713 - loss: 0.2757 - val_binary_accuracy: 0.9255 - val_dice_coefficient: 0.8412 - val_iou_coefficient: 0.7331 - val_loss: 0.3678
Epoch 28/30
79/79 - 20s - 251ms/step - binary_accuracy: 0.9518 - dice_coefficient: 0.8760 - iou_coefficient: 0.7829 - loss: 0.2539 - val_binary_accuracy: 0.9435 - val_dice_coefficient: 0.8678 - val_iou_coefficient: 0.7727 - val_loss: 0.2826
Epoch 29/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9532 - dice_coefficient: 0.8771 - iou_coefficient: 0.7841 - loss: 0.2481 - val_binary_accuracy: 0.9433 - val_dice_coefficient: 0.8635 - val_iou_coefficient: 0.7638 - val_loss: 0.2816
Epoch 30/30
79/79 - 20s - 249ms/step - binary_accuracy: 0.9508 - dice_coefficient: 0.8790 - iou_coefficient: 0.7869 - loss: 0.2525 - val_binary_accuracy: 0.9436 - val_dice_coefficient: 0.8695 - val_iou_coefficient: 0.7745 - val_loss: 0.2897
  → Dice=0.8695  IoU=0.7745  (632s, 30 epochs)

============================================================
  batch=8  |  params: 31,402,497
============================================================
Epoch 1/30
40/40 - 51s - 1s/step - binary_accuracy: 0.8298 - dice_coefficient: 0.5918 - iou_coefficient: 0.4278 - loss: 0.8270 - val_binary_accuracy: 0.7205 - val_dice_coefficient: 0.3325 - val_iou_coefficient: 0.2009 - val_loss: 1.2620
Epoch 2/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.8857 - dice_coefficient: 0.6927 - iou_coefficient: 0.5343 - loss: 0.6134 - val_binary_accuracy: 0.7242 - val_dice_coefficient: 0.4336 - val_iou_coefficient: 0.2812 - val_loss: 1.2111
Epoch 3/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9052 - dice_coefficient: 0.7380 - iou_coefficient: 0.5873 - loss: 0.5204 - val_binary_accuracy: 0.7948 - val_dice_coefficient: 0.4500 - val_iou_coefficient: 0.2951 - val_loss: 1.0805
Epoch 4/30
40/40 - 19s - 464ms/step - binary_accuracy: 0.9126 - dice_coefficient: 0.7573 - iou_coefficient: 0.6126 - loss: 0.4795 - val_binary_accuracy: 0.8488 - val_dice_coefficient: 0.6130 - val_iou_coefficient: 0.4468 - val_loss: 0.7966
Epoch 5/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9167 - dice_coefficient: 0.7724 - iou_coefficient: 0.6328 - loss: 0.4514 - val_binary_accuracy: 0.8328 - val_dice_coefficient: 0.6193 - val_iou_coefficient: 0.4541 - val_loss: 0.8728
Epoch 6/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9211 - dice_coefficient: 0.7792 - iou_coefficient: 0.6430 - loss: 0.4346 - val_binary_accuracy: 0.8983 - val_dice_coefficient: 0.7285 - val_iou_coefficient: 0.5784 - val_loss: 0.5546
Epoch 7/30
40/40 - 18s - 456ms/step - binary_accuracy: 0.9218 - dice_coefficient: 0.7828 - iou_coefficient: 0.6478 - loss: 0.4255 - val_binary_accuracy: 0.8976 - val_dice_coefficient: 0.7149 - val_iou_coefficient: 0.5609 - val_loss: 0.5793
Epoch 8/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9211 - dice_coefficient: 0.7829 - iou_coefficient: 0.6467 - loss: 0.4309 - val_binary_accuracy: 0.8954 - val_dice_coefficient: 0.7682 - val_iou_coefficient: 0.6257 - val_loss: 0.5662
Epoch 9/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9292 - dice_coefficient: 0.8075 - iou_coefficient: 0.6802 - loss: 0.3842 - val_binary_accuracy: 0.9149 - val_dice_coefficient: 0.7995 - val_iou_coefficient: 0.6702 - val_loss: 0.4477
Epoch 10/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9312 - dice_coefficient: 0.8120 - iou_coefficient: 0.6861 - loss: 0.3722 - val_binary_accuracy: 0.9377 - val_dice_coefficient: 0.8252 - val_iou_coefficient: 0.7053 - val_loss: 0.3358
Epoch 11/30
40/40 - 18s - 460ms/step - binary_accuracy: 0.9357 - dice_coefficient: 0.8210 - iou_coefficient: 0.7000 - loss: 0.3585 - val_binary_accuracy: 0.9354 - val_dice_coefficient: 0.8360 - val_iou_coefficient: 0.7222 - val_loss: 0.3460
Epoch 12/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9387 - dice_coefficient: 0.8302 - iou_coefficient: 0.7142 - loss: 0.3362 - val_binary_accuracy: 0.9408 - val_dice_coefficient: 0.8500 - val_iou_coefficient: 0.7396 - val_loss: 0.3190
Epoch 13/30
40/40 - 18s - 462ms/step - binary_accuracy: 0.9389 - dice_coefficient: 0.8322 - iou_coefficient: 0.7149 - loss: 0.3295 - val_binary_accuracy: 0.9371 - val_dice_coefficient: 0.8502 - val_iou_coefficient: 0.7409 - val_loss: 0.3270
Epoch 14/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9383 - dice_coefficient: 0.8318 - iou_coefficient: 0.7161 - loss: 0.3337 - val_binary_accuracy: 0.9415 - val_dice_coefficient: 0.8541 - val_iou_coefficient: 0.7469 - val_loss: 0.3087
Epoch 15/30
40/40 - 19s - 474ms/step - binary_accuracy: 0.9421 - dice_coefficient: 0.8442 - iou_coefficient: 0.7328 - loss: 0.3150 - val_binary_accuracy: 0.9380 - val_dice_coefficient: 0.8477 - val_iou_coefficient: 0.7377 - val_loss: 0.3221
Epoch 16/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9413 - dice_coefficient: 0.8472 - iou_coefficient: 0.7365 - loss: 0.3112 - val_binary_accuracy: 0.9441 - val_dice_coefficient: 0.8621 - val_iou_coefficient: 0.7589 - val_loss: 0.2888
Epoch 17/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9430 - dice_coefficient: 0.8479 - iou_coefficient: 0.7387 - loss: 0.3062 - val_binary_accuracy: 0.9468 - val_dice_coefficient: 0.8667 - val_iou_coefficient: 0.7669 - val_loss: 0.2782
Epoch 18/30
40/40 - 19s - 465ms/step - binary_accuracy: 0.9428 - dice_coefficient: 0.8470 - iou_coefficient: 0.7378 - loss: 0.3117 - val_binary_accuracy: 0.9420 - val_dice_coefficient: 0.8578 - val_iou_coefficient: 0.7517 - val_loss: 0.2927
Epoch 19/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9442 - dice_coefficient: 0.8529 - iou_coefficient: 0.7471 - loss: 0.3006 - val_binary_accuracy: 0.9481 - val_dice_coefficient: 0.8698 - val_iou_coefficient: 0.7708 - val_loss: 0.2697
Epoch 20/30
40/40 - 19s - 467ms/step - binary_accuracy: 0.9453 - dice_coefficient: 0.8510 - iou_coefficient: 0.7443 - loss: 0.2928 - val_binary_accuracy: 0.9481 - val_dice_coefficient: 0.8721 - val_iou_coefficient: 0.7753 - val_loss: 0.2689
Epoch 21/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9434 - dice_coefficient: 0.8515 - iou_coefficient: 0.7455 - loss: 0.3024 - val_binary_accuracy: 0.9431 - val_dice_coefficient: 0.8599 - val_iou_coefficient: 0.7565 - val_loss: 0.2896
Epoch 22/30
40/40 - 18s - 461ms/step - binary_accuracy: 0.9444 - dice_coefficient: 0.8526 - iou_coefficient: 0.7456 - loss: 0.2944 - val_binary_accuracy: 0.9431 - val_dice_coefficient: 0.8764 - val_iou_coefficient: 0.7815 - val_loss: 0.2842
Epoch 23/30
40/40 - 19s - 471ms/step - binary_accuracy: 0.9477 - dice_coefficient: 0.8670 - iou_coefficient: 0.7666 - loss: 0.2735 - val_binary_accuracy: 0.9438 - val_dice_coefficient: 0.8675 - val_iou_coefficient: 0.7674 - val_loss: 0.2881
Epoch 24/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9459 - dice_coefficient: 0.8569 - iou_coefficient: 0.7525 - loss: 0.2857 - val_binary_accuracy: 0.9463 - val_dice_coefficient: 0.8680 - val_iou_coefficient: 0.7676 - val_loss: 0.2848
Epoch 25/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9492 - dice_coefficient: 0.8648 - iou_coefficient: 0.7644 - loss: 0.2688 - val_binary_accuracy: 0.9458 - val_dice_coefficient: 0.8751 - val_iou_coefficient: 0.7801 - val_loss: 0.2743
Epoch 26/30
40/40 - 19s - 468ms/step - binary_accuracy: 0.9482 - dice_coefficient: 0.8660 - iou_coefficient: 0.7661 - loss: 0.2762 - val_binary_accuracy: 0.9498 - val_dice_coefficient: 0.8782 - val_iou_coefficient: 0.7840 - val_loss: 0.2606
Epoch 27/30
40/40 - 19s - 466ms/step - binary_accuracy: 0.9526 - dice_coefficient: 0.8792 - iou_coefficient: 0.7857 - loss: 0.2470 - val_binary_accuracy: 0.9432 - val_dice_coefficient: 0.8649 - val_iou_coefficient: 0.7631 - val_loss: 0.2826
Epoch 28/30
40/40 - 19s - 470ms/step - binary_accuracy: 0.9522 - dice_coefficient: 0.8759 - iou_coefficient: 0.7808 - loss: 0.2540 - val_binary_accuracy: 0.9412 - val_dice_coefficient: 0.8560 - val_iou_coefficient: 0.7494 - val_loss: 0.3018
Epoch 29/30
40/40 - 19s - 469ms/step - binary_accuracy: 0.9543 - dice_coefficient: 0.8794 - iou_coefficient: 0.7863 - loss: 0.2443 - val_binary_accuracy: 0.9433 - val_dice_coefficient: 0.8549 - val_iou_coefficient: 0.7491 - val_loss: 0.2879
Epoch 30/30
40/40 - 19s - 463ms/step - binary_accuracy: 0.9549 - dice_coefficient: 0.8798 - iou_coefficient: 0.7870 - loss: 0.2441 - val_binary_accuracy: 0.9411 - val_dice_coefficient: 0.8583 - val_iou_coefficient: 0.7532 - val_loss: 0.2932
  → Dice=0.8782  IoU=0.7840  (592s, 30 epochs)

============================================================
  batch=16  |  params: 31,402,497
============================================================
Epoch 1/30
20/20 - 120s - 6s/step - binary_accuracy: 0.8329 - dice_coefficient: 0.5980 - iou_coefficient: 0.4331 - loss: 0.7998 - val_binary_accuracy: 0.7173 - val_dice_coefficient: 0.2849 - val_iou_coefficient: 0.1663 - val_loss: 1.2955
Epoch 2/30
20/20 - 18s - 895ms/step - binary_accuracy: 0.8832 - dice_coefficient: 0.7028 - iou_coefficient: 0.5444 - loss: 0.6043 - val_binary_accuracy: 0.7290 - val_dice_coefficient: 0.3217 - val_iou_coefficient: 0.1925 - val_loss: 1.2181
Epoch 3/30
20/20 - 18s - 891ms/step - binary_accuracy: 0.8965 - dice_coefficient: 0.7311 - iou_coefficient: 0.5781 - loss: 0.5434 - val_binary_accuracy: 0.7529 - val_dice_coefficient: 0.3434 - val_iou_coefficient: 0.2078 - val_loss: 1.1951
Epoch 4/30
20/20 - 18s - 891ms/step - binary_accuracy: 0.9158 - dice_coefficient: 0.7722 - iou_coefficient: 0.6299 - loss: 0.4605 - val_binary_accuracy: 0.8517 - val_dice_coefficient: 0.5586 - val_iou_coefficient: 0.3897 - val_loss: 0.8369
Epoch 5/30
20/20 - 18s - 884ms/step - binary_accuracy: 0.9212 - dice_coefficient: 0.7940 - iou_coefficient: 0.6600 - loss: 0.4222 - val_binary_accuracy: 0.8125 - val_dice_coefficient: 0.5098 - val_iou_coefficient: 0.3461 - val_loss: 0.9669
Epoch 6/30
20/20 - 18s - 887ms/step - binary_accuracy: 0.9265 - dice_coefficient: 0.8092 - iou_coefficient: 0.6809 - loss: 0.3913 - val_binary_accuracy: 0.8528 - val_dice_coefficient: 0.6286 - val_iou_coefficient: 0.4600 - val_loss: 0.7467
Epoch 7/30
20/20 - 18s - 879ms/step - binary_accuracy: 0.9249 - dice_coefficient: 0.8075 - iou_coefficient: 0.6783 - loss: 0.3936 - val_binary_accuracy: 0.8812 - val_dice_coefficient: 0.7227 - val_iou_coefficient: 0.5682 - val_loss: 0.6315
Epoch 8/30
20/20 - 17s - 873ms/step - binary_accuracy: 0.9229 - dice_coefficient: 0.8036 - iou_coefficient: 0.6745 - loss: 0.4008 - val_binary_accuracy: 0.8704 - val_dice_coefficient: 0.6849 - val_iou_coefficient: 0.5253 - val_loss: 0.7363
Epoch 9/30
20/20 - 17s - 869ms/step - binary_accuracy: 0.9346 - dice_coefficient: 0.8297 - iou_coefficient: 0.7109 - loss: 0.3469 - val_binary_accuracy: 0.8653 - val_dice_coefficient: 0.6884 - val_iou_coefficient: 0.5275 - val_loss: 0.7413
Epoch 10/30
20/20 - 17s - 872ms/step - binary_accuracy: 0.9359 - dice_coefficient: 0.8336 - iou_coefficient: 0.7154 - loss: 0.3385 - val_binary_accuracy: 0.8809 - val_dice_coefficient: 0.7214 - val_iou_coefficient: 0.5683 - val_loss: 0.6569
Epoch 11/30
20/20 - 18s - 881ms/step - binary_accuracy: 0.9394 - dice_coefficient: 0.8438 - iou_coefficient: 0.7308 - loss: 0.3192 - val_binary_accuracy: 0.8950 - val_dice_coefficient: 0.7494 - val_iou_coefficient: 0.6012 - val_loss: 0.5686
Epoch 12/30
20/20 - 18s - 894ms/step - binary_accuracy: 0.9378 - dice_coefficient: 0.8389 - iou_coefficient: 0.7245 - loss: 0.3307 - val_binary_accuracy: 0.9027 - val_dice_coefficient: 0.7778 - val_iou_coefficient: 0.6410 - val_loss: 0.5181
Epoch 13/30
20/20 - 17s - 869ms/step - binary_accuracy: 0.9355 - dice_coefficient: 0.8365 - iou_coefficient: 0.7210 - loss: 0.3411 - val_binary_accuracy: 0.8885 - val_dice_coefficient: 0.7467 - val_iou_coefficient: 0.5973 - val_loss: 0.6275
Epoch 14/30
20/20 - 17s - 860ms/step - binary_accuracy: 0.9451 - dice_coefficient: 0.8553 - iou_coefficient: 0.7486 - loss: 0.2939 - val_binary_accuracy: 0.9191 - val_dice_coefficient: 0.7941 - val_iou_coefficient: 0.6642 - val_loss: 0.4129
Epoch 15/30
20/20 - 17s - 874ms/step - binary_accuracy: 0.9458 - dice_coefficient: 0.8583 - iou_coefficient: 0.7536 - loss: 0.2890 - val_binary_accuracy: 0.9091 - val_dice_coefficient: 0.8093 - val_iou_coefficient: 0.6851 - val_loss: 0.4503
Epoch 16/30
20/20 - 18s - 881ms/step - binary_accuracy: 0.9486 - dice_coefficient: 0.8652 - iou_coefficient: 0.7635 - loss: 0.2738 - val_binary_accuracy: 0.9229 - val_dice_coefficient: 0.8091 - val_iou_coefficient: 0.6802 - val_loss: 0.4020
Epoch 17/30
20/20 - 18s - 878ms/step - binary_accuracy: 0.9487 - dice_coefficient: 0.8680 - iou_coefficient: 0.7676 - loss: 0.2719 - val_binary_accuracy: 0.9287 - val_dice_coefficient: 0.8353 - val_iou_coefficient: 0.7184 - val_loss: 0.3542
Epoch 18/30
20/20 - 18s - 885ms/step - binary_accuracy: 0.9499 - dice_coefficient: 0.8724 - iou_coefficient: 0.7747 - loss: 0.2601 - val_binary_accuracy: 0.9237 - val_dice_coefficient: 0.8208 - val_iou_coefficient: 0.6966 - val_loss: 0.4008
Epoch 19/30
20/20 - 17s - 868ms/step - binary_accuracy: 0.9479 - dice_coefficient: 0.8645 - iou_coefficient: 0.7630 - loss: 0.2737 - val_binary_accuracy: 0.9258 - val_dice_coefficient: 0.8364 - val_iou_coefficient: 0.7206 - val_loss: 0.3685
Epoch 20/30
20/20 - 18s - 875ms/step - binary_accuracy: 0.9486 - dice_coefficient: 0.8689 - iou_coefficient: 0.7694 - loss: 0.2668 - val_binary_accuracy: 0.9323 - val_dice_coefficient: 0.8467 - val_iou_coefficient: 0.7355 - val_loss: 0.3480
Epoch 21/30
20/20 - 18s - 882ms/step - binary_accuracy: 0.9472 - dice_coefficient: 0.8657 - iou_coefficient: 0.7644 - loss: 0.2762 - val_binary_accuracy: 0.9314 - val_dice_coefficient: 0.8436 - val_iou_coefficient: 0.7303 - val_loss: 0.3551
Epoch 22/30
20/20 - 18s - 882ms/step - binary_accuracy: 0.9506 - dice_coefficient: 0.8721 - iou_coefficient: 0.7744 - loss: 0.2595 - val_binary_accuracy: 0.9367 - val_dice_coefficient: 0.8605 - val_iou_coefficient: 0.7554 - val_loss: 0.3255
Epoch 23/30
20/20 - 17s - 870ms/step - binary_accuracy: 0.9530 - dice_coefficient: 0.8820 - iou_coefficient: 0.7893 - loss: 0.2435 - val_binary_accuracy: 0.9302 - val_dice_coefficient: 0.8582 - val_iou_coefficient: 0.7522 - val_loss: 0.3504
Epoch 24/30
20/20 - 18s - 889ms/step - binary_accuracy: 0.9525 - dice_coefficient: 0.8781 - iou_coefficient: 0.7835 - loss: 0.2500 - val_binary_accuracy: 0.9417 - val_dice_coefficient: 0.8628 - val_iou_coefficient: 0.7599 - val_loss: 0.3008
Epoch 25/30
20/20 - 18s - 881ms/step - binary_accuracy: 0.9556 - dice_coefficient: 0.8863 - iou_coefficient: 0.7970 - loss: 0.2346 - val_binary_accuracy: 0.9444 - val_dice_coefficient: 0.8728 - val_iou_coefficient: 0.7747 - val_loss: 0.2781
Epoch 26/30
20/20 - 18s - 890ms/step - binary_accuracy: 0.9571 - dice_coefficient: 0.8886 - iou_coefficient: 0.8004 - loss: 0.2248 - val_binary_accuracy: 0.9332 - val_dice_coefficient: 0.8512 - val_iou_coefficient: 0.7432 - val_loss: 0.3436
Epoch 27/30
20/20 - 18s - 877ms/step - binary_accuracy: 0.9507 - dice_coefficient: 0.8757 - iou_coefficient: 0.7811 - loss: 0.2577 - val_binary_accuracy: 0.9423 - val_dice_coefficient: 0.8695 - val_iou_coefficient: 0.7699 - val_loss: 0.2954
Epoch 28/30
20/20 - 17s - 868ms/step - binary_accuracy: 0.9538 - dice_coefficient: 0.8818 - iou_coefficient: 0.7893 - loss: 0.2415 - val_binary_accuracy: 0.9386 - val_dice_coefficient: 0.8696 - val_iou_coefficient: 0.7704 - val_loss: 0.3166
Epoch 29/30
20/20 - 18s - 896ms/step - binary_accuracy: 0.9529 - dice_coefficient: 0.8827 - iou_coefficient: 0.7907 - loss: 0.2441 - val_binary_accuracy: 0.9306 - val_dice_coefficient: 0.8411 - val_iou_coefficient: 0.7274 - val_loss: 0.3671
Epoch 30/30
20/20 - 18s - 887ms/step - binary_accuracy: 0.9537 - dice_coefficient: 0.8823 - iou_coefficient: 0.7902 - loss: 0.2382 - val_binary_accuracy: 0.9403 - val_dice_coefficient: 0.8691 - val_iou_coefficient: 0.7698 - val_loss: 0.3041
  → Dice=0.8728  IoU=0.7747  (631s, 30 epochs)
</pre>


### Récap `batch_size`

| Config | Params | Dice | IoU | Durée |
|---|---:|---:|---:|---|
| `batch=4` | 31,402,497 | 0.8695 | 0.7745 | 632s, 30 epochs |
| `batch=8` | 31,402,497 | 0.8782 | 0.7840 | 592s, 30 epochs |
| `batch=16` | 31,402,497 | 0.8728 | 0.7747 | 631s, 30 epochs |

**Meilleur:** `batch=8` (Dice=0.8782, IoU=0.7840).


### Figure — Courbes de validation pour `batch size`

![Figure — Courbes de validation pour `batch size`](Screenshot%202026-02-19%20at%2012.54.51.png)
